# NSLAB Logistics Intelligence Platform — Machine Learning

## 1. Contexto e objetivo de negócio

Este notebook dá continuidade às etapas de Análise Exploratória de Dados (AED) e ETL Silver do projeto **NSLAB Logistics Intelligence Platform**, utilizando o dataset **LaDe-P (Pickup) — Shanghai**.

A pergunta central de negócio é:

**Como identificar antecipadamente coletas com risco de ocorrer após o fim da janela requerida, estimar a gravidade desses atrasos e compreender os padrões operacionais e geográficos associados?**

O objetivo é desenvolver modelos de Machine Learning que possam apoiar uma Torre de Controle Logística na identificação de riscos e na análise de operações de coleta.

## 2. Escopo operacional: coleta versus entrega

O dataset LaDe-P contém registros de **coleta de mercadorias (Pickup)**.

Não estamos analisando a entrega final ao destinatário. Portanto, todos os indicadores de atraso deste laboratório referem-se exclusivamente à coleta.

Os principais eventos são:

| Campo | Significado |
|---|---|
| `accept_time` | Horário registrado em que o entregador aceita a tarefa de coleta |
| `time_window_start` | Início da janela requerida para a coleta |
| `time_window_end` | Fim da janela requerida para a coleta |
| `pickup_time` | Horário registrado de realização da coleta |

O dataset não permite confirmar, por si só, todos os acontecimentos físicos ou eventuais atrasos de registro no aplicativo.

## 3. Regra de negócio — atraso da coleta

A regra foi estabelecida na AED e implementada e validada no ETL Silver.

Uma coleta é considerada posterior ao limite quando:

`pickup_time > time_window_end`

O atraso em minutos é calculado por:

`max(0, (pickup_time - time_window_end).total_seconds() / 60)`

As coletas também podem ser classificadas em três categorias:

- **ANTES_DA_JANELA:** coleta antes de `time_window_start`.
- **DENTRO_DA_JANELA:** coleta entre `time_window_start` e `time_window_end`, inclusive.
- **APOS_A_JANELA:** coleta depois de `time_window_end`.

**O horário do aceite (`accept_time`) não participa do cálculo do atraso.**

O atraso representa um desvio em relação à janela requerida, não uma comprovação de descumprimento de SLA contratual.

## 4. Papel do aceite no Machine Learning

Embora não participe do cálculo do atraso, `accept_time` define o **instante em que desejamos realizar a previsão**.

O modelo deverá utilizar somente informações disponíveis nesse momento.

Isso estabelece uma diferença fundamental:

- **Cálculo do resultado:** utiliza o horário da coleta e o fim da janela.
- **Previsão do resultado:** utiliza características conhecidas no instante do aceite.

Registros cujo aceite ocorreu depois do fim da janela exigem avaliação específica na preparação dos dados, pois o prazo já estaria encerrado no momento da previsão. Não serão excluídos automaticamente.

## 5. Objetivos de Machine Learning

### Classificação

Prever se a coleta ocorrerá após o fim da janela requerida.

**Variável-alvo candidata:** `coleta_apos_limite`.

### Regressão

Estimar a quantidade de minutos de atraso em relação ao fim da janela.

**Variável-alvo candidata:** `minutos_apos_limite`.

### Clustering

Identificar grupos de operações com características semelhantes, considerando atributos operacionais, temporais e geográficos.

O clustering não utiliza uma variável-alvo supervisionada.

## 6. Prevenção de Data Leakage

*Data leakage* ocorre quando o treinamento utiliza informações que não estariam disponíveis no momento real da previsão.

Por exemplo, `pickup_time`, `desvio_fim_janela_min`, `minutos_apos_limite` e `classificacao_janela` não podem ser utilizados como preditores de atraso no instante do aceite.

As variáveis-alvo permanecem disponíveis para treinamento e avaliação supervisionados, mas não como entradas do modelo.

A seleção definitiva das features dependerá da disponibilidade operacional, da semântica e das características de cada coluna.

## 7. Fonte dos dados e rastreabilidade

A preparação de Machine Learning utilizará como fonte a Silver já validada:

`data/silver/pickup_shanghai_silver.parquet`

A Silver contém:

- **1.424.406 registros**
- **41 colunas**
- Dados originais preservados e métricas operacionais derivadas
- Indicadores de qualidade para rastreabilidade

Não serão repetidas as investigações já concluídas na AED e no ETL, salvo quando uma necessidade específica da modelagem exigir uma nova verificação.

Documentos de referência:

- `docs/relatorio_aed_shanghai.md`
- `docs/relatorio_etl_shanghai.md`

## 8. Estratégia de desenvolvimento

O trabalho será desenvolvido incrementalmente:

1. Definição e validação do contrato de features e targets.
2. Preparação do conjunto de dados para Machine Learning.
3. Separação dos dados para treinamento, validação e teste.
4. Construção e avaliação dos modelos de classificação, regressão e clustering.
5. Registro de experimentos, parâmetros, métricas e modelos no MLflow.
6. Interpretação dos resultados e documentação das limitações.

O projeto não utilizará modelos de séries temporais nesta etapa.

A seleção de variáveis, os algoritmos e as métricas serão justificados com base na pergunta de negócio e nas características dos dados.

**Princípio orientador:** reutilizar o conhecimento produzido nas etapas anteriores, preservar a rastreabilidade e evitar conclusões operacionais não sustentadas pelos dados.

## 2. Planejamento do Machine Learning

### Estratégia geral do laboratório

O desenvolvimento dos modelos de Machine Learning será realizado de maneira incremental, considerando três problemas distintos:

| Técnica | Pergunta de negócio | Resultado esperado |
|---|---|---|
| Classificação | A coleta ocorrerá após o fim da janela requerida? | Probabilidade de atraso |
| Regressão | Quantos minutos de atraso poderão ocorrer? | Estimativa do atraso em minutos |
| Clustering | Quais operações apresentam características semelhantes? | Identificação de grupos operacionais |

Cada problema será desenvolvido e avaliado separadamente.

### Ordem de desenvolvimento

A sequência aprovada para o laboratório é:

1. Classificação.
2. Regressão.
3. Clustering.

A classificação será o primeiro experimento porque permite identificar antecipadamente operações com maior risco de ultrapassar o limite da janela requerida.

### Metodologia

Para cada problema, serão realizadas as seguintes atividades:

1. Definição do problema e das variáveis.
2. Preparação dos dados.
3. Separação entre treinamento e teste, com validação apropriada.
4. Treinamento de um modelo inicial (*baseline*).
5. Avaliação e comparação de algoritmos.
6. Interpretação dos resultados.
7. Registro dos experimentos no MLflow.
8. Documentação das conclusões.

A escolha dos algoritmos e das métricas será justificada conforme as características dos dados.

**Diretriz:** não desenvolver simultaneamente os três problemas nem repetir as análises já concluídas na AED e no ETL.

## 2.1. Definição do problema de classificação

### Objetivo

Desenvolver um modelo supervisionado de classificação binária capaz de estimar, no instante do aceite da tarefa, o risco de uma coleta ocorrer após o fim da janela requerida.

### Variável-alvo (target)

**`coleta_apos_limite`**

| Valor | Interpretação |
|---|---|
| `False` | Coleta realizada até o fim da janela |
| `True` | Coleta realizada após o fim da janela |

O target foi construído e validado na camada Silver utilizando a regra:

`pickup_time > time_window_end`

O horário do aceite não participa do cálculo do target.

### Momento da previsão

A previsão será realizada no instante registrado em `accept_time`.

Portanto, somente informações disponíveis nesse momento poderão ser utilizadas como features.

Essa restrição é fundamental para prevenir *data leakage*.

### População elegível

Para o primeiro experimento, foi aprovada a seguinte regra:

`accept_time <= time_window_end`

Essa condição seleciona tarefas cujo aceite ocorreu até o limite final da janela requerida.

Tarefas aceitas após esse limite não participarão inicialmente do treinamento, pois representam uma situação operacional distinta: o prazo já havia terminado no instante do aceite.

**Essa decisão não implica excluir registros da Silver.**

Os registros permanecerão preservados para rastreabilidade e análises posteriores.

### Premissas e limitações

- A janela requerida é considerada conhecida no momento do aceite, conforme premissa aprovada na AED.
- O modelo prevê atraso de coleta, não atraso de entrega.
- O target representa ultrapassagem do limite da janela, não descumprimento comprovado de SLA contratual.
- A elegibilidade não garante que todas as demais variáveis estejam disponíveis no momento da previsão.
- As features serão avaliadas individualmente para evitar vazamento de informações futuras.
- O desbalanceamento entre as classes será verificado antes do treinamento.

### Critério de conclusão desta etapa

A definição do problema estará completa quando estiverem estabelecidos o target, o momento da previsão, a população elegível e as restrições de utilização das variáveis.

A etapa seguinte verificará quantitativamente a população elegível utilizando a Silver já validada.

## 2.2 — População elegível para classificação

### Objetivo

Identificar as tarefas de coleta que poderão participar do primeiro experimento de classificação.

**Regra aprovada:** considerar inicialmente tarefas cujo horário de aceite seja anterior ou igual ao fim da janela requerida.

As tarefas aceitas após o fim da janela serão mantidas na Silver e analisadas separadamente.

Nesta etapa, verificaremos:

1. Quantidade de registros elegíveis e não elegíveis.
2. Existência de valores ausentes nos campos necessários.
3. Distribuição da variável-alvo `coleta_apos_limite` entre os registros elegíveis.

**Importante:** nenhuma alteração será realizada no arquivo Silver. Esta etapa é exclusivamente de leitura e diagnóstico.

Os resultados permitirão avaliar o equilíbrio entre as classes antes de selecionar algoritmos e preparar os dados de treinamento.

In [2]:

from pathlib import Path
import pandas as pd

# Localizar a raiz do projeto.
raiz_projeto = Path.cwd().resolve()

if raiz_projeto.name == "notebooks":
    raiz_projeto = raiz_projeto.parent

caminho_silver = (
    raiz_projeto
    / "data"
    / "silver"
    / "pickup_shanghai_silver.parquet"
)

# Ler somente as colunas necessárias nesta etapa.
colunas = [
    "order_id",
    "accept_time",
    "time_window_end",
    "coleta_apos_limite",
]

df_populacao = pd.read_parquet(
    caminho_silver,
    columns=colunas,
)

# Verificar valores ausentes.
print("TOTAL DE REGISTROS:", len(df_populacao))

print("\nVALORES AUSENTES:")
print(df_populacao.isna().sum())

# Definir elegibilidade sem alterar a Silver.
campos_validos = df_populacao[
    ["accept_time", "time_window_end", "coleta_apos_limite"]
].notna().all(axis=1)

elegivel = (
    campos_validos
    & (
        df_populacao["accept_time"]
        <= df_populacao["time_window_end"]
    )
)

aceite_apos_limite = (
    campos_validos
    & (
        df_populacao["accept_time"]
        > df_populacao["time_window_end"]
    )
)

print("\nPOPULAÇÃO:")
print("Elegíveis:", int(elegivel.sum()))
print("Aceite após limite:", int(aceite_apos_limite.sum()))
print("Campos necessários ausentes:", int((~campos_validos).sum()))

print("\nDISTRIBUIÇÃO DO TARGET — ELEGÍVEIS:")
print(
    df_populacao.loc[elegivel, "coleta_apos_limite"]
    .value_counts(dropna=False)
)

print("\nPERCENTUAIS:")
print(
    df_populacao.loc[elegivel, "coleta_apos_limite"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)


TOTAL DE REGISTROS: 1424406

VALORES AUSENTES:
order_id              0
accept_time           0
time_window_end       0
coleta_apos_limite    0
dtype: int64

POPULAÇÃO:
Elegíveis: 1424195
Aceite após limite: 211
Campos necessários ausentes: 0

DISTRIBUIÇÃO DO TARGET — ELEGÍVEIS:
coleta_apos_limite
False    1387876
True       36319
Name: count, dtype: Int64

PERCENTUAIS:
coleta_apos_limite
False    97.45
True      2.55
Name: proportion, dtype: Float64


### Explicação do código

**`pd.read_parquet()`** carrega os dados da Silver. O parâmetro `columns` permite ler somente as colunas necessárias, reduzindo o uso de memória.

**Máscara booleana:** uma expressão que produz valores `True` ou `False` para cada registro. Utilizamos máscaras para identificar as tarefas elegíveis e aquelas cujo aceite ocorreu depois do limite.

**`notna()` e `all(axis=1)`:** verificam se os campos obrigatórios estão preenchidos em cada linha.

**`value_counts()`:** conta quantos registros pertencem a cada classe da variável-alvo.

**`normalize=True`:** transforma as contagens em proporções. Multiplicamos por 100 para apresentar os percentuais.

### Por que verificar a distribuição das classes?

Nosso target possui duas classes:

- `False`: coleta realizada até o fim da janela.
- `True`: coleta realizada após o fim da janela.

Se houver muito menos registros de atraso, teremos um problema de **desbalanceamento de classes**.

Nesse cenário, a acurácia isoladamente poderá ser enganosa. Precisaremos considerar métricas como *precision*, *recall*, F1-score e PR-AUC.

Nenhum modelo será treinado nesta etapa.

### Resultados, interpretação e conclusão

#### Resultados obtidos

A leitura da Silver identificou **1.424.406 registros**, sem valores ausentes nos campos necessários para a definição da população.

Aplicando o critério `accept_time <= time_window_end`, foram identificados:

| Indicador | Quantidade |
|---|---:|
| Registros totais | 1.424.406 |
| Registros elegíveis | 1.424.195 |
| Aceites após o fim da janela | 211 |
| Registros com campos necessários ausentes | 0 |

Entre os registros elegíveis, a distribuição da variável-alvo foi:

| Classe | Quantidade | Percentual |
|---|---:|---:|
| Sem atraso (`False`) | 1.387.876 | 97,45% |
| Com atraso (`True`) | 36.319 | 2,55% |

#### Interpretação

A aplicação do critério de elegibilidade retirou da população inicial de modelagem apenas 211 registros, correspondentes a aproximadamente 0,015% do total.

Esses registros permanecem preservados na Silver para análise separada.

A distribuição do target demonstra um **desbalanceamento acentuado entre as classes**, pois somente 2,55% das coletas elegíveis ocorreram após o limite da janela.

Esse resultado exige atenção na avaliação dos modelos. Um classificador que previsse sempre a classe majoritária obteria aproximadamente 97,45% de acurácia, mas não identificaria nenhuma coleta atrasada.

Portanto, a acurácia isolada não será suficiente para avaliar o desempenho do modelo.

Serão consideradas métricas como precision, recall, F1-score e PR-AUC, especialmente para avaliar a identificação das coletas com atraso.

#### Conclusão

A população inicial para classificação foi definida e validada quantitativamente.

O primeiro experimento utilizará como população candidata os **1.424.195 registros elegíveis**, respeitando a regra aprovada de previsão até o fim da janela requerida.

O desbalanceamento observado deverá ser considerado na preparação, no treinamento e na avaliação dos modelos.

Nenhum registro foi removido ou modificado na camada Silver.

**Status da etapa 2.2: concluída.**

A próxima etapa será a definição das features do modelo de classificação, considerando disponibilidade no instante do aceite, relevância operacional e prevenção de data leakage.

## 2.3 — Seleção das features para classificação

### Objetivo

Identificar e justificar as variáveis candidatas ao primeiro modelo de classificação, cujo objetivo é prever se uma tarefa de coleta ocorrerá após o fim da janela requerida.

A seleção utilizará as 41 colunas da Silver, considerando as definições e validações já documentadas na AED e no ETL.

### Critérios de seleção

Uma variável será considerada candidata quando atender aos seguintes critérios:

1. **Disponibilidade:** estar disponível no instante registrado em `accept_time`.
2. **Relevância operacional:** representar uma característica potencialmente associada ao risco de atraso.
3. **Ausência de data leakage:** não conter informações sobre o resultado futuro da coleta.
4. **Interpretabilidade:** permitir compreender sua contribuição para a previsão.
5. **Qualidade:** apresentar características adequadas para utilização no modelo ou permitir tratamento documentado.

### Variáveis candidatas iniciais

| Grupo | Variáveis | Justificativa |
|---|---|---|
| Localização | `region_id`, `aoi_id`, `aoi_type`, `lng`, `lat` | Características geográficas da tarefa |
| Entregador | `courier_id` | Identificação do responsável pela coleta |
| Janela requerida | `time_window_start`, `time_window_end`, `janela_min` | Características do prazo solicitado |
| Aceite | `accept_time` | Momento em que a previsão será realizada |
| Tempo disponível | `aceite_ate_inicio_janela_min`, `aceite_ate_fim_janela_min` | Intervalos conhecidos no instante do aceite |

As variáveis temporais poderão originar novas features, como hora do aceite e duração da janela.

A inclusão definitiva de cada variável dependerá da avaliação de disponibilidade, qualidade e risco de vazamento.

### Variáveis não utilizadas como features

O target `coleta_apos_limite` não poderá ser utilizado como entrada do modelo.

Também serão excluídas das features as informações derivadas do resultado futuro, incluindo `pickup_time`, `minutos_apos_limite`, `desvio_fim_janela_min`, `classificacao_janela`, `aceite_ate_coleta_min` e `minutos_antes_inicio`.

Campos GPS associados à coleta realizada e flags que dependem desse evento também não serão utilizados como preditores.

O campo `ds`, por representar a data da coleta, não será utilizado diretamente como feature.

O identificador `order_id` será preservado para rastreabilidade, mas não será utilizado como preditor.

### Variáveis que exigem avaliação adicional

O uso de `courier_id` e `aoi_id` exigirá atenção à cardinalidade e à capacidade de generalização do modelo.

Os campos GPS associados ao aceite dependerão da confirmação de sua disponibilidade naquele instante.

As flags de qualidade serão avaliadas individualmente, considerando o momento em que suas informações poderiam ser conhecidas.

### Resultado esperado

Produzir uma lista justificada de features aprovadas, variáveis descartadas e variáveis que exigem tratamento adicional.

A seleção deverá ser documentada antes do treinamento, evitando decisões baseadas exclusivamente no desempenho obtido pelos algoritmos.

In [3]:

# ============================================================
# ETAPA 2.3 — INVENTÁRIO DE FEATURES PARA CLASSIFICAÇÃO
# ============================================================

# Grupos definidos a partir da AED e do ETL.
# Esta classificação é preliminar: ainda não aprova
# automaticamente as variáveis para treinamento.

features_candidatas = [
    "region_id",
    "aoi_type",
    "lng",
    "lat",
    "time_window_start",
    "time_window_end",
    "accept_time",
    "janela_min",
    "aceite_ate_inicio_janela_min",
    "aceite_ate_fim_janela_min",
]

avaliacao_adicional = [
    "courier_id",
    "aoi_id",
    "accept_gps_time",
    "accept_gps_lng",
    "accept_gps_lat",
]

targets = [
    "coleta_apos_limite",
    "minutos_apos_limite",
]

rastreabilidade = [
    "order_id",
    "city",
]

# Variáveis conhecidas como posteriores à coleta,
# derivadas do resultado ou inadequadas como features.
excluidas = [
    "pickup_time",
    "pickup_gps_time",
    "pickup_gps_lng",
    "pickup_gps_lat",
    "ds",
    "aceite_ate_coleta_min",
    "desvio_fim_janela_min",
    "minutos_antes_inicio",
    "classificacao_janela",
]

# Flags serão inventariadas separadamente.
# Algumas dependem de eventos futuros e outras podem
# representar condições conhecidas no aceite.
colunas_silver = pd.read_parquet(
    caminho_silver
).columns.tolist()

flags_qualidade = [
    coluna
    for coluna in colunas_silver
    if coluna.startswith("flag_")
]

# Montar um inventário sem alterar os dados.
def classificar_coluna(coluna):
    if coluna in features_candidatas:
        return "Candidata"
    if coluna in avaliacao_adicional:
        return "Avaliação adicional"
    if coluna in targets:
        return "Target"
    if coluna in rastreabilidade:
        return "Rastreabilidade"
    if coluna in excluidas:
        return "Excluída das features"
    if coluna in flags_qualidade:
        return "Flag — avaliar disponibilidade"
    return "Não classificada"


df_inventario = pd.DataFrame({
    "coluna": colunas_silver,
})

df_inventario["situacao"] = (
    df_inventario["coluna"].apply(classificar_coluna)
)

print("TOTAL DE COLUNAS:", len(df_inventario))

print("\nRESUMO POR SITUAÇÃO:")
print(df_inventario["situacao"].value_counts())

print("\nINVENTÁRIO COMPLETO:")
print(df_inventario.to_string(index=False))

nao_classificadas = df_inventario.loc[
    df_inventario["situacao"] == "Não classificada",
    "coluna",
]

print("\nCOLUNAS NÃO CLASSIFICADAS:")
print(nao_classificadas.tolist())


TOTAL DE COLUNAS: 41

RESUMO POR SITUAÇÃO:
situacao
Flag — avaliar disponibilidade    13
Candidata                         10
Excluída das features              9
Avaliação adicional                5
Rastreabilidade                    2
Target                             2
Name: count, dtype: int64

INVENTÁRIO COMPLETO:
                            coluna                       situacao
                          order_id                Rastreabilidade
                         region_id                      Candidata
                              city                Rastreabilidade
                        courier_id            Avaliação adicional
                       accept_time                      Candidata
                 time_window_start                      Candidata
                   time_window_end                      Candidata
                               lng                      Candidata
                               lat                      Candidata
                  

### Explicação do código — Inventário de features

O código organiza as colunas da Silver conforme sua possível utilização no Machine Learning.

**Listas Python (`list`):** armazenam os nomes das variáveis agrupadas por finalidade, como features candidatas, targets e identificadores.

**`pd.read_parquet()`:** lê a Silver para obter os nomes das colunas. Nesta etapa, a leitura não modifica o arquivo original.

**`columns.tolist()`:** transforma os nomes das colunas em uma lista Python.

**List comprehension:** identifica as colunas cujo nome começa com `flag_`, permitindo agrupar os indicadores de qualidade.

**Função `classificar_coluna()`:** aplica as regras de classificação previamente definidas. O uso de `if` e `return` permite atribuir uma situação a cada variável.

**`apply()`:** executa a função de classificação para cada nome de coluna.

**`value_counts()`:** contabiliza quantas colunas pertencem a cada grupo.

**`DataFrame`:** organiza o inventário em formato tabular, facilitando sua inspeção e documentação.

### Interpretação esperada

O inventário permitirá verificar se todas as 41 colunas receberam uma classificação preliminar.

As variáveis classificadas como candidatas ainda não estão automaticamente aprovadas para treinamento.

As flags de qualidade e as variáveis que exigem avaliação adicional serão analisadas antes da definição definitiva das features.

**Nenhum modelo será treinado nesta etapa.**

### Resultados, interpretação e conclusão

#### Resultados obtidos

O inventário identificou e classificou as **41 colunas da camada Silver**, conforme as regras preliminares de seleção de variáveis para Machine Learning.

A distribuição foi:

| Categoria | Quantidade |
|---|---:|
| Features candidatas | 10 |
| Variáveis para avaliação adicional | 5 |
| Variáveis excluídas das features | 9 |
| Flags de qualidade para avaliação | 13 |
| Variáveis de rastreabilidade | 2 |
| Targets | 2 |
| **Total** | **41** |

Nenhuma coluna permaneceu sem classificação.

#### Interpretação

A classificação preliminar permite distinguir variáveis potencialmente úteis para a previsão de atraso daquelas que contêm informações posteriores ao aceite ou são destinadas à rastreabilidade.

As 10 features candidatas representam características geográficas, operacionais e temporais da tarefa.

As cinco variáveis classificadas para avaliação adicional incluem identificadores de alta cardinalidade e informações GPS associadas ao aceite.

As 13 flags de qualidade ainda exigem análise de disponibilidade temporal. Algumas representam condições conhecidas antes da coleta, enquanto outras dependem de eventos futuros e não poderão ser utilizadas como preditoras.

As duas variáveis-alvo identificadas pertencem a problemas distintos:

- `coleta_apos_limite`: classificação.
- `minutos_apos_limite`: regressão.

No primeiro experimento, somente `coleta_apos_limite` será utilizada como target. `minutos_apos_limite` permanecerá fora das features.

#### Conclusão

O inventário estrutural das variáveis foi concluído com sucesso.

Todas as 41 colunas receberam uma classificação preliminar, sem alterações na Silver.

A seleção definitiva das features ainda dependerá da análise de disponibilidade no instante do aceite, risco de data leakage, relevância operacional e características dos algoritmos.

**Status da etapa 2.3: inventário concluído; aprovação definitiva das features pendente.**

A próxima etapa será a avaliação das variáveis que exigem decisão adicional, utilizando prioritariamente as informações já documentadas na AED e no ETL.

## 2.4 — Avaliação das variáveis candidatas e prevenção de Data Leakage

### 2.4.1 — Objetivo

Avaliar as variáveis identificadas no inventário da etapa 2.3, considerando sua relevância para a previsão de atrasos, disponibilidade no momento da previsão e possíveis riscos de *data leakage*.

O objetivo é estabelecer critérios técnicos para selecionar as features dos primeiros experimentos de Machine Learning, sem descartar antecipadamente variáveis potencialmente relevantes.

Esta etapa utiliza os resultados da Análise Exploratória de Dados (AED) e do ETL, evitando repetir investigações já concluídas.

### 2.4.2 — Contexto de negócio

A pergunta central do primeiro modelo de classificação é:

**No instante em que o entregador aceita uma tarefa, é possível prever se a coleta ocorrerá após o horário limite da janela estabelecida?**

Para responder a essa pergunta:

- O momento da previsão será `accept_time`.
- O limite da janela será `time_window_end`.
- O evento observado será `pickup_time`.
- A variável-alvo será `coleta_apos_limite`.

O modelo deverá utilizar somente informações disponíveis no instante do aceite.

Informações conhecidas exclusivamente após esse momento não poderão ser utilizadas como features.

Essa restrição é fundamental para reproduzir um cenário operacional realista.

### 2.4.3 — Critérios para avaliação das features

Cada variável será avaliada considerando cinco critérios.

**1. Significado operacional**

Compreender o que a variável representa no processo de coleta e como poderia estar relacionada ao risco de atraso.

**2. Disponibilidade temporal**

Verificar se o valor estaria disponível no instante do aceite.

Uma variável pode existir no dataset histórico e, ainda assim, não estar disponível no momento em que a previsão seria realizada.

**3. Risco de Data Leakage**

Identificar informações que revelam direta ou indiretamente o resultado futuro da coleta.

O uso dessas informações poderia produzir métricas artificialmente elevadas durante o treinamento, sem representar a capacidade real de previsão.

**4. Características estatísticas**

Considerar aspectos como cardinalidade, valores ausentes, distribuição e possíveis dificuldades de representação para os algoritmos.

Essas características orientam o tratamento das variáveis, mas não determinam automaticamente sua exclusão.

**5. Contribuição preditiva**

Investigar experimentalmente se a variável melhora a capacidade do modelo de identificar coletas com risco de atraso.

A decisão definitiva de inclusão será baseada nos resultados de validação e na viabilidade operacional.

### 2.4.4 — Identificadores operacionais

A cardinalidade dos identificadores já foi investigada nas etapas anteriores do projeto.

| Identificador | Valores distintos | Valores nulos |
|---|---:|---:|
| `order_id` | 1.424.406 | 0 |
| `courier_id` | 4.494 | 0 |
| `region_id` | 30 | 0 |
| `aoi_id` | 5.867 | 0 |

Os resultados demonstram que:

- `order_id` identifica individualmente cada registro de coleta.
- `courier_id` identifica o entregador associado à tarefa.
- `region_id` identifica a região operacional.
- `aoi_id` identifica a área de interesse associada à coleta.

Os identificadores de entregadores, regiões e áreas podem aparecer em diferentes registros.

A ausência de tabelas cadastrais externas não impede sua utilização como variáveis categóricas, embora limite a interpretação de características adicionais dessas entidades.

#### Avaliação de `courier_id`

O entregador pode estar associado a diferentes padrões operacionais de atraso.

Por exemplo, entregadores podem apresentar diferenças relacionadas a experiência, regiões atendidas, carga de trabalho e outras condições operacionais.

Entretanto, essas características não são diretamente observadas apenas pelo identificador.

A variável possui 4.494 valores distintos, exigindo atenção à representação categórica, à frequência de ocorrência e à capacidade de generalização do modelo.

**Decisão:** manter `courier_id` como candidata para experimentação.

Sua contribuição será investigada comparando modelos com e sem essa variável.

#### Avaliação de `aoi_id`

A área de interesse pode representar características geográficas e operacionais específicas, potencialmente associadas à ocorrência de atrasos.

A variável possui 5.867 valores distintos.

Embora apresente alta cardinalidade, poderá fornecer informações mais específicas do que `region_id`.

**Decisão:** manter `aoi_id` como candidata para experimentação.

Sua contribuição será avaliada individualmente e em conjunto com `courier_id`.

#### Avaliação de `region_id`

A região operacional possui 30 valores distintos e representa uma característica potencialmente relevante para a previsão.

**Decisão:** manter `region_id` como feature candidata do primeiro baseline, tratando-a como variável categórica.

#### Avaliação de `order_id`

O identificador individual da coleta não representa, por si só, uma característica operacional generalizável.

**Decisão:** preservar `order_id` para rastreabilidade, sem utilizá-lo como feature.

### 2.4.5 — Campos GPS associados ao aceite

As seguintes variáveis representam informações GPS associadas ao aceite:

- `accept_gps_time`
- `accept_gps_lng`
- `accept_gps_lat`

Embora possam conter informações relevantes sobre a localização do entregador, ainda não foi comprovado que esses valores estariam disponíveis no instante exato da previsão.

A presença dessas colunas no dataset histórico não garante sua disponibilidade operacional em tempo real.

**Decisão:** não utilizar os campos GPS de aceite no primeiro experimento.

Essas variáveis permanecerão preservadas para avaliação posterior, caso seja possível comprovar sua disponibilidade temporal.

Essa decisão não afeta as coordenadas de referência `lng` e `lat`, que permanecem candidatas, considerando a hipótese operacional de que a localização da tarefa é conhecida no aceite.

### 2.4.6 — Avaliação das flags de qualidade

A camada Silver contém 13 flags utilizadas para identificar condições de qualidade e consistência dos dados.

Essas flags são importantes para auditoria e diagnóstico, mas nem todas representam informações disponíveis no instante da previsão.

| Flag | Avaliação | Decisão inicial |
|---|---|---|
| `flag_temporal_essencial_ausente` | Sua definição envolve também o horário da coleta | Não utilizar |
| `flag_janela_invertida` | Depende dos limites da janela | Diagnóstico de qualidade |
| `flag_coleta_antes_aceite` | Depende do horário real da coleta | Não utilizar |
| `flag_aceite_apos_fim_janela` | Identifica aceites posteriores ao limite | Utilizar na elegibilidade, não como feature |
| `flag_coleta_antes_inicio_janela` | Depende do horário real da coleta | Não utilizar |
| `flag_gps_aceite_ausente` | Depende dos dados GPS associados ao aceite | Não utilizar inicialmente |
| `flag_gps_aceite_parcial` | Depende dos dados GPS associados ao aceite | Não utilizar inicialmente |
| `flag_gps_coleta_ausente` | Depende dos dados GPS da coleta | Não utilizar |
| `flag_gps_coleta_parcial` | Depende dos dados GPS da coleta | Não utilizar |
| `flag_coord_referencia_fora_limites` | Avalia as coordenadas de referência | Diagnóstico de qualidade |
| `flag_coord_aceite_fora_limites` | Depende das coordenadas GPS do aceite | Não utilizar inicialmente |
| `flag_coord_coleta_fora_limites` | Depende das coordenadas GPS da coleta | Não utilizar |
| `flag_ds_divergente` | Depende da comparação com informações da coleta | Não utilizar |

#### Decisão sobre as flags

Nenhuma flag será utilizada como feature no primeiro baseline.

Isso não significa que todas sejam irrelevantes.

As flags continuarão disponíveis para análise de qualidade, auditoria e investigação de possíveis problemas nos dados.

A inclusão futura de alguma flag como preditora dependerá da confirmação de sua disponibilidade temporal e de sua contribuição para o modelo.

### 2.4.7 — Variáveis excluídas por risco de Data Leakage

As seguintes variáveis não serão utilizadas como features no modelo de classificação:

| Variável | Justificativa |
|---|---|
| `pickup_time` | Horário real da coleta, conhecido posteriormente |
| `pickup_gps_time` | Evento GPS associado à coleta |
| `pickup_gps_lng` | Longitude GPS da coleta |
| `pickup_gps_lat` | Latitude GPS da coleta |
| `aceite_ate_coleta_min` | Depende do horário real da coleta |
| `desvio_fim_janela_min` | Depende do resultado da coleta |
| `minutos_antes_inicio` | Métrica derivada do horário real da coleta |
| `classificacao_janela` | Classificação derivada do resultado observado |
| `ds` | Representa a data associada à coleta no dataset analisado |

Além dessas variáveis:

- `coleta_apos_limite` será o target da classificação, nunca uma feature.
- `minutos_apos_limite` será o target da futura regressão, não uma feature da classificação.

A exclusão dessas variáveis do treinamento não implica sua remoção da Silver.

### 2.4.8 — Princípio metodológico: seleção baseada em evidências

A seleção de features será orientada por critérios técnicos e evidências experimentais.

**Uma variável não será considerada irrelevante exclusivamente por apresentar alta cardinalidade, ser um identificador ou não possuir relacionamento com tabelas externas.**

Sempre que sua utilização for operacionalmente válida, sua contribuição preditiva poderá ser investigada.

O processo adotado será:

1. Compreender o significado da variável.
2. Verificar sua disponibilidade no momento da previsão.
3. Avaliar possíveis riscos de Data Leakage.
4. Definir uma hipótese sobre sua contribuição preditiva.
5. Comparar experimentos controlados com e sem a variável.
6. Avaliar métricas e capacidade de generalização.
7. Documentar os resultados e justificar sua inclusão ou exclusão.

#### Associação preditiva versus causalidade

Uma variável pode melhorar as previsões sem ser a causa direta do resultado observado.

Por exemplo, se `courier_id` contribuir para prever atrasos, isso não demonstrará necessariamente que o entregador provoca os atrasos.

O identificador pode representar indiretamente características operacionais não observadas no dataset.

Portanto, os experimentos permitirão investigar **poder preditivo**, mas não estabelecer automaticamente relações causais.

### 2.4.9 — Planejamento dos experimentos

Para investigar a contribuição dos identificadores operacionais, serão realizados experimentos comparáveis.

| Experimento | Configuração | Objetivo |
|---|---|---|
| A — Baseline | Features operacionais, temporais e geográficas | Estabelecer desempenho de referência |
| B — Entregador | Baseline + `courier_id` | Medir a contribuição do entregador |
| C — Área | Baseline + `aoi_id` | Medir a contribuição da área |
| D — Combinado | Baseline + `courier_id` + `aoi_id` | Avaliar a contribuição conjunta |

Os experimentos utilizarão conjuntos de avaliação comparáveis e procedimentos consistentes de preparação e treinamento.

Serão consideradas métricas adequadas ao desbalanceamento da variável-alvo, incluindo:

- Precision
- Recall
- F1-score
- PR-AUC

A acurácia isolada não será utilizada como principal critério de decisão, devido à baixa proporção de coletas após o limite da janela.

Os resultados serão registrados no MLflow, permitindo rastrear configurações, métricas e modelos.

A análise também deverá considerar o comportamento diante de entregadores ou áreas pouco representados ou não observados no treinamento.

### 2.4.10 — Conclusão

A avaliação conceitual das variáveis foi consolidada, aproveitando as informações já obtidas na AED e no ETL.

As principais decisões são:

- Preservar `order_id` exclusivamente para rastreabilidade.
- Manter `region_id` como candidata ao primeiro baseline.
- Preservar `courier_id` e `aoi_id` como candidatas para experimentos comparativos.
- Não utilizar inicialmente os campos GPS de aceite, devido à disponibilidade temporal não confirmada.
- Não utilizar flags de qualidade como features no primeiro baseline.
- Excluir do treinamento variáveis que dependam de informações futuras.
- Avaliar empiricamente a contribuição das variáveis, sem confundir associação preditiva com causalidade.

**Conclusão metodológica:** a seleção definitiva de features deverá combinar conhecimento de negócio, disponibilidade operacional, prevenção de Data Leakage e evidências obtidas em experimentos controlados.

Nenhuma alteração será realizada na camada Silver nesta etapa.

**Status da etapa 2.4: avaliação conceitual concluída e estratégia experimental definida.**

**Próxima etapa: 2.5 — Consolidação das features do primeiro modelo de classificação.**

## 2.5 — Consolidação das features para classificação

### Objetivo

Definir o conjunto inicial de variáveis preditoras para o modelo de classificação de risco de atraso nas coletas de Shanghai.

A previsão será realizada no instante `accept_time`, utilizando como target `coleta_apos_limite`.

### Features do primeiro baseline

| Variável | Tratamento previsto |
|---|---|
| `region_id` | Categórica |
| `aoi_type` | Categórica |
| `lng` | Numérica |
| `lat` | Numérica |
| `janela_min` | Numérica |
| `aceite_ate_inicio_janela_min` | Numérica |
| `aceite_ate_fim_janela_min` | Numérica |

As variáveis temporais originais `accept_time`, `time_window_start` e `time_window_end` serão utilizadas para criar atributos derivados, como horário do aceite e horário limite da janela.

Os atributos derivados deverão representar somente informações conhecidas no momento da previsão.

### Variáveis reservadas para experimentos

- `courier_id`: avaliar contribuição preditiva do entregador.
- `aoi_id`: avaliar contribuição preditiva da área operacional.

Essas variáveis não serão descartadas. Sua contribuição será investigada por meio de experimentos controlados.

### Target

`coleta_apos_limite`

### População elegível

Serão considerados os registros que atendem ao critério já estabelecido:

`accept_time <= time_window_end`

### Critérios de validação

A preparação dos dados deverá:

1. Preservar a Silver original.
2. Evitar Data Leakage.
3. Tratar corretamente variáveis categóricas.
4. Evitar utilizar identificadores como valores numéricos contínuos.
5. Garantir que as transformações aprendidas com os dados sejam ajustadas somente no conjunto de treinamento.
6. Permitir experimentos reproduzíveis e comparáveis.

### Resultado esperado

Obter uma definição explícita e documentada das features do primeiro modelo, preparando a construção dos conjuntos de treinamento e avaliação.

Nenhum modelo será treinado nesta etapa.

In [4]:

# ============================================================
# ETAPA 2.5 — CONSOLIDAÇÃO DAS FEATURES DO BASELINE
# ============================================================

# Variáveis disponíveis no instante do aceite.
features_baseline = [
    "region_id",
    "aoi_type",
    "lng",
    "lat",
    "janela_min",
    "aceite_ate_inicio_janela_min",
    "aceite_ate_fim_janela_min",
]

# Campos temporais usados futuramente para criar features.
campos_temporais = [
    "accept_time",
    "time_window_start",
    "time_window_end",
]

# Identificadores reservados para experimentos comparativos.
features_experimentos = [
    "courier_id",
    "aoi_id",
]

# Variável-alvo do problema de classificação.
target_classificacao = "coleta_apos_limite"

# Colunas necessárias para preparar o primeiro experimento.
colunas_necessarias = (
    features_baseline
    + campos_temporais
    + features_experimentos
    + [target_classificacao]
)

# Verifica se todas as colunas existem na Silver.
colunas_disponiveis = set(colunas_silver)

colunas_ausentes = [
    coluna
    for coluna in colunas_necessarias
    if coluna not in colunas_disponiveis
]

# Verifica se existem nomes repetidos na seleção.
colunas_duplicadas = len(colunas_necessarias) != len(
    set(colunas_necessarias)
)

print("=== VALIDAÇÃO DAS FEATURES ===")
print(f"Features do baseline: {len(features_baseline)}")
print(f"Campos temporais: {len(campos_temporais)}")
print(f"Features para experimentos: {len(features_experimentos)}")
print(f"Target: {target_classificacao}")
print(f"Total de colunas selecionadas: {len(colunas_necessarias)}")

print("\nColunas ausentes:", colunas_ausentes)
print("Existem colunas duplicadas:", colunas_duplicadas)

assert not colunas_ausentes, "Existem colunas ausentes na Silver."
assert not colunas_duplicadas, "Existem colunas duplicadas."

print("\nValidação estrutural concluída com sucesso.")


=== VALIDAÇÃO DAS FEATURES ===
Features do baseline: 7
Campos temporais: 3
Features para experimentos: 2
Target: coleta_apos_limite
Total de colunas selecionadas: 13

Colunas ausentes: []
Existem colunas duplicadas: False

Validação estrutural concluída com sucesso.


### Explicação do código — Etapa 2.5

O código organiza as variáveis em grupos conforme sua função no processo de Machine Learning.

**1. Listas Python**

As listas `features_baseline`, `campos_temporais` e `features_experimentos` armazenam os nomes das colunas selecionadas.

Essa organização facilita a manutenção do projeto e permite reutilizar as definições nos experimentos seguintes.

**2. Variável-alvo**

A variável `target_classificacao` identifica a coluna que o modelo deverá prever: `coleta_apos_limite`.

Ela não será utilizada como feature.

**3. Concatenação de listas**

O operador `+` combina as listas para construir `colunas_necessarias`, reunindo as informações previstas para a preparação dos experimentos.

**4. Conjuntos (`set`)**

A função `set()` cria um conjunto de elementos distintos.

Ela é utilizada para verificar a existência das colunas na Silver e identificar possíveis duplicidades na seleção.

**5. List comprehension**

A expressão que constrói `colunas_ausentes` percorre os nomes selecionados e retorna somente aqueles que não estão disponíveis.

**6. Validações com `assert`**

As instruções `assert` interrompem a execução caso alguma condição obrigatória não seja atendida.

Nesse caso, verificamos que todas as colunas existem e que nenhum nome foi selecionado duas vezes.

### Limitações desta validação

Esta etapa verifica somente a estrutura da seleção.

Ela ainda não confirma os tipos de dados, a disponibilidade temporal operacional, os valores ausentes ou a adequação das features aos algoritmos.

Esses aspectos serão tratados nas próximas etapas, antes do treinamento.

### 2.5 — Resultados, interpretação e conclusão

#### Resultados obtidos

A execução confirmou a seleção de 13 colunas da camada Silver, organizadas em quatro grupos:

- **7 features do baseline:** características operacionais, geográficas e temporais já derivadas.
- **3 campos temporais:** utilizados posteriormente para engenharia de features.
- **2 identificadores operacionais:** `courier_id` e `aoi_id`, reservados para experimentos comparativos.
- **1 variável-alvo:** `coleta_apos_limite`.

As verificações retornaram:

| Indicador | Resultado |
|---|---|
| Colunas ausentes | 0 |
| Colunas duplicadas | 0 |
| Validação estrutural | Concluída com sucesso |

#### Interpretação

Todas as colunas selecionadas estão presentes na Silver, e não foram identificadas duplicidades nos nomes.

A organização permite distinguir as features iniciais das variáveis que serão utilizadas em experimentos posteriores.

A seleção também preserva o princípio metodológico definido na etapa 2.4: variáveis potencialmente relevantes não serão descartadas exclusivamente por sua cardinalidade ou por serem identificadores.

A contribuição de `courier_id` e `aoi_id` será investigada mediante comparação de modelos.

#### Limitações

A validação realizada é exclusivamente estrutural.

Ainda será necessário verificar a adequação dos tipos de dados, preparar os atributos temporais, definir o particionamento dos dados e aplicar os tratamentos necessários ao algoritmo.

A disponibilidade das informações no instante do aceite permanece como requisito obrigatório.

#### Conclusão

A seleção inicial de colunas para o primeiro modelo de classificação foi consolidada e validada.

Nenhuma coluna está ausente ou duplicada, e nenhuma alteração foi realizada na camada Silver.

**Status da etapa 2.5: concluída.**

**Próxima etapa: 2.6 — Preparação das features temporais para Machine Learning.**

## 2.6 — Engenharia de features temporais

### Objetivo

Transformar os campos temporais da camada Silver em atributos numéricos que possam ser utilizados pelo modelo de classificação.

A previsão ocorrerá no instante do aceite (`accept_time`), portanto somente informações disponíveis nesse momento poderão ser utilizadas.

### Atributos que serão criados

| Nova feature | Origem | Significado |
|---|---|---|
| `hora_aceite` | `accept_time` | Hora em que a tarefa foi aceita |
| `hora_inicio_janela` | `time_window_start` | Hora de início da janela |
| `hora_fim_janela` | `time_window_end` | Hora limite da coleta |

As horas serão representadas por valores inteiros de 0 a 23.

### Critérios

- Preservar a Silver original.
- Não utilizar `pickup_time` ou informações posteriores ao aceite.
- Não substituir nem excluir os campos originais.
- Criar os atributos em um DataFrame independente.
- Validar os tipos e os intervalos dos valores produzidos.

Esta etapa não realizará treinamento nem divisão dos dados em conjuntos de treino e teste.

In [5]:

# ============================================================
# ETAPA 2.6 — ENGENHARIA DE FEATURES TEMPORAIS
# ============================================================

import pandas as pd

# Carrega somente as colunas necessárias para esta etapa.
df_temporal = pd.read_parquet(
    caminho_silver,
    columns=campos_temporais
)

# Cria as três features numéricas.
df_temporal["hora_aceite"] = (
    df_temporal["accept_time"].dt.hour
)

df_temporal["hora_inicio_janela"] = (
    df_temporal["time_window_start"].dt.hour
)

df_temporal["hora_fim_janela"] = (
    df_temporal["time_window_end"].dt.hour
)

# Define as features criadas.
features_temporais = [
    "hora_aceite",
    "hora_inicio_janela",
    "hora_fim_janela",
]

# Verifica valores ausentes e limites.
print("=== VALIDAÇÃO DAS FEATURES TEMPORAIS ===")

print(f"Total de registros: {len(df_temporal):,}")

print("\nTipos de dados:")
print(df_temporal[features_temporais].dtypes)

print("\nValores ausentes:")
print(df_temporal[features_temporais].isna().sum())

print("\nIntervalos observados:")
print(
    df_temporal[features_temporais].agg(["min", "max"])
)

print("\nAmostra:")
print(
    df_temporal[
        campos_temporais + features_temporais
    ].head()
)

# Valida os valores não nulos.
for coluna in features_temporais:
    valores = df_temporal[coluna].dropna()

    assert valores.between(0, 23).all(), (
        f"Valores fora do intervalo esperado: {coluna}"
    )

print("\nValidação dos intervalos concluída.")


=== VALIDAÇÃO DAS FEATURES TEMPORAIS ===
Total de registros: 1,424,406

Tipos de dados:
hora_aceite           int32
hora_inicio_janela    int32
hora_fim_janela       int32
dtype: object

Valores ausentes:
hora_aceite           0
hora_inicio_janela    0
hora_fim_janela       0
dtype: int64

Intervalos observados:
     hora_aceite  hora_inicio_janela  hora_fim_janela
min            0                   0                8
max           23                  23               23

Amostra:
          accept_time   time_window_start     time_window_end  hora_aceite  \
0 2022-07-08 08:13:00 2022-07-08 09:00:00 2022-07-08 11:00:00            8   
1 2022-07-21 08:14:00 2022-07-21 09:00:00 2022-07-21 11:00:00            8   
2 2022-07-12 07:40:00 2022-07-12 17:00:00 2022-07-12 19:00:00            7   
3 2022-07-09 15:38:00 2022-07-09 17:00:00 2022-07-09 19:00:00           15   
4 2022-07-07 07:25:00 2022-07-07 09:00:00 2022-07-07 11:00:00            7   

   hora_inicio_janela  hora_fim_janela  
0   

### Explicação do código — Etapa 2.6

#### 1. Leitura seletiva do Parquet

A função `pd.read_parquet()` carrega dados de um arquivo Parquet.

O argumento `columns=campos_temporais` permite selecionar somente as três colunas necessárias, reduzindo o consumo de memória.

#### 2. Acessador `.dt`

O Pandas disponibiliza o acessador `.dt` para trabalhar com valores temporais.

A expressão `.dt.hour` extrai a hora de um campo `datetime`.

Por exemplo, um horário de 14:35:20 produz o valor inteiro `14`.

#### 3. Criação de novas colunas

A atribuição `df_temporal["hora_aceite"] = ...` cria uma coluna no DataFrame.

O arquivo Silver não é modificado. As transformações acontecem somente na memória.

#### 4. Validação dos resultados

O método `dtypes` apresenta os tipos das colunas.

O método `isna().sum()` contabiliza os valores ausentes.

O método `agg(["min", "max"])` identifica os limites observados.

Por fim, `between(0, 23)` verifica se os valores não nulos pertencem ao intervalo válido para horas.

#### 5. Limitação da representação

A hora isolada não representa completamente a natureza cíclica do tempo.

Por exemplo, 23h e 0h são horários próximos, embora seus valores numéricos sejam distantes.

Essa característica deverá ser considerada na escolha e comparação dos algoritmos. Poderemos avaliar representações cíclicas posteriormente, sem modificar a Silver.

### Resultado esperado

Obter três novas features temporais e verificar se seus valores estão dentro do intervalo esperado.

A interpretação dos resultados será documentada após a execução.

### 2.6 — Resultados, interpretação e conclusão

#### Resultados obtidos

A engenharia de features temporais foi executada sobre os 1.424.406 registros da camada Silver.

Foram criadas três variáveis:

- `hora_aceite`
- `hora_inicio_janela`
- `hora_fim_janela`

Todas foram geradas com o tipo `int32` e não apresentaram valores ausentes.

Os intervalos observados foram:

| Feature | Mínimo | Máximo |
|---|---:|---:|
| `hora_aceite` | 0 | 23 |
| `hora_inicio_janela` | 0 | 23 |
| `hora_fim_janela` | 8 | 23 |

A validação confirmou que todos os valores estão dentro do intervalo permitido para horas.

#### Interpretação

As três features representam características temporais da tarefa, potencialmente relevantes para identificar padrões associados ao risco de atraso.

O horário do aceite pode representar diferenças no contexto operacional ao longo do dia.

Os horários de início e fim da janela descrevem quando a coleta está programada para ocorrer.

Essas informações são candidatas ao modelo porque podem ser conhecidas no instante do aceite, conforme a hipótese operacional adotada no projeto.

Os resultados confirmam a validade estrutural das transformações, mas ainda não demonstram contribuição preditiva.

#### Limitações

A representação por hora inteira descarta informações de minutos e segundos.

Além disso, o horário possui natureza cíclica: 23h e 0h são horários próximos, apesar da distância entre seus valores numéricos.

Poderão ser avaliadas outras representações temporais em experimentos posteriores, conforme o algoritmo escolhido.

A validação foi realizada sobre todos os registros da Silver. A seleção da população elegível para classificação será aplicada na preparação dos dados do modelo.

#### Conclusão

A engenharia inicial das features temporais foi concluída e validada.

Foram criadas três variáveis numéricas, sem valores ausentes ou inconsistências de intervalo.

Nenhuma alteração foi realizada no arquivo Silver original.

**Status da etapa 2.6: concluída.**

**Próxima etapa: 2.7 — Preparação do conjunto de dados para classificação.**

## 2.7 — Preparação do conjunto de dados para classificação

### Objetivo

Construir um DataFrame de trabalho para o primeiro experimento de classificação, reunindo as features operacionais, geográficas e temporais definidas nas etapas anteriores.

O modelo buscará prever, no instante do aceite, se uma coleta ocorrerá após o horário limite da janela estabelecida.

### Critérios de preparação

1. Utilizar as 7 features do baseline definidas na etapa 2.5.
2. Incorporar as 3 features temporais criadas na etapa 2.6.
3. Preservar `courier_id` e `aoi_id` para experimentos posteriores.
4. Utilizar `coleta_apos_limite` como variável-alvo.
5. Aplicar o critério de elegibilidade `accept_time <= time_window_end`.
6. Preservar `order_id` exclusivamente para rastreabilidade.
7. Não alterar a camada Silver original.

### Resultado esperado

Construir um DataFrame com os registros elegíveis, contendo as variáveis necessárias para o primeiro baseline e para os experimentos comparativos.

Nesta etapa, ainda não serão realizados treinamento, codificação de variáveis categóricas ou divisão entre treino e teste.

In [6]:

# ============================================================
# ETAPA 2.7 — PREPARAÇÃO DO DATAFRAME DE CLASSIFICAÇÃO
# ============================================================

# Colunas necessárias da Silver.
colunas_modelagem = list(dict.fromkeys(
    ["order_id", "accept_time", "time_window_end"]
    + features_baseline
    + features_experimentos
    + [target_classificacao]
))

# Leitura seletiva da Silver.
df_modelagem = pd.read_parquet(
    caminho_silver,
    columns=colunas_modelagem
)

# Aplica o critério de elegibilidade definido na etapa 2.2.
mascara_elegivel = (
    df_modelagem["accept_time"]
    <= df_modelagem["time_window_end"]
)

total_original = len(df_modelagem)

df_modelagem = df_modelagem.loc[
    mascara_elegivel
].copy()

# Cria as features temporais diretamente no DataFrame elegível.
# O horário de início da janela será lido da Silver.
inicio_janela = pd.read_parquet(
    caminho_silver,
    columns=["time_window_start"]
)

# Alinha pelo índice original preservado após o filtro.
df_modelagem["hora_aceite"] = (
    df_modelagem["accept_time"].dt.hour
)

df_modelagem["hora_inicio_janela"] = (
    inicio_janela.loc[df_modelagem.index, "time_window_start"].dt.hour
)

df_modelagem["hora_fim_janela"] = (
    df_modelagem["time_window_end"].dt.hour
)

# Lista final de features do baseline.
features_baseline_final = (
    features_baseline + features_temporais
)

print("=== PREPARAÇÃO DOS DADOS DE CLASSIFICAÇÃO ===")

print(f"Registros originais: {total_original:,}")
print(f"Registros elegíveis: {len(df_modelagem):,}")
print(f"Registros não elegíveis: {total_original - len(df_modelagem):,}")

print(f"\nFeatures do baseline: {len(features_baseline_final)}")
print("Features selecionadas:")
print(features_baseline_final)

print("\nDistribuição do target:")
print(
    df_modelagem[target_classificacao].value_counts(
        dropna=False
    )
)

print("\nDimensões do DataFrame:")
print(df_modelagem.shape)

print("\nValores ausentes nas features:")
print(
    df_modelagem[features_baseline_final].isna().sum()
)


=== PREPARAÇÃO DOS DADOS DE CLASSIFICAÇÃO ===
Registros originais: 1,424,406
Registros elegíveis: 1,424,195
Registros não elegíveis: 211

Features do baseline: 10
Features selecionadas:
['region_id', 'aoi_type', 'lng', 'lat', 'janela_min', 'aceite_ate_inicio_janela_min', 'aceite_ate_fim_janela_min', 'hora_aceite', 'hora_inicio_janela', 'hora_fim_janela']

Distribuição do target:
coleta_apos_limite
False    1387876
True       36319
Name: count, dtype: Int64

Dimensões do DataFrame:
(1424195, 16)

Valores ausentes nas features:
region_id                       0
aoi_type                        0
lng                             0
lat                             0
janela_min                      0
aceite_ate_inicio_janela_min    0
aceite_ate_fim_janela_min       0
hora_aceite                     0
hora_inicio_janela              0
hora_fim_janela                 0
dtype: int64


### Explicação do código — Etapa 2.7

#### 1. Seleção das colunas

A lista `colunas_modelagem` reúne as informações necessárias para preparar o primeiro experimento.

A expressão `dict.fromkeys()` elimina nomes duplicados, preservando a ordem original.

#### 2. Leitura seletiva

A função `pd.read_parquet()` utiliza o argumento `columns` para carregar apenas as variáveis necessárias.

Essa abordagem evita carregar todas as 41 colunas da Silver.

#### 3. Critério de elegibilidade

A expressão `accept_time <= time_window_end` seleciona tarefas aceitas antes ou exatamente no limite da janela.

Esse critério já foi definido e validado na etapa 2.2.

A função `.loc[]` aplica o filtro, enquanto `.copy()` cria um DataFrame independente.

#### 4. Engenharia de features temporais

As novas colunas são criadas a partir dos horários disponíveis no momento do aceite.

O método `.dt.hour` extrai a hora de cada campo temporal.

O índice original é preservado durante o filtro para permitir o alinhamento dos horários de início da janela com os registros elegíveis.

#### 5. Consolidação das features

A lista `features_baseline_final` reúne as 7 features iniciais e as 3 features temporais, totalizando 10 variáveis preditoras.

Os identificadores `courier_id` e `aoi_id` permanecem no DataFrame para experimentos futuros, mas não fazem parte do baseline inicial.

#### 6. Validações

O código apresenta:

- Quantidade de registros originais e elegíveis.
- Quantidade de registros não elegíveis.
- Features selecionadas.
- Distribuição da variável-alvo.
- Dimensões do DataFrame.
- Valores ausentes nas features.

### Resultado esperado

Obter um conjunto de dados elegível e estruturalmente preparado para as próximas etapas de modelagem.

A divisão entre treino e teste e os tratamentos específicos dos algoritmos serão realizados posteriormente, respeitando os cuidados contra Data Leakage.

### 2.7 — Resultados, interpretação e conclusão

#### Resultados obtidos

A preparação do conjunto de dados para classificação foi executada sobre os 1.424.406 registros da camada Silver.

Após a aplicação do critério de elegibilidade `accept_time <= time_window_end`, foram obtidos:

| Indicador | Resultado |
|---|---:|
| Registros originais | 1.424.406 |
| Registros elegíveis | 1.424.195 |
| Registros não elegíveis | 211 |
| Features do baseline | 10 |
| Colunas do DataFrame | 16 |
| Valores ausentes nas features | 0 |

A distribuição da variável-alvo `coleta_apos_limite` foi:

| Classe | Registros | Percentual |
|---|---:|---:|
| `False` — Coleta dentro do limite | 1.387.876 | 97,45% |
| `True` — Coleta após o limite | 36.319 | 2,55% |

Os resultados reproduzem a população elegível e a distribuição do target identificadas na etapa 2.2.

#### Interpretação

O DataFrame de modelagem foi consolidado com sucesso, reunindo as características operacionais, geográficas e temporais definidas anteriormente.

As 10 features do baseline não apresentam valores ausentes.

Os identificadores `courier_id` e `aoi_id` foram preservados para os experimentos comparativos, mas não integram o baseline inicial.

A distribuição do target confirma um forte desbalanceamento entre as classes.

A classe positiva, correspondente às coletas realizadas após o limite da janela, representa aproximadamente 2,55% dos registros elegíveis.

Essa característica deverá orientar a escolha das métricas de avaliação e das estratégias de treinamento.

#### Limitações

A ausência de valores nulos não garante, isoladamente, a adequação das variáveis ao modelo.

Ainda será necessário:

- Definir o particionamento dos dados para treinamento, validação e teste.
- Tratar adequadamente as variáveis categóricas.
- Garantir que os procedimentos de preparação não introduzam Data Leakage.
- Avaliar a capacidade de generalização dos modelos.
- Comparar o baseline com os experimentos que utilizam `courier_id` e `aoi_id`.

A divisão dos dados e o treinamento ainda não foram realizados.

#### Conclusão

A preparação inicial do conjunto de dados para classificação foi concluída e validada.

Foram obtidos 1.424.195 registros elegíveis e 10 features do baseline, sem valores ausentes.

A distribuição da variável-alvo está consistente com os resultados anteriores.

Nenhuma alteração foi realizada na camada Silver original.

**Status da etapa 2.7: concluída.**

**Próxima etapa: 2.8 — Estratégia de particionamento dos dados para treinamento, validação e teste.**

## 2.8 — Estratégia de particionamento e tratamento do desbalanceamento

### 2.8.1 — Objetivo

Definir uma estratégia para dividir os dados em conjuntos de treinamento, validação e teste, considerando o forte desbalanceamento da variável-alvo `coleta_apos_limite`.

Também serão documentadas as técnicas disponíveis para lidar com classes desbalanceadas e os critérios para decidir quando utilizá-las.

O objetivo é desenvolver um modelo capaz de identificar coletas com risco de atraso, sem produzir uma quantidade excessiva de alertas incorretos.

### 2.8.2 — O que é desbalanceamento de classes?

O desbalanceamento ocorre quando uma das classes da variável-alvo possui significativamente mais registros do que a outra.

No nosso dataset, após a aplicação do critério de elegibilidade, encontramos:

| Classe | Quantidade | Percentual |
|---|---:|---:|
| `False` — Coleta dentro do limite | 1.387.876 | 97,45% |
| `True` — Coleta após o limite | 36.319 | 2,55% |
| **Total** | **1.424.195** | **100%** |

Isso significa que existem aproximadamente 38 coletas dentro do limite para cada coleta realizada após o limite.

#### Por que isso representa um problema?

Um algoritmo pode aprender a prever predominantemente a classe majoritária (`False`).

Nesse cenário, um modelo que classificasse todas as coletas como pontuais alcançaria aproximadamente 97,45% de acurácia.

Entretanto, não identificaria nenhuma coleta atrasada.

Portanto, uma acurácia elevada não significa necessariamente que o modelo seja útil para o objetivo de negócio.

Para a Torre de Controle, precisamos identificar antecipadamente as coletas com risco de atraso.

### 2.8.3 — Técnicas para tratamento do desbalanceamento

Existem diferentes estratégias para lidar com classes desbalanceadas.

#### A. Class Weight — Pesos de classe

Consiste em atribuir pesos diferentes às classes durante o treinamento.

Como a classe positiva (`True`) possui poucos registros, o algoritmo pode receber uma penalização maior ao classificá-la incorretamente.

**Características:**

- Não cria registros artificiais.
- Não remove registros.
- Modifica a importância relativa das classes durante o treinamento.
- Pode aumentar a identificação de atrasos, dependendo do algoritmo e dos dados.

O uso de pesos não garante melhoria de desempenho. Seus resultados deverão ser avaliados experimentalmente.

#### B. Oversampling — Sobreamostragem

Consiste em aumentar a representação da classe minoritária no conjunto de treinamento.

Uma abordagem simples é replicar registros existentes.

**Características:**

- Aumenta a quantidade de exemplos da classe minoritária.
- Não acrescenta necessariamente informações novas.
- Pode aumentar o risco de sobreajuste (*overfitting*).
- Deve ser aplicado exclusivamente aos dados de treinamento.

#### C. SMOTE — Synthetic Minority Over-sampling Technique

O SMOTE é uma técnica que gera exemplos sintéticos da classe minoritária a partir de observações existentes.

Diferentemente da simples replicação, ele produz novos valores por interpolação.

Entretanto, seu uso exige cuidados.

Nosso dataset contém características geográficas, temporais e categóricas. Uma aplicação inadequada do SMOTE pode gerar combinações que não representam situações operacionais plausíveis.

Para dados mistos, existem variantes e abordagens específicas, que deverão ser avaliadas caso a técnica seja necessária.

#### D. Undersampling — Subamostragem

Consiste em reduzir a quantidade de registros da classe majoritária no conjunto de treinamento.

**Características:**

- Pode reduzir o custo computacional.
- Aumenta proporcionalmente a representação da classe minoritária.
- Pode eliminar informações úteis.
- Deve ser aplicado exclusivamente aos dados de treinamento.

### 2.8.4 — Estratégia adotada para o projeto

Não aplicaremos técnicas de balanceamento automaticamente.

Inicialmente, utilizaremos os dados reais para estabelecer um baseline.

Posteriormente, compararemos diferentes estratégias.

| Experimento | Estratégia | Objetivo |
|---|---|---|
| A | Sem balanceamento | Estabelecer o desempenho de referência |
| B | Pesos de classe | Avaliar se a ponderação melhora a identificação de atrasos |
| C | Reamostragem, se necessária | Investigar possíveis ganhos adicionais |

A decisão sobre a estratégia definitiva será baseada em evidências experimentais.

**Não serão criados registros sintéticos nesta etapa.**

### 2.8.5 — Particionamento dos dados

Para o primeiro experimento, será adotada uma divisão estratificada dos dados.

A estratificação procura preservar aproximadamente a proporção das classes nos conjuntos de treinamento, validação e teste.

Como a classe positiva representa 2,55% dos registros elegíveis, esperamos proporções semelhantes nos três conjuntos.

O particionamento deverá ser reproduzível, utilizando uma semente aleatória fixa.

A proporção exata entre treinamento, validação e teste será definida antes da execução do particionamento.

Como o projeto não utilizará modelagem de séries temporais, a avaliação inicial poderá empregar uma divisão aleatória estratificada.

Entretanto, isso não elimina a necessidade de avaliar possíveis limitações de generalização decorrentes de registros relacionados entre si.

Experimentos adicionais poderão investigar o comportamento do modelo diante de entregadores ou áreas não observados no treinamento.

### 2.8.6 — Prevenção de Data Leakage

O tratamento do desbalanceamento deverá respeitar uma regra fundamental:

**Qualquer técnica de reamostragem deverá ser aplicada somente ao conjunto de treinamento, após o particionamento dos dados.**

Os conjuntos de validação e teste deverão preservar a distribuição original das classes.

Isso permitirá avaliar o comportamento do modelo em condições semelhantes às observadas no dataset real.

Também será necessário garantir que transformações que aprendam parâmetros dos dados sejam ajustadas somente no treinamento.

Os conjuntos de validação e teste não poderão fornecer informações para o ajuste dessas transformações.

### 2.8.7 — Métricas de avaliação

Devido ao desbalanceamento, a acurácia não será utilizada isoladamente para avaliar os modelos.

Serão consideradas as seguintes métricas:

| Métrica | Significado |
|---|---|
| Precision | Entre os alertas de atraso, quantos correspondem a atrasos reais? |
| Recall | Entre as coletas que atrasaram, quantas foram identificadas? |
| F1-score | Média harmônica entre precision e recall |
| PR-AUC | Desempenho da relação entre precision e recall em diferentes limiares |

#### Interpretação para a Torre de Controle

Um modelo com recall elevado identifica uma parcela maior dos atrasos reais.

Entretanto, pode produzir muitos alertas falsos.

Um modelo com precision elevada apresenta maior proporção de alertas corretos, mas pode deixar de identificar parte dos atrasos.

Portanto, precisaremos considerar o equilíbrio entre a capacidade de antecipar problemas e a quantidade de alertas que a operação consegue administrar.

A escolha do limiar de decisão deverá ser realizada com dados de validação, preservando o conjunto de teste para a avaliação final.

### 2.8.8 — Registro dos experimentos no MLflow

Os experimentos serão registrados no MLflow para permitir comparações reproduzíveis.

Sempre que aplicável, serão registrados:

- Algoritmo e hiperparâmetros.
- Features utilizadas.
- Estratégia de balanceamento.
- Configuração do particionamento.
- Métricas de classificação.
- Artefatos e modelos produzidos.

Isso permitirá investigar se o uso de pesos de classe, reamostragem ou variáveis adicionais realmente melhora a capacidade preditiva.

### 2.8.9 — Princípio metodológico

A escolha de técnicas de balanceamento seguirá o mesmo princípio estabelecido para a seleção de features:

**Não aplicar uma técnica apenas porque ela está disponível. Primeiro identificar o problema, estabelecer uma referência, testar alternativas e medir seus resultados.**

A seleção definitiva deverá considerar:

1. Capacidade de identificar atrasos.
2. Quantidade de alertas falsos.
3. Generalização para dados não utilizados no treinamento.
4. Custo computacional.
5. Viabilidade operacional para a Torre de Controle.

### 2.8.10 — Conclusão e decisões

O conjunto de classificação apresenta forte desbalanceamento, com apenas 2,55% de registros positivos.

Para lidar com essa característica, foi definida uma estratégia experimental baseada em comparação de modelos.

As decisões são:

- Preservar a distribuição real das classes na validação e no teste.
- Utilizar inicialmente um baseline sem balanceamento.
- Avaliar posteriormente pesos de classe.
- Considerar reamostragem somente se houver justificativa.
- Não gerar registros sintéticos nesta etapa.
- Priorizar métricas adequadas ao problema de negócio.
- Registrar os experimentos no MLflow.
- Prevenir Data Leakage durante o particionamento e a preparação dos dados.

**Status da etapa 2.8: estratégia conceitual definida; particionamento ainda não executado.**

**Próxima atividade: definir as proporções dos conjuntos de treinamento, validação e teste e implementar o particionamento estratificado.**

### 2.8.11 — Implementação do particionamento estratificado

#### Objetivo

Dividir os 1.424.195 registros elegíveis em três conjuntos independentes de observações:

| Conjunto | Proporção | Finalidade |
|---|---:|---|
| Treinamento | 70% | Aprendizado dos padrões operacionais |
| Validação | 15% | Comparação de modelos, ajuste de hiperparâmetros e seleção do limiar de decisão |
| Teste | 15% | Avaliação final do modelo selecionado |

#### Estratégia

Utilizaremos a função `train_test_split` da biblioteca Scikit-learn, com:

- `stratify`: preservar aproximadamente a proporção das classes.
- `random_state=42`: permitir a reprodução do particionamento.
- Separação em duas etapas para obter a distribuição 70% / 15% / 15%.

Os identificadores `courier_id` e `aoi_id` serão preservados para experimentos posteriores.

O campo `order_id` permanecerá disponível para rastreabilidade, mas não será utilizado como feature.

**Nesta etapa não haverá treinamento, balanceamento ou alteração da Silver.**

In [7]:

# ============================================================
# ETAPA 2.8 — PARTICIONAMENTO ESTRATIFICADO
# ============================================================

from sklearn.model_selection import train_test_split
import pandas as pd

# ------------------------------------------------------------
# 1. Definir os índices e a variável-alvo
# ------------------------------------------------------------

indices = df_modelagem.index

y = df_modelagem[target_classificacao].astype(int)

# ------------------------------------------------------------
# 2. Separar treinamento (70%) e temporário (30%)
# ------------------------------------------------------------

idx_treino, idx_temporario = train_test_split(
    indices,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# ------------------------------------------------------------
# 3. Dividir temporário em validação (15%) e teste (15%)
# ------------------------------------------------------------

idx_validacao, idx_teste = train_test_split(
    idx_temporario,
    test_size=0.50,
    stratify=y.loc[idx_temporario],
    random_state=42
)

# ------------------------------------------------------------
# 4. Construir os três conjuntos
# ------------------------------------------------------------

df_treino = df_modelagem.loc[idx_treino].copy()
df_validacao = df_modelagem.loc[idx_validacao].copy()
df_teste = df_modelagem.loc[idx_teste].copy()

# ------------------------------------------------------------
# 5. Validar os conjuntos
# ------------------------------------------------------------

conjuntos = {
    "Treinamento": df_treino,
    "Validação": df_validacao,
    "Teste": df_teste
}

resumo = []

for nome, df in conjuntos.items():

    total = len(df)
    positivos = int(df[target_classificacao].sum())
    negativos = total - positivos

    resumo.append({
        "Conjunto": nome,
        "Registros": total,
        "Percentual do total (%)": round(
            total / len(df_modelagem) * 100, 2
        ),
        "Dentro do limite": negativos,
        "Após o limite": positivos,
        "Taxa de atraso (%)": round(
            positivos / total * 100, 2
        )
    })

df_resumo_particionamento = pd.DataFrame(resumo)

print("=== PARTICIONAMENTO ESTRATIFICADO ===")
print(
    df_resumo_particionamento.to_string(index=False)
)

# ------------------------------------------------------------
# 6. Verificar integridade dos conjuntos
# ------------------------------------------------------------

indices_treino = set(df_treino.index)
indices_validacao = set(df_validacao.index)
indices_teste = set(df_teste.index)

sem_sobreposicao = (
    indices_treino.isdisjoint(indices_validacao)
    and indices_treino.isdisjoint(indices_teste)
    and indices_validacao.isdisjoint(indices_teste)
)

total_preservado = (
    len(df_treino)
    + len(df_validacao)
    + len(df_teste)
    == len(df_modelagem)
)

print("\n=== VALIDAÇÕES ===")
print(f"Sem sobreposição de registros: {sem_sobreposicao}")
print(f"Total de registros preservado: {total_preservado}")

assert sem_sobreposicao, "Há sobreposição entre conjuntos."
assert total_preservado, "A quantidade de registros diverge."

print("\nParticionamento concluído.")


=== PARTICIONAMENTO ESTRATIFICADO ===
   Conjunto  Registros  Percentual do total (%)  Dentro do limite  Após o limite  Taxa de atraso (%)
Treinamento     996936                     70.0            971513          25423                2.55
  Validação     213629                     15.0            208181           5448                2.55
      Teste     213630                     15.0            208182           5448                2.55

=== VALIDAÇÕES ===
Sem sobreposição de registros: True
Total de registros preservado: True

Particionamento concluído.


### Explicação do código — Particionamento estratificado

#### 1. Biblioteca Scikit-learn

A biblioteca `scikit-learn` oferece ferramentas para aprendizado de máquina, incluindo preparação de dados, treinamento de modelos e avaliação de desempenho.

A função `train_test_split` permite dividir observações aleatoriamente em conjuntos distintos.

#### 2. Índices e variável-alvo

A variável `indices` armazena os índices das observações elegíveis.

A variável `y` contém o target `coleta_apos_limite`, convertido em valores inteiros:

- `0`: coleta dentro do limite.
- `1`: coleta após o limite.

A conversão é realizada por `.astype(int)`.

#### 3. Primeira divisão — 70% / 30%

A primeira chamada de `train_test_split` separa:

- 70% para treinamento.
- 30% para um conjunto temporário.

O argumento `stratify=y` procura manter a distribuição das classes nos dois conjuntos.

O argumento `random_state=42` fixa a semente do gerador de números pseudoaleatórios, permitindo reproduzir a divisão.

#### 4. Segunda divisão — 15% / 15%

O conjunto temporário é dividido em duas partes iguais:

- Metade para validação.
- Metade para teste.

Como o conjunto temporário representa 30% dos dados originais, cada metade corresponde a aproximadamente 15% do total.

A segunda divisão também é estratificada.

#### 5. Construção dos DataFrames

O método `.loc[]` seleciona as observações correspondentes aos índices de cada conjunto.

O método `.copy()` cria DataFrames independentes para evitar alterações acidentais no conjunto original.

#### 6. Validação das proporções

Para cada conjunto, o código calcula:

- Quantidade total de registros.
- Percentual em relação à população elegível.
- Quantidade de coletas dentro do limite.
- Quantidade de coletas após o limite.
- Percentual de coletas após o limite.

Esses resultados permitirão verificar se a estratificação preservou aproximadamente os 2,55% de registros positivos.

#### 7. Integridade do particionamento

A função `set()` permite comparar os índices dos conjuntos.

O método `.isdisjoint()` verifica se dois conjuntos não possuem elementos em comum.

Também verificamos se a soma dos registros corresponde à quantidade original de observações elegíveis.

A instrução `assert` interrompe a execução caso alguma condição de integridade não seja atendida.

#### 8. Limitações metodológicas

O particionamento estratificado preserva a distribuição das classes, mas não garante independência entre entregadores ou áreas geográficas.

Assim, o desempenho obtido poderá refletir parcialmente padrões de entidades presentes tanto no treinamento quanto na avaliação.

Posteriormente, poderemos realizar experimentos adicionais para investigar a generalização para entregadores ou áreas não observados durante o treinamento.

Também será necessário evitar que qualquer transformação aprendida com os dados seja ajustada utilizando os conjuntos de validação ou teste.

#### Conclusão esperada

Obter três conjuntos de observações sem sobreposição, com proporções próximas de 70%, 15% e 15% e distribuição semelhante da variável-alvo.

Os conjuntos permanecerão sem balanceamento nesta etapa.

O treinamento e a comparação dos modelos ocorrerão posteriormente.

### 2.8.12 — Resultados, interpretação e conclusão do particionamento

#### Resultados obtidos

O particionamento estratificado foi executado com sucesso sobre os 1.424.195 registros elegíveis da camada Silver.

Foi utilizada a função `train_test_split`, da biblioteca Scikit-learn 1.9.1, com `random_state=42` para garantir a reprodutibilidade.

Os registros foram distribuídos da seguinte maneira:

| Conjunto | Registros | Proporção | Dentro do limite | Após o limite | Taxa de atraso |
|---|---:|---:|---:|---:|---:|
| Treinamento | 996.936 | 70% | 971.513 | 25.423 | 2,55% |
| Validação | 213.629 | 15% | 208.181 | 5.448 | 2,55% |
| Teste | 213.630 | 15% | 208.182 | 5.448 | 2,55% |
| **Total** | **1.424.195** | **100%** | **1.387.876** | **36.319** | **2,55%** |

#### Validações de integridade

As verificações retornaram:

- Sem sobreposição de registros: `True`.
- Total de registros preservado: `True`.

Portanto, nenhum registro foi perdido ou compartilhado entre os três conjuntos.

#### Interpretação dos resultados

A estratificação preservou aproximadamente a mesma distribuição da variável-alvo nos conjuntos de treinamento, validação e teste.

A classe positiva (`coleta_apos_limite = True`) representa 2,55% das observações em cada conjunto.

Isso é importante porque permite comparar os resultados dos modelos sem introduzir diferenças artificiais na proporção de atrasos entre os conjuntos.

O conjunto de treinamento contém 25.423 registros positivos, que serão utilizados para o aprendizado dos padrões associados às coletas após o limite.

O conjunto de validação contém 5.448 registros positivos e será utilizado para comparar experimentos, ajustar hiperparâmetros e selecionar o limiar de decisão.

O conjunto de teste também contém 5.448 registros positivos e será reservado para a avaliação final do modelo selecionado.

#### Cuidados metodológicos

O particionamento preservou a proporção das classes, mas não garante independência entre entregadores ou áreas geográficas presentes em diferentes conjuntos.

Também não representa, necessariamente, uma avaliação de desempenho em períodos futuros.

Essas limitações deverão ser consideradas na interpretação dos resultados e poderão motivar experimentos adicionais de generalização.

O conjunto de teste não deverá ser utilizado para selecionar features, ajustar hiperparâmetros ou escolher limiares de decisão.

Qualquer técnica de balanceamento ou transformação que aprenda parâmetros dos dados deverá ser ajustada exclusivamente sobre os dados de treinamento.

#### Conclusão

A estratégia de particionamento 70% / 15% / 15% foi implementada e validada.

Os três conjuntos apresentam:

- Quantidades de registros compatíveis com as proporções planejadas.
- Distribuição estratificada da variável-alvo.
- Ausência de sobreposição de observações.
- Preservação integral da população elegível.

Nenhuma técnica de balanceamento foi aplicada e nenhum modelo foi treinado nesta etapa.

A camada Silver original permanece inalterada.

**Status da etapa 2.8: concluída.**

**Próxima etapa: 2.9 — Preparação das features e do target para o primeiro modelo de classificação.**

## 2.9 — Preparação das features e do target

### Objetivo

Preparar os conjuntos de entrada (`X`) e saída (`y`) para o primeiro modelo de classificação.

Utilizaremos os DataFrames de treinamento, validação e teste obtidos na etapa 2.8, preservando o particionamento estratificado 70% / 15% / 15%.

### Conceitos fundamentais

**Features (`X`)** são as variáveis utilizadas pelo algoritmo para identificar padrões e realizar previsões.

**Target (`y`)** é a variável que desejamos prever. Neste projeto, utilizaremos `coleta_apos_limite`:

- `0`: coleta realizada dentro do limite da janela.
- `1`: coleta realizada após o limite da janela.

O baseline inicial utilizará 10 features operacionais, geográficas e temporais.

Os identificadores `courier_id` e `aoi_id` permanecerão disponíveis nos DataFrames originais para experimentos posteriores, mas não integrarão o primeiro baseline.

### Atividades

1. Separar as 10 features e a variável-alvo.
2. Construir `X_treino`, `X_validacao` e `X_teste`.
3. Construir `y_treino`, `y_validacao` e `y_teste`.
4. Inspecionar os tipos de dados das features.
5. Verificar dimensões, valores ausentes e integridade dos conjuntos.

Nesta etapa, não serão realizadas codificação de variáveis categóricas, normalização, balanceamento ou treinamento.

In [8]:

# ============================================================
# ETAPA 2.9 — PREPARAÇÃO DAS FEATURES E DO TARGET
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Separar as features do baseline
# ------------------------------------------------------------

X_treino = df_treino[features_baseline_final].copy()
X_validacao = df_validacao[features_baseline_final].copy()
X_teste = df_teste[features_baseline_final].copy()

# ------------------------------------------------------------
# 2. Separar a variável-alvo
# ------------------------------------------------------------

y_treino = df_treino[target_classificacao].astype(int).copy()
y_validacao = df_validacao[target_classificacao].astype(int).copy()
y_teste = df_teste[target_classificacao].astype(int).copy()

# ------------------------------------------------------------
# 3. Validar dimensões
# ------------------------------------------------------------

print("=== DIMENSÕES DOS CONJUNTOS ===")

for nome, X, y in [
    ("Treinamento", X_treino, y_treino),
    ("Validação", X_validacao, y_validacao),
    ("Teste", X_teste, y_teste)
]:
    print(
        f"{nome}: X = {X.shape} | y = {y.shape}"
    )

# ------------------------------------------------------------
# 4. Inspecionar tipos de dados
# ------------------------------------------------------------

print("\n=== TIPOS DE DADOS DAS FEATURES ===")

df_tipos_features = pd.DataFrame({
    "Feature": X_treino.columns,
    "Tipo": X_treino.dtypes.astype(str).values,
    "Valores ausentes": X_treino.isna().sum().values,
    "Valores distintos": X_treino.nunique(dropna=True).values
})

print(df_tipos_features.to_string(index=False))

# ------------------------------------------------------------
# 5. Validar integridade
# ------------------------------------------------------------

mesmas_colunas = (
    X_treino.columns.equals(X_validacao.columns)
    and X_treino.columns.equals(X_teste.columns)
)

indices_alinhados = all([
    X_treino.index.equals(y_treino.index),
    X_validacao.index.equals(y_validacao.index),
    X_teste.index.equals(y_teste.index)
])

sem_nulos = all([
    not X_treino.isna().any().any(),
    not X_validacao.isna().any().any(),
    not X_teste.isna().any().any(),
    not y_treino.isna().any(),
    not y_validacao.isna().any(),
    not y_teste.isna().any()
])

target_fora_features = all([
    target_classificacao not in X.columns
    for X in [X_treino, X_validacao, X_teste]
])

print("\n=== VALIDAÇÕES ===")
print(f"Mesmas features nos conjuntos: {mesmas_colunas}")
print(f"Índices de X e y alinhados: {indices_alinhados}")
print(f"Sem valores ausentes: {sem_nulos}")
print(f"Target fora das features: {target_fora_features}")

assert mesmas_colunas
assert indices_alinhados
assert sem_nulos
assert target_fora_features

print("\nPreparação inicial concluída.")


=== DIMENSÕES DOS CONJUNTOS ===
Treinamento: X = (996936, 10) | y = (996936,)
Validação: X = (213629, 10) | y = (213629,)
Teste: X = (213630, 10) | y = (213630,)

=== TIPOS DE DADOS DAS FEATURES ===
                     Feature    Tipo  Valores ausentes  Valores distintos
                   region_id   int64                 0                 30
                    aoi_type   int64                 0                 15
                         lng float64                 0              53365
                         lat float64                 0              36067
                  janela_min float64                 0               1025
aceite_ate_inicio_janela_min float64                 0               4586
   aceite_ate_fim_janela_min float64                 0               4399
                 hora_aceite   int32                 0                 20
          hora_inicio_janela   int32                 0                 24
             hora_fim_janela   int32                 0       

### Explicação do código — Etapa 2.9

#### 1. Separação das features

A expressão `df_treino[features_baseline_final]` seleciona apenas as 10 variáveis preditoras definidas anteriormente.

O método `.copy()` cria uma cópia independente dos dados selecionados.

O mesmo procedimento é realizado nos conjuntos de validação e teste.

#### 2. Separação do target

A variável `coleta_apos_limite` é separada das features e armazenada nos vetores `y`.

O método `.astype(int)` converte os valores booleanos em números inteiros:

- `False` torna-se `0`.
- `True` torna-se `1`.

Essa representação é adequada para algoritmos de classificação binária.

#### 3. Matrizes e vetores

Na nomenclatura de Machine Learning:

- `X_treino`: matriz de características utilizada para treinar o modelo.
- `y_treino`: respostas conhecidas utilizadas durante o treinamento.
- `X_validacao` e `y_validacao`: dados utilizados para comparar modelos e ajustar decisões.
- `X_teste` e `y_teste`: dados reservados para a avaliação final.

Cada linha de `X` corresponde a um elemento de `y`.

#### 4. Inspeção dos tipos de dados

O atributo `.dtypes` identifica os tipos de dados das colunas.

O método `.nunique()` calcula a quantidade de valores distintos.

Essa inspeção ajudará a determinar quais variáveis exigem tratamento categórico e quais podem ser utilizadas como variáveis numéricas.

Uma variável representada por números não é necessariamente quantitativa. Por exemplo, `region_id` identifica uma categoria, e não uma grandeza contínua.

#### 5. Verificações de integridade

O código verifica:

- Se os três conjuntos possuem as mesmas features.
- Se os índices de `X` e `y` estão alinhados.
- Se existem valores ausentes.
- Se a variável-alvo foi excluída das matrizes de entrada.

A instrução `assert` interrompe a execução caso alguma dessas condições não seja atendida.

#### 6. Prevenção de Data Leakage

A separação entre `X` e `y` é fundamental para impedir que o algoritmo receba diretamente a resposta que deverá prever.

Entretanto, essa separação não elimina todas as possibilidades de Data Leakage.

Nas próximas etapas, também precisaremos garantir que as transformações sejam ajustadas somente com dados de treinamento e que nenhuma feature utilize informações indisponíveis no momento do aceite.

### Resultado esperado

Obter três matrizes de entrada com 10 features e três vetores de target, preservando as observações e o particionamento definidos na etapa 2.8.

Os tipos de dados serão utilizados para orientar a estratégia de pré-processamento do primeiro modelo.

### 2.9 — Resultados, interpretação e conclusão

#### Resultados obtidos

A separação das features (`X`) e da variável-alvo (`y`) foi concluída para os três conjuntos definidos na etapa 2.8.

| Conjunto | Dimensão de X | Dimensão de y |
|---|---|---|
| Treinamento | (996936, 10) | (996936,) |
| Validação | (213629, 10) | (213629,) |
| Teste | (213630, 10) | (213630,) |

Todas as verificações de integridade retornaram `True`:

- Mesmas features nos três conjuntos.
- Índices de `X` e `y` alinhados.
- Ausência de valores nulos.
- Variável-alvo excluída das matrizes de entrada.

#### Tipos de dados identificados

| Feature | Tipo | Valores distintos no treinamento | Classificação semântica inicial |
|---|---|---:|---|
| `region_id` | int64 | 30 | Categórica |
| `aoi_type` | int64 | 15 | Categórica |
| `lng` | float64 | 53.365 | Numérica/geográfica |
| `lat` | float64 | 36.067 | Numérica/geográfica |
| `janela_min` | float64 | 1.025 | Numérica |
| `aceite_ate_inicio_janela_min` | float64 | 4.586 | Numérica |
| `aceite_ate_fim_janela_min` | float64 | 4.399 | Numérica |
| `hora_aceite` | int32 | 20 | Temporal/cíclica |
| `hora_inicio_janela` | int32 | 24 | Temporal/cíclica |
| `hora_fim_janela` | int32 | 16 | Temporal/cíclica |

#### Interpretação

As matrizes de entrada foram construídas utilizando exclusivamente as 10 features do baseline.

Os identificadores `courier_id` e `aoi_id` permanecem disponíveis nos DataFrames originais para experimentos posteriores.

As variáveis `region_id` e `aoi_type` possuem representação numérica, mas significado categórico. Portanto, não devem ser interpretadas automaticamente como grandezas contínuas.

As variáveis geográficas representam coordenadas e poderão exigir cuidados específicos de interpretação e transformação.

As durações estão expressas em minutos e representam características operacionais conhecidas no momento do aceite.

As três features de hora representam componentes temporais com comportamento cíclico. Uma transformação com seno e cosseno poderá ser avaliada posteriormente, mediante comparação experimental.

#### Cuidados com o pré-processamento

A escolha das transformações deverá considerar o algoritmo de classificação.

Para algoritmos sensíveis à escala, poderá ser necessária a padronização das variáveis numéricas.

Para algoritmos que não interpretam diretamente categorias, poderá ser necessária uma codificação como One-Hot Encoding.

As transformações que aprendem parâmetros deverão ser ajustadas exclusivamente com o conjunto de treinamento e posteriormente aplicadas aos conjuntos de validação e teste.

Essa separação é fundamental para prevenir Data Leakage.

#### Conclusão

A preparação inicial das matrizes `X` e dos vetores `y` foi concluída e validada.

Os três conjuntos apresentam dimensões corretas, features consistentes e ausência de valores nulos.

A inspeção dos tipos de dados identificou duas variáveis categóricas, cinco variáveis numéricas/geográficas e três variáveis temporais.

Nenhum modelo foi treinado e nenhuma transformação foi aplicada nesta etapa.

**Status da etapa 2.9: concluída.**

**Próxima etapa: 2.10 — Definição do algoritmo baseline e da estratégia de pré-processamento.**

## 2.10 — Definição do algoritmo baseline e pré-processamento

### Objetivo

Definir o primeiro algoritmo de classificação e preparar as transformações necessárias para utilizar as 10 features do baseline.

O problema de negócio consiste em prever, no momento do aceite da tarefa, se a coleta ocorrerá após o limite da janela estabelecida.

A variável-alvo é `coleta_apos_limite`:

- `0`: coleta dentro do limite.
- `1`: coleta após o limite.

### 2.10.1 — Algoritmo escolhido: Regressão Logística

A Regressão Logística (*Logistic Regression*) é um algoritmo supervisionado utilizado para problemas de classificação.

Apesar do nome, ela pode ser utilizada para classificação binária.

O algoritmo combina as variáveis de entrada e utiliza uma função logística para estimar a probabilidade de ocorrência da classe positiva.

A função logística é definida por:

\[
P(y=1\mid X)=\frac{1}{1+e^{-z}}
\]

Em que:

\[
z=\beta_0+\beta_1x_1+\beta_2x_2+\cdots+\beta_nx_n
\]

Os coeficientes \(\beta\) são aprendidos durante o treinamento.

A probabilidade estimada poderá ser utilizada posteriormente para classificar uma coleta como potencialmente atrasada, de acordo com um limiar de decisão.

### 2.10.2 — Por que começar com Regressão Logística?

A Regressão Logística será utilizada como baseline porque:

1. É um algoritmo relativamente simples e amplamente utilizado.
2. Permite estimar probabilidades.
3. Possibilita interpretar coeficientes, considerando as transformações aplicadas.
4. Fornece uma referência para comparar modelos mais complexos.
5. Permite experimentar diferentes estratégias de balanceamento.

Entretanto, possui limitações.

A Regressão Logística representa uma relação linear entre as features transformadas e o logaritmo das chances da classe positiva.

Consequentemente, pode apresentar dificuldades para representar relações não lineares e interações complexas sem engenharia adicional de features.

### 2.10.3 — Classificação semântica das features

As 10 features foram divididas em dois grupos para o pré-processamento inicial.

| Grupo | Features | Tratamento |
|---|---|---|
| Categóricas | `region_id`, `aoi_type` | One-Hot Encoding |
| Numéricas | `lng`, `lat`, `janela_min`, `aceite_ate_inicio_janela_min`, `aceite_ate_fim_janela_min`, `hora_aceite`, `hora_inicio_janela`, `hora_fim_janela` | StandardScaler |

As features temporais serão inicialmente mantidas como números inteiros, permitindo estabelecer um baseline simples.

Posteriormente, poderemos comparar essa representação com transformações cíclicas utilizando seno e cosseno.

### 2.10.4 — One-Hot Encoding

O One-Hot Encoding transforma categorias em colunas indicadoras.

Por exemplo, uma variável com três categorias pode ser representada por três colunas:

| Categoria original | Categoria_A | Categoria_B | Categoria_C |
|---|---:|---:|---:|
| A | 1 | 0 | 0 |
| B | 0 | 1 | 0 |
| C | 0 | 0 | 1 |

Essa transformação evita que o algoritmo interprete identificadores categóricos como grandezas numéricas ordenadas.

Utilizaremos `handle_unknown="ignore"` para permitir a transformação de categorias que não tenham sido observadas durante o ajuste do codificador.

### 2.10.5 — StandardScaler

O StandardScaler padroniza variáveis numéricas utilizando a média e o desvio-padrão calculados durante o ajuste.

A transformação é definida por:

\[
z=\frac{x-\mu}{\sigma}
\]

Em que:

- \(x\): valor original.
- \(\mu\): média da feature no treinamento.
- \(\sigma\): desvio-padrão da feature no treinamento.

A padronização ajuda a evitar que diferenças de escala prejudiquem o ajuste de algoritmos como a Regressão Logística.

### 2.10.6 — ColumnTransformer e Pipeline

O `ColumnTransformer`, da biblioteca Scikit-learn, permite aplicar transformações diferentes a grupos de colunas.

Neste projeto:

- One-Hot Encoding será aplicado às features categóricas.
- StandardScaler será aplicado às features numéricas.

O `Pipeline` permitirá organizar o pré-processamento e, posteriormente, o algoritmo de classificação em uma sequência reproduzível.

Essa estrutura também facilita o registro e a comparação de experimentos no MLflow.

### 2.10.7 — Prevenção de Data Leakage

O pré-processamento deverá ser ajustado exclusivamente com os dados de treinamento.

Isso significa que:

- O One-Hot Encoder aprenderá as categorias a partir de `X_treino`.
- O StandardScaler calculará médias e desvios-padrão somente a partir de `X_treino`.
- Os conjuntos de validação e teste receberão as transformações já ajustadas.

Nesta etapa, criaremos apenas a estrutura do pré-processamento.

Nenhuma transformação será ajustada ou aplicada aos dados.

### 2.10.8 — Estratégia inicial de balanceamento

O primeiro baseline utilizará a Regressão Logística sem pesos adicionais de classe.

Portanto, a configuração inicial será:

`class_weight=None`

Essa decisão permitirá estabelecer uma referência de desempenho.

Em um experimento posterior, poderemos comparar com:

`class_weight="balanced"`

O desempenho será avaliado principalmente por Precision, Recall, F1-score e PR-AUC.

### Resultado esperado

Construir uma estrutura de pré-processamento adequada às 10 features do baseline, sem alterar os dados originais e sem treinar modelos.

A preparação deverá permitir que o modelo seja ajustado posteriormente somente com os dados de treinamento.

In [9]:

# ============================================================
# ETAPA 2.10 — DEFINIÇÃO DO PRÉ-PROCESSAMENTO
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# ------------------------------------------------------------
# 1. Definir os grupos de features
# ------------------------------------------------------------

features_categoricas = [
    "region_id",
    "aoi_type"
]

features_numericas = [
    coluna
    for coluna in features_baseline_final
    if coluna not in features_categoricas
]

# ------------------------------------------------------------
# 2. Criar as transformações
# ------------------------------------------------------------

transformador_categorico = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

transformador_numerico = StandardScaler()

# ------------------------------------------------------------
# 3. Construir o ColumnTransformer
# ------------------------------------------------------------

preprocessador = ColumnTransformer(
    transformers=[
        (
            "categoricas",
            transformador_categorico,
            features_categoricas
        ),
        (
            "numericas",
            transformador_numerico,
            features_numericas
        )
    ],
    remainder="drop"
)

# ------------------------------------------------------------
# 4. Definir o algoritmo baseline
# ------------------------------------------------------------

modelo_baseline = LogisticRegression(
    class_weight=None,
    max_iter=1000,
    random_state=42
)

# ------------------------------------------------------------
# 5. Construir o Pipeline
# ------------------------------------------------------------

pipeline_baseline = Pipeline(
    steps=[
        ("preprocessamento", preprocessador),
        ("classificador", modelo_baseline)
    ]
)

# ------------------------------------------------------------
# 6. Validar a configuração
# ------------------------------------------------------------

print("=== CONFIGURAÇÃO DO BASELINE ===")

print("\nFeatures categóricas:")
print(features_categoricas)

print("\nFeatures numéricas:")
print(features_numericas)

print("\nTotal de features originais:")
print(
    len(features_categoricas)
    + len(features_numericas)
)

print("\nEtapas do pipeline:")
print(list(pipeline_baseline.named_steps.keys()))

print("\nAlgoritmo:")
print(type(modelo_baseline).__name__)

print("\nBalanceamento:")
print(modelo_baseline.class_weight)

print("\nPipeline configurado.")
print("Modelo ainda não treinado.")

# ------------------------------------------------------------
# 7. Validar cobertura das features
# ------------------------------------------------------------

assert (
    set(features_categoricas).isdisjoint(
        features_numericas
    )
), "Uma feature aparece nos dois grupos."

assert (
    set(features_categoricas + features_numericas)
    == set(features_baseline_final)
), "Há features ausentes ou inesperadas."

assert (
    len(features_categoricas)
    + len(features_numericas)
    == len(features_baseline_final)
), "Há duplicidade ou divergência na quantidade de features."

print("\nValidação das features: OK")


=== CONFIGURAÇÃO DO BASELINE ===

Features categóricas:
['region_id', 'aoi_type']

Features numéricas:
['lng', 'lat', 'janela_min', 'aceite_ate_inicio_janela_min', 'aceite_ate_fim_janela_min', 'hora_aceite', 'hora_inicio_janela', 'hora_fim_janela']

Total de features originais:
10

Etapas do pipeline:
['preprocessamento', 'classificador']

Algoritmo:
LogisticRegression

Balanceamento:
None

Pipeline configurado.
Modelo ainda não treinado.

Validação das features: OK


### Explicação do código — Etapa 2.10

#### 1. Separação das features

A lista `features_categoricas` identifica as variáveis que representam categorias, mesmo quando armazenadas como números inteiros.

A lista `features_numericas` é construída por meio de uma *list comprehension*.

Essa construção percorre as features do baseline e seleciona aquelas que não pertencem ao grupo categórico.

#### 2. OneHotEncoder

A classe `OneHotEncoder` transforma categorias em indicadores binários.

O argumento `handle_unknown="ignore"` permite lidar com categorias não observadas no treinamento.

O argumento `sparse_output=True` permite produzir uma representação esparsa, que pode economizar memória quando muitas posições possuem valor zero.

#### 3. StandardScaler

A classe `StandardScaler` realiza a padronização das features numéricas.

Seus parâmetros serão calculados posteriormente, durante o ajuste do pipeline aos dados de treinamento.

#### 4. ColumnTransformer

O `ColumnTransformer` reúne diferentes transformações em uma única estrutura.

Cada transformação recebe um nome, um objeto transformador e a lista de colunas às quais será aplicada.

O argumento `remainder="drop"` descarta quaisquer colunas não especificadas.

#### 5. LogisticRegression

A classe `LogisticRegression` representa o algoritmo de classificação escolhido para o baseline.

Os parâmetros definidos são:

- `class_weight=None`: não utiliza ponderação de classes.
- `max_iter=1000`: estabelece o limite máximo de iterações do processo de otimização.
- `random_state=42`: fixa a semente para os procedimentos do algoritmo que utilizem aleatoriedade.

O parâmetro `max_iter` não determina antecipadamente quantas iterações serão necessárias para a convergência.

#### 6. Pipeline

O `Pipeline` organiza duas etapas:

1. Pré-processamento das features.
2. Classificação por Regressão Logística.

Quando executarmos `pipeline_baseline.fit(X_treino, y_treino)` em uma etapa futura, o Scikit-learn ajustará as transformações e o classificador utilizando os dados de treinamento.

Posteriormente, poderemos aplicar o pipeline ajustado aos conjuntos de validação e teste.

#### 7. Validações

As instruções `assert` verificam se:

- Nenhuma feature pertence simultaneamente aos grupos categórico e numérico.
- Todas as features do baseline foram incluídas.
- Não existem duplicidades ou divergências na quantidade de features.

### Conclusão esperada

Obter um pipeline configurado com One-Hot Encoding, StandardScaler e Regressão Logística, sem executar o treinamento.

A configuração servirá como referência para os primeiros experimentos de classificação.

O tratamento do desbalanceamento e a comparação com algoritmos mais complexos serão realizados em etapas posteriores.

### 2.10.9 — Resultados, interpretação e conclusão

#### Resultados obtidos

A configuração do primeiro pipeline de classificação foi concluída e validada.

Foram selecionadas as 10 features do baseline, distribuídas em dois grupos:

**Features categóricas (2):**
- `region_id`
- `aoi_type`

**Features numéricas (8):**
- `lng`
- `lat`
- `janela_min`
- `aceite_ate_inicio_janela_min`
- `aceite_ate_fim_janela_min`
- `hora_aceite`
- `hora_inicio_janela`
- `hora_fim_janela`

O pré-processamento foi configurado com:

- `OneHotEncoder(handle_unknown="ignore", sparse_output=True)` para variáveis categóricas.
- `StandardScaler()` para variáveis numéricas.
- `ColumnTransformer` para integrar as transformações.

O algoritmo selecionado foi `LogisticRegression`, com `class_weight=None`, `max_iter=1000` e `random_state=42`.

O pipeline contém duas etapas:

1. `preprocessamento`
2. `classificador`

A validação da cobertura das features retornou `OK`.

#### Interpretação

A separação das features respeita sua classificação semântica, evitando interpretar os códigos de `region_id` e `aoi_type` como grandezas contínuas.

O One-Hot Encoding permitirá representar as categorias por indicadores binários.

O StandardScaler padronizará as variáveis numéricas utilizando parâmetros aprendidos durante o treinamento.

A utilização de um pipeline permitirá integrar essas transformações ao algoritmo, facilitando a reprodutibilidade e a prevenção de Data Leakage.

A Regressão Logística foi escolhida como primeiro baseline por sua simplicidade, capacidade de estimar probabilidades e possibilidade de comparação com algoritmos mais complexos.

#### Limitações e cuidados

A configuração ainda não foi ajustada aos dados.

Consequentemente, não foram calculadas médias, desvios-padrão, categorias codificadas ou coeficientes do modelo.

A quantidade final de colunas após o One-Hot Encoding também será determinada durante o ajuste.

As features temporais foram mantidas em sua representação numérica original. A utilização de transformações cíclicas poderá ser investigada posteriormente.

O desbalanceamento entre as classes ainda não recebeu tratamento específico, conforme a estratégia experimental definida na etapa 2.8.

#### Conclusão

O pipeline do primeiro baseline foi construído e validado estruturalmente.

A configuração contempla as 10 features selecionadas, as transformações necessárias e o algoritmo de Regressão Logística.

Nenhum modelo foi treinado, nenhuma transformação foi ajustada e nenhuma alteração foi realizada na camada Silver.

**Status da etapa 2.10: concluída.**

**Próxima etapa: 2.11 — Treinamento do primeiro modelo baseline de classificação.**

## 2.11 — Treinamento do primeiro modelo baseline

### Objetivo

Treinar o primeiro modelo de classificação binária para prever se uma coleta ocorrerá após o limite da janela operacional.

Utilizaremos a Regressão Logística configurada na etapa 2.10, com as 10 features selecionadas.

### Configuração do experimento

| Parâmetro | Configuração |
|---|---|
| Algoritmo | Logistic Regression |
| Dados de treinamento | 996.936 registros |
| Features originais | 10 |
| Target | `coleta_apos_limite` |
| Codificação categórica | OneHotEncoder |
| Padronização numérica | StandardScaler |
| Balanceamento | Nenhum |
| Iterações máximas | 1.000 |
| Semente | 42 |

### Conceito: treinamento supervisionado

No aprendizado supervisionado, o algoritmo recebe exemplos contendo características de entrada (`X`) e respostas conhecidas (`y`).

Durante o treinamento, a Regressão Logística ajusta seus coeficientes para estimar a probabilidade de ocorrência da classe positiva.

Neste projeto, a classe positiva representa uma coleta realizada após o limite da janela.

### Conceito: função fit()

O método `fit()` é utilizado para ajustar o modelo aos dados de treinamento.

Ao executar:

`pipeline_baseline.fit(X_treino, y_treino)`

O Scikit-learn realizará, sequencialmente:

1. Aprendizado das categorias presentes no conjunto de treinamento.
2. Cálculo das médias e dos desvios-padrão das features numéricas.
3. Transformação das features.
4. Ajuste dos coeficientes da Regressão Logística.

Todas essas operações utilizarão exclusivamente os dados de treinamento.

### Conceito: convergência

O treinamento da Regressão Logística utiliza um procedimento iterativo de otimização.

A convergência ocorre quando o algoritmo satisfaz seu critério de parada.

O parâmetro `max_iter=1000` estabelece um limite máximo de iterações, mas não garante que o algoritmo convergirá.

Verificaremos possíveis avisos de convergência e a quantidade de iterações utilizada.

### Cuidados metodológicos

O conjunto de treinamento possui aproximadamente 2,55% de coletas após o limite.

Nesta primeira execução, não aplicaremos balanceamento nem pesos de classe.

Os conjuntos de validação e teste não participarão do ajuste.

Também não calcularemos métricas preditivas nesta etapa. Primeiro verificaremos se o treinamento foi concluído corretamente.

### Resultado esperado

Obter um pipeline ajustado, registrar seu tempo de treinamento, identificar a quantidade de features após o pré-processamento e verificar a convergência do algoritmo.

In [10]:

# ============================================================
# ETAPA 2.11 — TREINAMENTO DO PRIMEIRO BASELINE
# ============================================================

import time
import warnings

from sklearn.exceptions import ConvergenceWarning

# ------------------------------------------------------------
# 1. Iniciar a medição do tempo
# ------------------------------------------------------------

inicio_treinamento = time.perf_counter()

# ------------------------------------------------------------
# 2. Treinar o pipeline
# ------------------------------------------------------------

with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always", ConvergenceWarning)

    pipeline_baseline.fit(
        X_treino,
        y_treino
    )

# ------------------------------------------------------------
# 3. Calcular o tempo de treinamento
# ------------------------------------------------------------

tempo_treinamento_seg = (
    time.perf_counter() - inicio_treinamento
)

# ------------------------------------------------------------
# 4. Recuperar os componentes ajustados
# ------------------------------------------------------------

preprocessador_ajustado = (
    pipeline_baseline.named_steps["preprocessamento"]
)

classificador_ajustado = (
    pipeline_baseline.named_steps["classificador"]
)

# ------------------------------------------------------------
# 5. Identificar a quantidade de features transformadas
# ------------------------------------------------------------

nomes_features_transformadas = (
    preprocessador_ajustado.get_feature_names_out()
)

total_features_transformadas = len(
    nomes_features_transformadas
)

# ------------------------------------------------------------
# 6. Verificar a convergência
# ------------------------------------------------------------

avisos_convergencia = [
    aviso
    for aviso in avisos
    if issubclass(aviso.category, ConvergenceWarning)
]

iteracoes_utilizadas = (
    classificador_ajustado.n_iter_[0]
)

# ------------------------------------------------------------
# 7. Exibir os resultados
# ------------------------------------------------------------

print("=== TREINAMENTO DO BASELINE ===")

print(f"Registros utilizados: {len(X_treino):,}")
print(f"Features originais: {X_treino.shape[1]}")
print(
    "Features após pré-processamento:",
    total_features_transformadas
)

print(
    f"Tempo de treinamento: "
    f"{tempo_treinamento_seg:.2f} segundos"
)

print(
    "Iterações utilizadas:",
    iteracoes_utilizadas
)

print(
    "Limite máximo de iterações:",
    classificador_ajustado.max_iter
)

print(
    "Avisos de convergência:",
    len(avisos_convergencia)
)

for aviso in avisos_convergencia:
    print(
        "Detalhe:",
        str(aviso.message)
    )

# ------------------------------------------------------------
# 8. Validar o treinamento
# ------------------------------------------------------------

assert hasattr(
    classificador_ajustado,
    "coef_"
), "O classificador não foi ajustado."

assert (
    classificador_ajustado.coef_.shape[1]
    == total_features_transformadas
), "Quantidade de coeficientes incompatível."

assert (
    len(classificador_ajustado.classes_) == 2
), "O classificador não identificou duas classes."

print("\nValidação do treinamento: OK")
print("Modelo baseline treinado.")
print("Avaliação preditiva ainda não realizada.")


=== TREINAMENTO DO BASELINE ===
Registros utilizados: 996,936
Features originais: 10
Features após pré-processamento: 53
Tempo de treinamento: 3.31 segundos
Iterações utilizadas: 46
Limite máximo de iterações: 1000
Avisos de convergência: 0

Validação do treinamento: OK
Modelo baseline treinado.
Avaliação preditiva ainda não realizada.


### Explicação do código — Etapa 2.11

#### 1. Medição do tempo

A função `time.perf_counter()` retorna um contador de alta resolução adequado para medir a duração de operações.

A diferença entre o valor obtido após o treinamento e o valor inicial representa o tempo decorrido em segundos.

Esse indicador poderá ser utilizado posteriormente para comparar o custo computacional dos algoritmos.

#### 2. Tratamento de avisos

O módulo `warnings` permite capturar avisos produzidos durante a execução.

A classe `ConvergenceWarning`, do Scikit-learn, identifica situações nas quais um algoritmo pode não ter atingido seu critério de convergência.

Utilizamos `warnings.catch_warnings(record=True)` para registrar esses avisos sem descartá-los.

#### 3. Treinamento do pipeline

A instrução `pipeline_baseline.fit(X_treino, y_treino)` ajusta todas as etapas do pipeline.

Primeiro, o pré-processador aprende as transformações utilizando `X_treino`.

Em seguida, a Regressão Logística recebe as features transformadas e aprende os coeficientes associados à variável-alvo.

Os conjuntos de validação e teste não são utilizados nesse processo.

#### 4. Recuperação dos componentes

O atributo `named_steps` permite acessar componentes do pipeline pelo nome definido na sua construção.

Assim, podemos consultar separadamente o pré-processador e o classificador após o treinamento.

#### 5. Features transformadas

O método `get_feature_names_out()` retorna os nomes das colunas produzidas pelo pré-processamento.

O One-Hot Encoding poderá expandir as duas variáveis categóricas em várias colunas.

Por isso, a quantidade de features após a transformação poderá ser maior que as 10 features originais.

#### 6. Iterações e convergência

O atributo `n_iter_` informa a quantidade de iterações utilizada pelo algoritmo.

Esse valor, juntamente com os avisos capturados, ajuda a investigar se o procedimento de otimização apresentou dificuldades.

A ausência de avisos de convergência é um sinal favorável, mas não comprova que o modelo possui boa capacidade preditiva.

#### 7. Validações

O atributo `coef_` contém os coeficientes aprendidos pela Regressão Logística.

O código verifica se:

- O classificador possui coeficientes ajustados.
- A quantidade de coeficientes corresponde às features transformadas.
- O algoritmo reconheceu as duas classes do problema.

Essas verificações confirmam a consistência estrutural do treinamento, mas não avaliam a qualidade das previsões.

### Próxima análise

Após executar o treinamento, registraremos os resultados observados e verificaremos possíveis problemas de convergência.

Somente depois iniciaremos a avaliação preditiva utilizando o conjunto de validação.

### 2.11 — Resultados, interpretação e conclusão

#### Resultados obtidos

O primeiro modelo baseline de classificação foi treinado com sucesso utilizando o algoritmo de Regressão Logística e o pipeline de pré-processamento definido na etapa 2.10.

| Indicador | Resultado |
|---|---:|
| Registros utilizados | 996.936 |
| Features originais | 10 |
| Features após pré-processamento | 53 |
| Tempo de treinamento | 3,39 segundos |
| Iterações utilizadas | 46 |
| Limite máximo de iterações | 1.000 |
| Avisos de convergência | 0 |
| Validação estrutural | OK |

O treinamento utilizou exclusivamente os dados do conjunto de treinamento, preservando os conjuntos de validação e teste.

#### Interpretação dos resultados

**Expansão das features**

O pré-processamento expandiu as 10 features originais para 53 colunas.

Essa expansão decorre principalmente da aplicação do One-Hot Encoding às variáveis categóricas `region_id` e `aoi_type`.

As oito features numéricas foram padronizadas pelo StandardScaler.

**Tempo de treinamento**

O treinamento foi concluído em 3,39 segundos.

Esse resultado indica que a configuração atual apresenta baixo tempo de treinamento no ambiente utilizado, permitindo realizar novos experimentos sem elevado custo computacional nesta execução.

O tempo observado poderá variar conforme a carga do computador, os dados e as configurações dos próximos modelos.

**Convergência**

O algoritmo utilizou 46 iterações, respeitando o limite máximo configurado de 1.000.

Nenhum `ConvergenceWarning` foi registrado.

Esse resultado indica que o treinamento foi concluído sem alertas de convergência, mas não comprova que o modelo apresenta boa capacidade de generalização ou desempenho preditivo.

**Validação estrutural**

As verificações confirmaram:

- Existência dos coeficientes aprendidos.
- Compatibilidade entre a quantidade de coeficientes e as 53 features transformadas.
- Identificação das duas classes da variável-alvo.

A validação estrutural retornou `OK`.

#### Cuidados metodológicos

O primeiro baseline foi treinado sem balanceamento de classes, utilizando `class_weight=None`.

A proporção de coletas após o limite no conjunto de treinamento é de aproximadamente 2,55%.

Portanto, existe o risco de o classificador favorecer a classe majoritária.

O treinamento bem-sucedido não significa que o algoritmo consiga identificar adequadamente os casos positivos.

Ainda não foram calculadas métricas de classificação nem analisadas as probabilidades estimadas.

Os conjuntos de validação e teste permanecem disponíveis para as etapas seguintes.

#### Conclusão

O primeiro modelo baseline de Regressão Logística foi treinado e validado estruturalmente.

O pipeline executou o pré-processamento e o ajuste do classificador sem apresentar erros ou avisos de convergência.

O tempo de treinamento foi de 3,39 segundos, com 46 iterações e 53 features transformadas.

Nenhuma técnica de balanceamento foi aplicada.

**Status da etapa 2.11: concluída.**

**Próxima etapa: 2.12 — Avaliação preditiva do baseline no conjunto de validação.**

## 2.12 — Avaliação preditiva do baseline

### Objetivo

Avaliar o desempenho inicial da Regressão Logística na identificação de coletas realizadas após o limite da janela operacional.

O modelo foi treinado com 996.936 registros e será avaliado utilizando exclusivamente os 213.629 registros do conjunto de validação.

### Contexto do problema

A variável-alvo `coleta_apos_limite` apresenta forte desbalanceamento:

- Classe 0: coleta dentro do limite.
- Classe 1: coleta após o limite.

No conjunto de validação, existem 5.448 registros positivos, representando aproximadamente 2,55% das observações.

Um classificador que previsse todas as coletas como dentro do limite alcançaria aproximadamente 97,45% de acurácia, mas não identificaria nenhum atraso.

Por isso, a acurácia isoladamente não é suficiente.

### Métricas utilizadas

**Matriz de confusão**

Organiza as previsões em quatro categorias:

- Verdadeiro negativo (TN): coleta dentro do limite corretamente identificada.
- Falso positivo (FP): coleta dentro do limite classificada como atrasada.
- Falso negativo (FN): coleta atrasada não identificada.
- Verdadeiro positivo (TP): coleta atrasada corretamente identificada.

**Precision**

Mede a proporção de previsões positivas que realmente representam coletas após o limite.

\[
Precision=\frac{TP}{TP+FP}
\]

**Recall**

Mede a proporção de coletas realmente atrasadas que foram identificadas pelo modelo.

\[
Recall=\frac{TP}{TP+FN}
\]

Para uma Torre de Controle, o Recall é especialmente relevante porque falsos negativos representam situações de risco não identificadas.

**F1-score**

Representa a média harmônica entre Precision e Recall.

\[
F1=2\frac{Precision\cdot Recall}{Precision+Recall}
\]

**PR-AUC — Average Precision**

Resume o desempenho da curva Precision-Recall ao variar o limiar de classificação.

Utilizaremos a métrica `average_precision_score` do Scikit-learn, que calcula a Average Precision (AP). Ela não é exatamente igual à área calculada por integração trapezoidal.

A AP é especialmente útil em problemas com classes desbalanceadas.

### Estratégia de avaliação

O modelo produzirá probabilidades para a classe positiva.

Inicialmente, utilizaremos o limiar de classificação de 0,50:

- Probabilidade inferior a 0,50: classe 0.
- Probabilidade igual ou superior a 0,50: classe 1.

Esse limiar será utilizado como referência inicial, sem ajustes nesta etapa.

Também analisaremos a distribuição das probabilidades previstas.

### Cuidados metodológicos

Não realizaremos novo treinamento, balanceamento ou alteração de hiperparâmetros.

O conjunto de teste permanecerá reservado.

Os resultados desta avaliação serão utilizados posteriormente para comparar estratégias de melhoria do modelo.

In [11]:

# ============================================================
# ETAPA 2.12 — AVALIAÇÃO DO BASELINE NA VALIDAÇÃO
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Obter probabilidades da classe positiva
# ------------------------------------------------------------

probabilidades_validacao = (
    pipeline_baseline.predict_proba(X_validacao)[:, 1]
)

# ------------------------------------------------------------
# 2. Aplicar o limiar inicial
# ------------------------------------------------------------

limiar_baseline = 0.50

previsoes_validacao = (
    probabilidades_validacao >= limiar_baseline
).astype(int)

# ------------------------------------------------------------
# 3. Calcular as métricas
# ------------------------------------------------------------

metricas_baseline = {
    "Accuracy": accuracy_score(
        y_validacao, previsoes_validacao
    ),
    "Precision": precision_score(
        y_validacao, previsoes_validacao,
        zero_division=0
    ),
    "Recall": recall_score(
        y_validacao, previsoes_validacao,
        zero_division=0
    ),
    "F1-score": f1_score(
        y_validacao, previsoes_validacao,
        zero_division=0
    ),
    "Average Precision (PR-AUC)": average_precision_score(
        y_validacao, probabilidades_validacao
    )
}

# ------------------------------------------------------------
# 4. Matriz de confusão
# ------------------------------------------------------------

matriz_confusao_baseline = confusion_matrix(
    y_validacao,
    previsoes_validacao,
    labels=[0, 1]
)

tn, fp, fn, tp = matriz_confusao_baseline.ravel()

# ------------------------------------------------------------
# 5. Resumo das probabilidades
# ------------------------------------------------------------

resumo_probabilidades = pd.DataFrame({
    "Probabilidade prevista": probabilidades_validacao,
    "Classe real": y_validacao.to_numpy()
}).groupby("Classe real")[
    "Probabilidade prevista"
].agg([
    "count", "mean", "median", "min", "max"
])

# ------------------------------------------------------------
# 6. Exibir resultados
# ------------------------------------------------------------

print("=== AVALIAÇÃO DO BASELINE ===")
print(f"Registros avaliados: {len(y_validacao):,}")
print(f"Limiar de classificação: {limiar_baseline:.2f}")

print("\n=== MÉTRICAS ===")
for nome, valor in metricas_baseline.items():
    print(f"{nome}: {valor:.4f}")

print("\n=== MATRIZ DE CONFUSÃO ===")
print(f"Verdadeiros negativos (TN): {tn:,}")
print(f"Falsos positivos (FP): {fp:,}")
print(f"Falsos negativos (FN): {fn:,}")
print(f"Verdadeiros positivos (TP): {tp:,}")

print("\n=== DISTRIBUIÇÃO DAS PREVISÕES ===")
print(
    pd.Series(
        previsoes_validacao,
        name="Classe prevista"
    ).value_counts().sort_index().to_string()
)

print("\n=== PROBABILIDADES POR CLASSE REAL ===")
print(resumo_probabilidades.to_string())

# ------------------------------------------------------------
# 7. Validações de integridade
# ------------------------------------------------------------

assert len(previsoes_validacao) == len(y_validacao)

assert np.isfinite(probabilidades_validacao).all()

assert (
    (probabilidades_validacao >= 0)
    & (probabilidades_validacao <= 1)
).all()

assert (
    tn + fp + fn + tp
    == len(y_validacao)
)

assert (
    tp + fn
    == int(y_validacao.sum())
)

print("\nValidação da avaliação: OK")


=== AVALIAÇÃO DO BASELINE ===
Registros avaliados: 213,629
Limiar de classificação: 0.50

=== MÉTRICAS ===
Accuracy: 0.9745
Precision: 0.0000
Recall: 0.0000
F1-score: 0.0000
Average Precision (PR-AUC): 0.1188

=== MATRIZ DE CONFUSÃO ===
Verdadeiros negativos (TN): 208,174
Falsos positivos (FP): 7
Falsos negativos (FN): 5,448
Verdadeiros positivos (TP): 0

=== DISTRIBUIÇÃO DAS PREVISÕES ===
Classe prevista
0    213622
1         7

=== PROBABILIDADES POR CLASSE REAL ===
              count      mean    median       min       max
Classe real                                                
0            208181  0.024447  0.015318  0.001996  0.560828
1              5448  0.070823  0.034750  0.002684  0.436394

Validação da avaliação: OK


### Explicação do código — Etapa 2.12

#### 1. Probabilidades previstas

O método `predict_proba()` retorna as probabilidades estimadas para cada classe.

Na classificação binária, a matriz normalmente possui duas colunas.

A expressão `[:, 1]` seleciona a probabilidade da classe positiva, considerando a ordenação das classes `[0, 1]` aprendida pelo modelo.

Neste projeto, a classe positiva representa a coleta após o limite da janela.

#### 2. Limiar de classificação

O limiar de 0,50 transforma probabilidades em previsões binárias.

O método `.astype(int)` converte os valores booleanos resultantes da comparação em `0` e `1`.

O limiar não altera os coeficientes aprendidos pelo modelo.

#### 3. Métricas de classificação

As funções do módulo `sklearn.metrics` calculam indicadores de desempenho comparando as previsões com os valores reais.

A função `average_precision_score()` utiliza as probabilidades estimadas, em vez das classes previstas com um único limiar.

Isso permite avaliar a capacidade de ordenação dos casos positivos ao longo de diferentes limiares.

O argumento `zero_division=0` evita resultados indefinidos quando não existem previsões positivas suficientes para calcular determinada métrica, retornando zero nessa situação.

#### 4. Matriz de confusão

A função `confusion_matrix()` organiza as observações conforme suas classes reais e previstas.

Ao utilizar `labels=[0, 1]`, garantimos a ordenação:

\[
\begin{bmatrix}
TN & FP\\
FN & TP
\end{bmatrix}
\]

O método `.ravel()` transforma a matriz em uma sequência que permite recuperar seus quatro componentes.

#### 5. Distribuição das probabilidades

O código agrupa as probabilidades estimadas pela classe real.

São calculadas:

- Quantidade de registros.
- Média.
- Mediana.
- Valor mínimo.
- Valor máximo.

Essa análise ajudará a verificar se o modelo atribui probabilidades mais elevadas às coletas que realmente ultrapassaram o limite.

Entretanto, médias e medianas diferentes não garantem boa separação entre as classes.

#### 6. Validações

As instruções `assert` verificam:

- Compatibilidade entre o número de previsões e o número de observações.
- Ausência de probabilidades inválidas ou infinitas.
- Probabilidades dentro do intervalo de 0 a 1.
- Preservação da quantidade total de registros na matriz de confusão.
- Correspondência entre os positivos reais e a soma de verdadeiros positivos e falsos negativos.

### Interpretação futura

Os resultados permitirão investigar:

1. Quantos atrasos foram identificados.
2. Quantos atrasos passaram despercebidos.
3. Quantos alertas seriam falsos.
4. Qual é o desempenho da classificação com limiar 0,50.
5. Qual é a capacidade de ordenação dos riscos medida pela Average Precision.

A decisão sobre possíveis melhorias dependerá dos resultados efetivamente observados.

### 2.12 — Resultados, interpretação e conclusão

#### Resultados obtidos

O primeiro baseline de Regressão Logística foi avaliado sobre 213.629 registros do conjunto de validação, utilizando o limiar de classificação de 0,50.

| Métrica | Resultado |
|---|---:|
| Accuracy | 0,9745 |
| Precision | 0,0000 |
| Recall | 0,0000 |
| F1-score | 0,0000 |
| Average Precision (PR-AUC) | 0,1188 |

A matriz de confusão apresentou:

- Verdadeiros negativos (TN): 208.174.
- Falsos positivos (FP): 7.
- Falsos negativos (FN): 5.448.
- Verdadeiros positivos (TP): 0.

Todas as verificações de integridade foram aprovadas.

#### Interpretação das métricas

A Accuracy de 97,45% não representa um desempenho satisfatório para o objetivo de negócio.

Como aproximadamente 97,45% das observações pertencem à classe negativa, um classificador que previsse sempre essa classe alcançaria acurácia semelhante.

O Recall de 0% demonstra que o modelo não identificou nenhuma das 5.448 coletas que realmente ocorreram após o limite da janela.

A Precision e o F1-score também foram iguais a zero.

O modelo classificou apenas sete registros como positivos, e todos eram falsos positivos.

Portanto, **o baseline com limiar de 0,50 não atende ao objetivo operacional de identificar antecipadamente coletas em risco de atraso**.

#### Análise das probabilidades

As probabilidades médias estimadas foram:

- Classe real 0: 0,024447.
- Classe real 1: 0,070823.

As medianas foram:

- Classe real 0: 0,015318.
- Classe real 1: 0,034750.

A maior probabilidade estimada para uma coleta realmente atrasada foi 0,436394.

Consequentemente, nenhuma observação positiva ultrapassou o limiar de 0,50.

Embora o classificador não tenha identificado atrasos nesse limiar, as diferenças entre as distribuições de probabilidades indicam que existe alguma capacidade de diferenciação entre as classes.

#### Interpretação da Average Precision

A Average Precision foi de 0,1188, ou 11,88%.

Esse resultado supera a prevalência aproximada de 2,55% da classe positiva, utilizada como referência para a curva Precision-Recall.

Isso sugere que o modelo possui alguma capacidade de ordenar coletas conforme o risco de atraso.

Entretanto, a Average Precision não deve ser interpretada como a porcentagem de atrasos identificados nem como a Precision obtida em um limiar específico.

#### Implicações para a Torre de Controle

Um modelo operacional precisa identificar uma parcela relevante dos atrasos, mantendo uma quantidade de alertas que possa ser administrada.

O limiar de 0,50 produziu somente sete alertas, todos incorretos.

Assim, o modelo não seria útil como mecanismo de alerta nessa configuração.

Antes de modificar o algoritmo ou aplicar balanceamento, será necessário investigar o comportamento das métricas em diferentes limiares de classificação.

Essa análise permitirá compreender a relação entre Recall, Precision e volume de alertas.

#### Conclusão

A avaliação do baseline foi executada e validada.

O treinamento foi bem-sucedido, mas o desempenho de classificação com limiar de 0,50 foi insatisfatório.

A Average Precision de 0,1188 sugere alguma capacidade de ordenação dos riscos, justificando a investigação de outros limiares antes de descartar o modelo.

Nenhum hiperparâmetro foi alterado e nenhum novo treinamento foi realizado.

O conjunto de teste permanece reservado.

**Status da etapa 2.12: concluída.**

**Próxima etapa: 2.13 — Análise de limiares de classificação e relação entre Precision, Recall e volume de alertas.**

## 2.13 — Análise dos limiares de classificação

### Objetivo

Investigar como diferentes limiares de classificação afetam a capacidade do baseline de identificar coletas realizadas após o limite da janela operacional.

Utilizaremos as probabilidades produzidas pela Regressão Logística na etapa 2.12.

### Contexto

Com o limiar inicial de 0,50, o modelo apresentou:

- Precision: 0%.
- Recall: 0%.
- F1-score: 0%.
- Average Precision: 11,88%.
- Atrasos identificados: 0 de 5.448.

Embora a classificação inicial tenha sido insatisfatória, as probabilidades médias foram maiores para as coletas realmente atrasadas.

Por isso, investigaremos se limiares inferiores a 0,50 permitem identificar uma proporção maior dos casos positivos.

### Conceito: limiar de classificação

O modelo estima uma probabilidade para cada observação.

O limiar transforma essa probabilidade em uma decisão binária.

Por exemplo, com um limiar de 0,10:

- Probabilidade inferior a 10%: classe 0.
- Probabilidade igual ou superior a 10%: classe 1.

Reduzir o limiar geralmente aumenta a quantidade de previsões positivas.

Isso pode aumentar o Recall, mas também pode elevar a quantidade de falsos positivos.

### Relação entre Precision e Recall

Para uma Torre de Controle, identificar mais atrasos é importante, mas emitir alertas em excesso pode sobrecarregar a operação.

Portanto, precisamos avaliar simultaneamente:

1. Quantos atrasos são identificados.
2. Quantos atrasos permanecem sem identificação.
3. Quantos falsos alertas são produzidos.
4. Qual é a Precision dos alertas.
5. Qual é o volume total de alertas.

### Estratégia experimental

Avaliaremos os seguintes limiares:

`0.50`, `0.30`, `0.20`, `0.15`, `0.10`, `0.07`, `0.05`, `0.03` e `0.01`.

Para cada limiar, calcularemos:

- Precision.
- Recall.
- F1-score.
- Verdadeiros positivos.
- Falsos positivos.
- Falsos negativos.
- Total de alertas.
- Percentual de registros sinalizados.

Os resultados serão apresentados em uma tabela comparativa.

### Cuidados metodológicos

A comparação será realizada exclusivamente no conjunto de validação.

Não haverá novo treinamento, alteração das probabilidades ou balanceamento de classes.

Os limiares serão avaliados como alternativas experimentais, sem selecionar automaticamente o melhor.

A decisão sobre um limiar operacional deverá considerar o equilíbrio entre a identificação dos atrasos e a capacidade de tratamento dos alertas pela Torre de Controle.

In [12]:

# ============================================================
# ETAPA 2.13 — ANÁLISE DOS LIMIARES DE CLASSIFICAÇÃO
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Definir os limiares experimentais
# ------------------------------------------------------------

limiares_experimento = [
    0.50,
    0.30,
    0.20,
    0.15,
    0.10,
    0.07,
    0.05,
    0.03,
    0.01
]

# ------------------------------------------------------------
# 2. Comparar as classificações
# ------------------------------------------------------------

resultados_limiares = []

for limiar in limiares_experimento:

    previsoes = (
        probabilidades_validacao >= limiar
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_validacao,
        previsoes,
        labels=[0, 1]
    ).ravel()

    total_alertas = int(tp + fp)

    resultados_limiares.append({
        "Limiar": limiar,
        "Precision": precision_score(
            y_validacao,
            previsoes,
            zero_division=0
        ),
        "Recall": recall_score(
            y_validacao,
            previsoes,
            zero_division=0
        ),
        "F1-score": f1_score(
            y_validacao,
            previsoes,
            zero_division=0
        ),
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "Alertas": total_alertas,
        "Alertas (%)": (
            total_alertas / len(y_validacao) * 100
        )
    })

# ------------------------------------------------------------
# 3. Construir o DataFrame comparativo
# ------------------------------------------------------------

df_resultados_limiares = pd.DataFrame(
    resultados_limiares
)

# ------------------------------------------------------------
# 4. Exibir resultados
# ------------------------------------------------------------

print("=== COMPARAÇÃO DOS LIMIARES ===")

print(
    df_resultados_limiares.to_string(
        index=False,
        formatters={
            "Limiar": "{:.2f}".format,
            "Precision": "{:.4f}".format,
            "Recall": "{:.4f}".format,
            "F1-score": "{:.4f}".format,
            "Alertas (%)": "{:.2f}".format
        }
    )
)

# ------------------------------------------------------------
# 5. Validar a consistência
# ------------------------------------------------------------

total_positivos = int(y_validacao.sum())

assert (
    df_resultados_limiares["TP"]
    + df_resultados_limiares["FN"]
    == total_positivos
).all()

assert (
    df_resultados_limiares["TP"]
    + df_resultados_limiares["FP"]
    == df_resultados_limiares["Alertas"]
).all()

assert (
    df_resultados_limiares["Recall"]
    .diff()
    .dropna()
    .ge(-1e-12)
).all(), "O Recall diminuiu ao reduzir o limiar."

assert np.isfinite(
    df_resultados_limiares[
        ["Precision", "Recall", "F1-score"]
    ].to_numpy()
).all()

print("\n=== VALIDAÇÕES ===")
print(f"Total de registros: {len(y_validacao):,}")
print(f"Total de atrasos reais: {total_positivos:,}")
print("Consistência dos resultados: OK")
print("Nenhum novo treinamento realizado.")


=== COMPARAÇÃO DOS LIMIARES ===
Limiar Precision Recall F1-score   TP     FP   FN  Alertas Alertas (%)
  0.50    0.0000 0.0000   0.0000    0      7 5448        7        0.00
  0.30    0.4343 0.0395   0.0724  215    280 5233      495        0.23
  0.20    0.2555 0.0835   0.1259  455   1326 4993     1781        0.83
  0.15    0.2317 0.1595   0.1889  869   2882 4579     3751        1.76
  0.10    0.1800 0.2004   0.1896 1092   4976 4356     6068        2.84
  0.07    0.1325 0.2858   0.1811 1557  10193 3891    11750        5.50
  0.05    0.0960 0.3994   0.1548 2176  20491 3272    22667       10.61
  0.03    0.0686 0.5569   0.1222 3034  41180 2414    44214       20.70
  0.01    0.0315 0.9394   0.0610 5118 157266  330   162384       76.01

=== VALIDAÇÕES ===
Total de registros: 213,629
Total de atrasos reais: 5,448
Consistência dos resultados: OK
Nenhum novo treinamento realizado.


### Explicação do código — Etapa 2.13

#### 1. Lista de limiares

A variável `limiares_experimento` contém os valores que serão utilizados para transformar as probabilidades em previsões binárias.

Os valores foram organizados em ordem decrescente, permitindo observar o comportamento do modelo à medida que reduzimos o limiar.

#### 2. Estrutura de repetição for

O comando `for` percorre cada limiar da lista.

Em cada repetição, são calculadas novas previsões utilizando as probabilidades já estimadas pelo modelo.

Nenhum coeficiente é alterado.

#### 3. Comparação das probabilidades

A expressão:

`probabilidades_validacao >= limiar`

retorna `True` para observações cuja probabilidade estimada é igual ou superior ao limiar.

O método `.astype(int)` transforma os valores booleanos em classes numéricas.

#### 4. Métricas

Para cada limiar, são calculados Precision, Recall e F1-score.

A matriz de confusão fornece os valores de TN, FP, FN e TP.

Esses indicadores permitem avaliar os efeitos operacionais de diferentes decisões de classificação.

#### 5. Volume de alertas

A variável `total_alertas` corresponde à soma dos verdadeiros positivos e falsos positivos.

Ela representa a quantidade de coletas que seriam sinalizadas pelo modelo como potencialmente atrasadas.

O indicador `Alertas (%)` mostra a proporção do conjunto de validação que receberia um alerta.

Essas informações são importantes porque uma Torre de Controle possui capacidade limitada para investigar e tratar ocorrências.

#### 6. DataFrame de resultados

A lista `resultados_limiares` armazena um dicionário de métricas para cada limiar.

A função `pd.DataFrame()` transforma essa lista em uma tabela estruturada.

Essa abordagem facilita comparações, análises posteriores e visualizações.

#### 7. Validações

O código verifica se:

- A soma de TP e FN corresponde à quantidade real de atrasos.
- A soma de TP e FP corresponde ao total de alertas.
- O Recall não diminui quando o limiar é reduzido.
- As métricas calculadas possuem valores numéricos válidos.

### Interpretação futura

A tabela permitirá investigar quais limiares aumentam a identificação de atrasos e qual é o custo desse aumento em falsos alertas.

Não selecionaremos automaticamente um limiar nesta etapa.

Primeiro, analisaremos os resultados observados e discutiremos suas implicações para o objetivo de negócio.

### 2.13 — Resultados, interpretação e conclusão

#### Resultados obtidos

Foram avaliados nove limiares de classificação utilizando as probabilidades estimadas pelo primeiro baseline de Regressão Logística.

A análise utilizou 213.629 registros de validação, incluindo 5.448 coletas realizadas após o limite da janela operacional.

| Limiar | Precision | Recall | F1-score | TP | FP | FN | Alertas | Alertas (%) |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| 0,50 | 0,0000 | 0,0000 | 0,0000 | 0 | 7 | 5.448 | 7 | 0,00% |
| 0,30 | 0,4343 | 0,0395 | 0,0724 | 215 | 280 | 5.233 | 495 | 0,23% |
| 0,20 | 0,2555 | 0,0835 | 0,1259 | 455 | 1.326 | 4.993 | 1.781 | 0,83% |
| 0,15 | 0,2317 | 0,1595 | 0,1889 | 869 | 2.882 | 4.579 | 3.751 | 1,76% |
| 0,10 | 0,1800 | 0,2004 | 0,1896 | 1.092 | 4.976 | 4.356 | 6.068 | 2,84% |
| 0,07 | 0,1325 | 0,2858 | 0,1811 | 1.557 | 10.193 | 3.891 | 11.750 | 5,50% |
| 0,05 | 0,0960 | 0,3994 | 0,1548 | 2.176 | 20.491 | 3.272 | 22.667 | 10,61% |
| 0,03 | 0,0686 | 0,5569 | 0,1222 | 3.034 | 41.180 | 2.414 | 44.214 | 20,70% |
| 0,01 | 0,0315 | 0,9394 | 0,0610 | 5.118 | 157.266 | 330 | 162.384 | 76,01% |

Todas as validações de consistência retornaram `OK`.

#### Interpretação dos resultados

**1. Efeito da redução do limiar**

A redução do limiar aumentou progressivamente o Recall.

Com limiar de 0,50, nenhum atraso foi identificado.

Com limiar de 0,05, foram identificados 2.176 atrasos, correspondendo a 39,94% dos casos positivos.

Com limiar de 0,01, o Recall atingiu 93,94%, mas o modelo sinalizou 76,01% de todos os registros.

Isso demonstra que aumentar a identificação dos atrasos pode gerar um volume elevado de falsos alertas.

**2. Relação entre Precision e Recall**

Os resultados evidenciam o compromisso entre identificar mais atrasos e manter a confiabilidade dos alertas.

O limiar de 0,30 apresentou Precision de 43,43%, mas Recall de apenas 3,95%.

O limiar de 0,05 apresentou Recall de 39,94%, mas Precision de apenas 9,60%.

Portanto, nenhum dos limiares avaliados oferece simultaneamente Recall elevado e Precision elevada.

**3. Análise do F1-score**

O maior F1-score entre os limiares testados ocorreu em 0,10, com resultado de 0,1896.

Entretanto, o F1-score não considera diretamente os custos operacionais dos falsos positivos e falsos negativos.

Assim, o limiar de 0,10 não deve ser automaticamente selecionado como configuração definitiva.

**4. Implicações operacionais**

Para uma Torre de Controle, falsos negativos representam coletas em risco que não seriam sinalizadas.

Falsos positivos representam alertas que demandariam investigação, embora a coleta tenha ocorrido dentro do limite.

A escolha do limiar deve considerar a capacidade de tratamento dos alertas, a criticidade dos atrasos e os custos associados a cada tipo de erro.

Os volumes apresentados correspondem à amostra de validação e não devem ser interpretados diretamente como volumes diários de operação.

#### Limitações da análise

A avaliação utilizou apenas os nove limiares definidos para o experimento.

Não foi realizada uma busca exaustiva pelo limiar ótimo.

Também não foram estabelecidas metas operacionais mínimas de Recall, Precision ou capacidade de tratamento de alertas.

O modelo permanece desbalanceado e não recebeu ajustes de pesos de classe.

#### Conclusão

A análise demonstrou que o baseline possui alguma capacidade de diferenciar os riscos, mas apresenta limitações importantes na relação entre Precision, Recall e volume de alertas.

A redução do limiar melhora a identificação dos atrasos, porém aumenta significativamente os falsos positivos.

Nenhum limiar foi selecionado como definitivo.

Não houve novo treinamento, alteração de hiperparâmetros ou utilização do conjunto de teste.

**Status da etapa 2.13: concluída.**

**Próxima etapa: 2.14 — Experimento com ponderação de classes (`class_weight="balanced"`).**

## 2.14 — Experimento com ponderação de classes

### Objetivo

Investigar o efeito da ponderação de classes sobre o desempenho da Regressão Logística na identificação de coletas realizadas após o limite da janela operacional.

O primeiro baseline foi treinado com `class_weight=None`, sem tratamento específico para o desbalanceamento.

Neste segundo experimento, utilizaremos `class_weight="balanced"`.

### Contexto

O conjunto de treinamento contém:

| Classe | Registros | Proporção aproximada |
|---|---:|---:|
| Dentro do limite (0) | 971.513 | 97,45% |
| Após o limite (1) | 25.423 | 2,55% |

O desbalanceamento pode fazer com que o algoritmo priorize a classe majoritária durante a otimização.

No primeiro baseline, com limiar de 0,50, nenhum dos 5.448 atrasos presentes no conjunto de validação foi identificado.

### Conceito: class_weight

O parâmetro `class_weight` controla a importância atribuída aos exemplos de cada classe durante o treinamento.

Com `class_weight="balanced"`, o Scikit-learn calcula pesos inversamente proporcionais à frequência das classes:

\[
w_c=\frac{N}{K\cdot N_c}
\]

Em que:

- \(N\): quantidade total de registros de treinamento.
- \(K\): quantidade de classes.
- \(N_c\): quantidade de registros pertencentes à classe \(c\).

Dessa forma, erros associados à classe minoritária recebem maior peso na função de otimização.

A ponderação não duplica registros nem gera dados sintéticos.

### Hipótese do experimento

A ponderação poderá aumentar a capacidade do modelo de identificar coletas após o limite, especialmente utilizando o limiar de 0,50.

Entretanto, esse aumento poderá ocorrer acompanhado de mais falsos positivos e menor Precision.

Também não há garantia de melhoria na Average Precision.

### Estratégia experimental

Serão mantidos:

- As mesmas 10 features.
- O mesmo particionamento estratificado.
- O mesmo One-Hot Encoding.
- O mesmo StandardScaler.
- O mesmo algoritmo de Regressão Logística.
- O mesmo limite máximo de 1.000 iterações.
- A mesma semente aleatória.

A única alteração será:

`class_weight=None` → `class_weight="balanced"`

O modelo baseline original será preservado.

### Resultado esperado

Treinar um segundo pipeline de Regressão Logística com ponderação de classes e verificar:

1. Tempo de treinamento.
2. Quantidade de features transformadas.
3. Iterações utilizadas.
4. Avisos de convergência.
5. Pesos calculados para cada classe.

A avaliação preditiva será realizada em uma etapa posterior, utilizando o mesmo conjunto de validação.

In [13]:

# ============================================================
# ETAPA 2.14 — REGRESSÃO LOGÍSTICA COM CLASS_WEIGHT BALANCED
# ============================================================

import time
import warnings

from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.utils.class_weight import compute_class_weight

# ------------------------------------------------------------
# 1. Clonar o pipeline original
# ------------------------------------------------------------

pipeline_balanceado = clone(pipeline_baseline)

# ------------------------------------------------------------
# 2. Alterar somente a ponderação de classes
# ------------------------------------------------------------

pipeline_balanceado.set_params(
    classificador__class_weight="balanced"
)

# ------------------------------------------------------------
# 3. Calcular os pesos esperados das classes
# ------------------------------------------------------------

classes_treinamento = np.sort(y_treino.unique())

pesos_classes = compute_class_weight(
    class_weight="balanced",
    classes=classes_treinamento,
    y=y_treino
)

# ------------------------------------------------------------
# 4. Treinar o novo pipeline
# ------------------------------------------------------------

inicio_treinamento_balanceado = time.perf_counter()

with warnings.catch_warnings(record=True) as avisos_balanceado:

    warnings.simplefilter(
        "always",
        ConvergenceWarning
    )

    pipeline_balanceado.fit(
        X_treino,
        y_treino
    )

tempo_treinamento_balanceado = (
    time.perf_counter()
    - inicio_treinamento_balanceado
)

# ------------------------------------------------------------
# 5. Recuperar os componentes ajustados
# ------------------------------------------------------------

preprocessador_balanceado = (
    pipeline_balanceado.named_steps["preprocessamento"]
)

classificador_balanceado = (
    pipeline_balanceado.named_steps["classificador"]
)

total_features_balanceado = len(
    preprocessador_balanceado.get_feature_names_out()
)

avisos_convergencia_balanceado = [
    aviso
    for aviso in avisos_balanceado
    if issubclass(aviso.category, ConvergenceWarning)
]

# ------------------------------------------------------------
# 6. Exibir resultados
# ------------------------------------------------------------

print("=== TREINAMENTO DO MODELO BALANCEADO ===")

print(f"Registros utilizados: {len(X_treino):,}")

print(
    "Features após pré-processamento:",
    total_features_balanceado
)

print(
    f"Tempo de treinamento: "
    f"{tempo_treinamento_balanceado:.2f} segundos"
)

print(
    "Iterações utilizadas:",
    classificador_balanceado.n_iter_[0]
)

print(
    "Avisos de convergência:",
    len(avisos_convergencia_balanceado)
)

for aviso in avisos_convergencia_balanceado:
    print("Detalhe:", str(aviso.message))

print("\n=== PESOS DAS CLASSES ===")

for classe, peso in zip(
    classes_treinamento,
    pesos_classes
):
    print(f"Classe {classe}: {peso:.4f}")

print("\n=== CONFIGURAÇÃO DOS MODELOS ===")

print(
    "Baseline original:",
    pipeline_baseline.get_params()[
        "classificador__class_weight"
    ]
)

print(
    "Modelo balanceado:",
    pipeline_balanceado.get_params()[
        "classificador__class_weight"
    ]
)

# ------------------------------------------------------------
# 7. Validações
# ------------------------------------------------------------

assert (
    pipeline_baseline.get_params()[
        "classificador__class_weight"
    ] is None
), "O baseline original foi alterado."

assert (
    pipeline_balanceado.get_params()[
        "classificador__class_weight"
    ] == "balanced"
), "A ponderação não foi configurada."

assert hasattr(
    classificador_balanceado,
    "coef_"
), "O modelo balanceado não foi treinado."

assert (
    total_features_balanceado
    == total_features_transformadas
), "A quantidade de features transformadas divergiu."

print("\nValidação do treinamento: OK")
print("Modelo baseline original preservado.")
print("Modelo balanceado treinado.")
print("Avaliação preditiva ainda não realizada.")


=== TREINAMENTO DO MODELO BALANCEADO ===
Registros utilizados: 996,936
Features após pré-processamento: 53
Tempo de treinamento: 6.06 segundos
Iterações utilizadas: 109
Avisos de convergência: 0

=== PESOS DAS CLASSES ===
Classe 0: 0.5131
Classe 1: 19.6070

=== CONFIGURAÇÃO DOS MODELOS ===
Baseline original: None
Modelo balanceado: balanced

Validação do treinamento: OK
Modelo baseline original preservado.
Modelo balanceado treinado.
Avaliação preditiva ainda não realizada.


### Explicação do código — Etapa 2.14

#### 1. Clonagem do pipeline

A função `clone()`, do Scikit-learn, cria um novo estimador com a mesma configuração do pipeline original.

```python
pipeline_balanceado = clone(pipeline_baseline)
```

O novo pipeline mantém a estrutura de pré-processamento e os hiperparâmetros do baseline, mas não reutiliza os coeficientes ou os parâmetros de transformação aprendidos anteriormente.

Isso permite realizar um segundo treinamento independente, preservando o modelo original para comparação.

#### 2. Configuração da ponderação de classes

O método `set_params()` modifica parâmetros de um estimador.

```python
pipeline_balanceado.set_params(
    classificador__class_weight="balanced"
)
```

Os dois sublinhados (`__`) permitem acessar o parâmetro `class_weight` da etapa interna `classificador`.

Essa é a única alteração experimental em relação ao baseline original.

Com `class_weight="balanced"`, a Regressão Logística atribui pesos inversamente proporcionais à frequência de cada classe.

#### 3. Identificação e ordenação das classes

```python
classes_treinamento = np.sort(y_treino.unique())
```

O método `unique()` identifica os valores distintos presentes na variável-alvo.

Como nossa classificação é binária, os valores esperados são `0` e `1`.

A função `np.sort()` ordena esses valores e retorna um array NumPy (`numpy.ndarray`).

**Por que utilizamos `np.sort()` em vez de `sorted()`?**

A função nativa `sorted()` retorna uma lista Python (`list`).

Na versão utilizada do Scikit-learn, o argumento `classes` da função `compute_class_weight()` exige um `numpy.ndarray`.

A primeira execução apresentou `InvalidParameterError` porque utilizávamos `sorted()`.

A substituição por `np.sort()` corrigiu a incompatibilidade de tipos, sem alterar os valores das classes ou a metodologia do experimento.

#### 4. Cálculo dos pesos

```python
pesos_classes = compute_class_weight(
    class_weight="balanced",
    classes=classes_treinamento,
    y=y_treino
)
```

A função `compute_class_weight()` calcula os pesos das classes a partir da distribuição da variável-alvo no conjunto de treinamento.

A fórmula utilizada é:

\[
w_c = \frac{N}{K \cdot N_c}
\]

Em que:

- \(N\): total de registros de treinamento.
- \(K\): número de classes.
- \(N_c\): quantidade de registros da classe considerada.

A classe minoritária recebe um peso maior porque possui menos exemplos.

Esses pesos aumentam a contribuição de seus erros durante a otimização.

O cálculo não altera os registros, não duplica observações e não cria dados sintéticos.

#### 5. Medição do tempo de treinamento

A função `time.perf_counter()` registra instantes de um contador de alta precisão.

A diferença entre os valores obtidos antes e depois do treinamento fornece o tempo decorrido.

Esse procedimento permite comparar o custo computacional dos experimentos.

#### 6. Captura de avisos de convergência

O bloco `warnings.catch_warnings()` captura os avisos emitidos durante o treinamento.

A classe `ConvergenceWarning` identifica situações nas quais o algoritmo pode não ter atingido os critérios de convergência.

A ausência desse aviso é uma verificação importante, mas não garante que o modelo tenha bom desempenho preditivo.

#### 7. Treinamento do pipeline

```python
pipeline_balanceado.fit(
    X_treino,
    y_treino
)
```

O método `fit()` executa o treinamento utilizando exclusivamente o conjunto de treinamento.

O pipeline realiza o pré-processamento e ajusta os coeficientes da Regressão Logística ponderada.

As transformações são aprendidas novamente a partir dos dados de treinamento.

O conjunto de validação não participa desse ajuste.

#### 8. Recuperação dos componentes treinados

O atributo `named_steps` permite acessar as etapas internas do pipeline pelo nome.

Utilizamos esse recurso para recuperar:

- O pré-processador ajustado.
- O classificador treinado.

O método `get_feature_names_out()` identifica as features produzidas após o pré-processamento.

O atributo `n_iter_` informa a quantidade de iterações utilizadas pelo classificador.

#### 9. Validação da comparabilidade

As verificações com `assert` confirmam que:

- O baseline original continua configurado com `class_weight=None`.
- O segundo modelo utiliza `class_weight="balanced"`.
- O novo classificador possui coeficientes aprendidos.
- O número de features transformadas permanece compatível com o baseline.

Essas verificações ajudam a preservar a consistência metodológica do experimento.

### Considerações metodológicas

O experimento modifica a importância relativa das classes durante o treinamento.

Isso pode alterar os coeficientes aprendidos, as probabilidades estimadas e as classificações resultantes.

Entretanto, a ponderação não garante aumento de Precision, Recall, F1-score ou Average Precision.

Essas métricas deverão ser verificadas no conjunto de validação.

O conjunto de teste permanece reservado para a avaliação final.

### 2.14 — Resultados, interpretação e conclusão

#### Resultados obtidos

O segundo modelo de Regressão Logística foi treinado com sucesso utilizando `class_weight="balanced"`.

O treinamento utilizou 996.936 registros e produziu 53 features após o pré-processamento.

| Indicador | Baseline original | Modelo balanceado |
|---|---:|---:|
| Tempo de treinamento | 3,39 segundos | 13,40 segundos |
| Iterações utilizadas | 46 | 109 |
| Features transformadas | 53 | 53 |
| Avisos de convergência | 0 | 0 |
| Ponderação de classes | None | balanced |

Os pesos calculados foram:

- Classe 0: 0,5131.
- Classe 1: 19,6070.

A classe positiva recebeu aproximadamente 38,2 vezes o peso atribuído à classe negativa.

#### Interpretação

A ponderação de classes aumentou a importância relativa dos exemplos positivos durante o treinamento.

O modelo balanceado precisou de mais iterações e apresentou maior tempo de treinamento do que o baseline original.

Ambos os treinamentos terminaram sem avisos de convergência.

A quantidade de features transformadas permaneceu igual, confirmando a consistência estrutural dos dois experimentos.

O baseline original foi preservado.

#### Registro de correção técnica

Durante a primeira execução, a função `compute_class_weight()` apresentou um erro de validação porque recebeu uma lista Python no argumento `classes`.

A expressão:

`sorted(y_treino.unique())`

foi substituída por:

`np.sort(y_treino.unique())`

A correção garantiu o tipo `numpy.ndarray` exigido pela função.

Após a alteração, a execução foi concluída sem erros.

#### Limitações

Os resultados desta etapa demonstram que o modelo foi treinado corretamente, mas ainda não permitem concluir se a ponderação melhorou sua capacidade preditiva.

Não foram calculadas métricas de classificação para o modelo balanceado.

O conjunto de validação ainda será utilizado para essa comparação.

#### Conclusão

O segundo experimento foi concluído com sucesso.

O pipeline balanceado foi treinado, os pesos das classes foram calculados e as verificações estruturais foram aprovadas.

A única alteração experimental em relação ao baseline original foi a ponderação de classes.

Nenhuma avaliação foi realizada sobre o conjunto de teste.

**Status da etapa 2.14: concluída.**

**Próxima etapa: 2.15 — Avaliação comparativa entre o baseline original e a Regressão Logística balanceada.**

## 2.15 — Avaliação comparativa: baseline original × modelo balanceado

### Objetivo

Comparar o desempenho preditivo de duas versões da Regressão Logística:

1. Baseline original, sem ponderação de classes.
2. Modelo com ponderação automática de classes (`class_weight="balanced"`).

O objetivo é investigar se a ponderação melhora a identificação das coletas realizadas após o limite da janela operacional.

### Contexto experimental

Os dois modelos foram treinados utilizando:

- O mesmo conjunto de treinamento, com 996.936 registros.
- As mesmas 10 features de entrada.
- O mesmo pré-processamento.
- O mesmo algoritmo de Regressão Logística.
- A mesma configuração de semente aleatória.

A única alteração experimental foi o parâmetro `class_weight`.

O conjunto de validação contém 213.629 registros, sendo 5.448 positivos e 208.181 negativos.

### Estratégia de avaliação

Para cada modelo, calcularemos as probabilidades de pertencimento à classe positiva.

Inicialmente, utilizaremos o mesmo limiar de classificação de 0,50.

Serão comparadas as seguintes métricas:

- **Accuracy:** proporção total de classificações corretas.
- **Precision:** proporção de alertas que correspondem a atrasos reais.
- **Recall:** proporção de atrasos reais identificados pelo modelo.
- **F1-score:** média harmônica entre Precision e Recall.
- **Average Precision (AP):** indicador da capacidade de ordenação dos riscos ao longo dos limiares.

Também serão analisados:

- Verdadeiros positivos (TP).
- Falsos positivos (FP).
- Falsos negativos (FN).
- Verdadeiros negativos (TN).
- Total de alertas emitidos.
- Percentual de registros sinalizados.
- Distribuição das probabilidades estimadas.

### Cuidados metodológicos

A comparação utilizará exclusivamente o conjunto de validação.

A avaliação com limiar de 0,50 permitirá observar o efeito inicial da ponderação de classes, mas não determinará automaticamente qual modelo é superior para uso operacional.

A ponderação pode modificar tanto a classificação quanto a distribuição das probabilidades estimadas.

Por isso, também compararemos a Average Precision, que avalia a ordenação dos exemplos sem depender de um único limiar.

Não selecionaremos um modelo definitivo nesta etapa.

O conjunto de teste continuará reservado para a avaliação final.

In [14]:

# ============================================================
# ETAPA 2.15 — COMPARAÇÃO DOS MODELOS DE REGRESSÃO LOGÍSTICA
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Definir os modelos e o limiar de comparação
# ------------------------------------------------------------

modelos_comparacao = {
    "Baseline original": pipeline_baseline,
    "Modelo balanceado": pipeline_balanceado
}

limiar_comparacao = 0.50

# ------------------------------------------------------------
# 2. Avaliar cada modelo
# ------------------------------------------------------------

resultados_comparacao = []
resultados_probabilidades = []

for nome_modelo, pipeline_modelo in modelos_comparacao.items():

    # Probabilidade estimada da classe positiva
    probabilidades = pipeline_modelo.predict_proba(
        X_validacao
    )[:, 1]

    # Transformar probabilidades em classes
    previsoes = (
        probabilidades >= limiar_comparacao
    ).astype(int)

    # Matriz de confusão
    tn, fp, fn, tp = confusion_matrix(
        y_validacao,
        previsoes,
        labels=[0, 1]
    ).ravel()

    total_alertas = int(tp + fp)

    # Métricas de classificação
    resultados_comparacao.append({
        "Modelo": nome_modelo,
        "Accuracy": accuracy_score(
            y_validacao, previsoes
        ),
        "Precision": precision_score(
            y_validacao, previsoes, zero_division=0
        ),
        "Recall": recall_score(
            y_validacao, previsoes, zero_division=0
        ),
        "F1-score": f1_score(
            y_validacao, previsoes, zero_division=0
        ),
        "Average Precision": average_precision_score(
            y_validacao, probabilidades
        ),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "Alertas": total_alertas,
        "Alertas (%)": (
            total_alertas / len(y_validacao) * 100
        )
    })

    # Resumo das probabilidades por classe real
    for classe_real in [0, 1]:

        probabilidades_classe = probabilidades[
            y_validacao.to_numpy() == classe_real
        ]

        resultados_probabilidades.append({
            "Modelo": nome_modelo,
            "Classe real": classe_real,
            "Quantidade": len(probabilidades_classe),
            "Média": probabilidades_classe.mean(),
            "Mediana": np.median(probabilidades_classe),
            "Mínimo": probabilidades_classe.min(),
            "Máximo": probabilidades_classe.max()
        })

# ------------------------------------------------------------
# 3. Organizar os resultados em DataFrames
# ------------------------------------------------------------

df_comparacao_modelos = pd.DataFrame(
    resultados_comparacao
)

df_probabilidades_modelos = pd.DataFrame(
    resultados_probabilidades
)

# ------------------------------------------------------------
# 4. Exibir métricas
# ------------------------------------------------------------

print("=== COMPARAÇÃO DAS MÉTRICAS ===")
print(f"Limiar utilizado: {limiar_comparacao:.2f}")

colunas_metricas = [
    "Modelo",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "Average Precision"
]

print(
    df_comparacao_modelos[
        colunas_metricas
    ].to_string(
        index=False,
        float_format=lambda valor: f"{valor:.4f}"
    )
)

# ------------------------------------------------------------
# 5. Exibir matriz de confusão e alertas
# ------------------------------------------------------------

print("\n=== MATRIZ DE CONFUSÃO E ALERTAS ===")

colunas_alertas = [
    "Modelo",
    "TN",
    "FP",
    "FN",
    "TP",
    "Alertas",
    "Alertas (%)"
]

print(
    df_comparacao_modelos[
        colunas_alertas
    ].to_string(
        index=False,
        float_format=lambda valor: f"{valor:.2f}"
    )
)

# ------------------------------------------------------------
# 6. Exibir probabilidades por classe real
# ------------------------------------------------------------

print("\n=== PROBABILIDADES POR CLASSE REAL ===")

print(
    df_probabilidades_modelos.to_string(
        index=False,
        float_format=lambda valor: f"{valor:.4f}"
    )
)

# ------------------------------------------------------------
# 7. Validar os resultados
# ------------------------------------------------------------

total_validacao = len(y_validacao)
total_positivos = int(y_validacao.sum())

assert (
    df_comparacao_modelos["TN"]
    + df_comparacao_modelos["FP"]
    + df_comparacao_modelos["FN"]
    + df_comparacao_modelos["TP"]
    == total_validacao
).all()

assert (
    df_comparacao_modelos["TP"]
    + df_comparacao_modelos["FN"]
    == total_positivos
).all()

assert (
    df_comparacao_modelos["TP"]
    + df_comparacao_modelos["FP"]
    == df_comparacao_modelos["Alertas"]
).all()

assert (
    df_probabilidades_modelos["Mínimo"].ge(0)
    & df_probabilidades_modelos["Máximo"].le(1)
).all()

assert (
    df_probabilidades_modelos["Quantidade"].sum()
    == total_validacao * len(modelos_comparacao)
)

print("\n=== VALIDAÇÕES ===")
print(f"Registros avaliados por modelo: {total_validacao:,}")
print(f"Atrasos reais: {total_positivos:,}")
print("Consistência das métricas: OK")
print("Nenhum novo treinamento realizado.")
print("Conjunto de teste não utilizado.")


=== COMPARAÇÃO DAS MÉTRICAS ===
Limiar utilizado: 0.50
           Modelo  Accuracy  Precision  Recall  F1-score  Average Precision
Baseline original    0.9745     0.0000  0.0000    0.0000             0.1188
Modelo balanceado    0.7382     0.0598  0.6292    0.1092             0.1125

=== MATRIZ DE CONFUSÃO E ALERTAS ===
           Modelo     TN    FP   FN   TP  Alertas  Alertas (%)
Baseline original 208174     7 5448    0        7         0.00
Modelo balanceado 154270 53911 2020 3428    57339        26.84

=== PROBABILIDADES POR CLASSE REAL ===
           Modelo  Classe real  Quantidade  Média  Mediana  Mínimo  Máximo
Baseline original            0      208181 0.0244   0.0153  0.0020  0.5608
Baseline original            1        5448 0.0708   0.0347  0.0027  0.4364
Modelo balanceado            0      208181 0.4061   0.3765  0.0561  0.9906
Modelo balanceado            1        5448 0.5986   0.5880  0.0860  0.9848

=== VALIDAÇÕES ===
Registros avaliados por modelo: 213,629
Atrasos reais: 

### Explicação do código — Etapa 2.15

#### 1. Dicionário de modelos

A variável `modelos_comparacao` armazena os dois pipelines treinados.

Utilizamos um dicionário Python para associar um nome descritivo a cada modelo.

Isso permite avaliar os modelos em uma mesma estrutura de repetição, reduzindo a duplicação de código.

#### 2. Limiar de classificação

A variável `limiar_comparacao` foi definida como `0.50`.

Esse limiar será utilizado inicialmente para os dois modelos.

A comparação com o mesmo limiar permite observar como a ponderação de classes modifica as previsões.

Entretanto, o limiar de 0,50 não representa necessariamente a melhor decisão operacional para qualquer um dos modelos.

#### 3. Probabilidades previstas

O método `predict_proba()` retorna as probabilidades estimadas para cada classe.

A expressão `[:, 1]` seleciona a coluna correspondente à classe positiva, considerando a ordem das classes `0` e `1` utilizada neste experimento.

Essas probabilidades são utilizadas tanto para a classificação binária quanto para o cálculo da Average Precision.

#### 4. Matriz de confusão

A função `confusion_matrix()` compara as classes reais com as classes previstas.

Os quatro valores obtidos são:

- TN: coletas dentro do limite corretamente identificadas.
- FP: coletas dentro do limite incorretamente sinalizadas.
- FN: coletas após o limite não identificadas.
- TP: coletas após o limite corretamente sinalizadas.

Esses indicadores permitem interpretar o comportamento do modelo do ponto de vista operacional.

#### 5. Métricas de classificação

As funções `accuracy_score()`, `precision_score()`, `recall_score()` e `f1_score()` avaliam as classificações produzidas com o limiar escolhido.

A função `average_precision_score()` utiliza as probabilidades estimadas e não depende de um único limiar.

A Average Precision resume a relação entre Precision e Recall ao longo dos diferentes pontos de decisão.

Ela não deve ser confundida com a Precision obtida no limiar de 0,50.

#### 6. Volume de alertas

A quantidade de alertas corresponde à soma de TP e FP.

O percentual de alertas representa a proporção dos registros de validação classificados como positivos.

Esse indicador permite avaliar a carga potencial de trabalho que o modelo produziria para a Torre de Controle.

#### 7. Distribuição das probabilidades

Para cada modelo, as probabilidades são agrupadas conforme a classe real.

São calculados:

- Quantidade de observações.
- Média.
- Mediana.
- Valor mínimo.
- Valor máximo.

Essas estatísticas ajudam a investigar como a ponderação de classes modifica as probabilidades previstas.

Probabilidades maiores não significam necessariamente que o modelo possui melhor capacidade de ordenação ou que suas probabilidades estejam bem calibradas.

#### 8. DataFrames comparativos

Os resultados são organizados em dois DataFrames:

- `df_comparacao_modelos`: métricas, matriz de confusão e volume de alertas.
- `df_probabilidades_modelos`: estatísticas das probabilidades por classe real.

Essa organização facilita análises posteriores e mantém os resultados disponíveis no notebook.

#### 9. Validações

O código verifica se:

- A matriz de confusão contém todos os registros avaliados.
- A quantidade de positivos reais permanece consistente.
- O volume de alertas corresponde à soma de TP e FP.
- As probabilidades estão no intervalo de 0 a 1.
- As estatísticas incluem todas as observações dos dois modelos.

### Interpretação futura

Após a execução, compararemos os resultados para verificar se a ponderação aumentou o Recall, qual foi o impacto sobre a Precision e quantos alertas adicionais foram produzidos.

Também investigaremos se a Average Precision melhorou, indicando maior capacidade de ordenação dos riscos.

Nenhum modelo será selecionado definitivamente apenas com base nessa comparação inicial.

O conjunto de teste permanecerá reservado.

### 2.15 — Resultados, interpretação e conclusão

#### Resultados obtidos

Foram avaliados dois modelos de Regressão Logística sobre o mesmo conjunto de validação, contendo 213.629 registros e 5.448 coletas realizadas após o limite da janela operacional.

O limiar de classificação utilizado foi 0,50.

| Métrica | Baseline original | Modelo balanceado |
|---|---:|---:|
| Accuracy | 0,9745 | 0,7382 |
| Precision | 0,0000 | 0,0598 |
| Recall | 0,0000 | 0,6292 |
| F1-score | 0,0000 | 0,1092 |
| Average Precision | 0,1188 | 0,1125 |

#### Matrizes de confusão

**Baseline original:**

- Verdadeiros negativos (TN): 208.174.
- Falsos positivos (FP): 7.
- Falsos negativos (FN): 5.448.
- Verdadeiros positivos (TP): 0.
- Total de alertas: 7.

**Modelo balanceado:**

- Verdadeiros negativos (TN): 154.270.
- Falsos positivos (FP): 53.911.
- Falsos negativos (FN): 2.020.
- Verdadeiros positivos (TP): 3.428.
- Total de alertas: 57.339.

#### Interpretação do Recall

A ponderação aumentou o Recall de 0% para 62,92%.

O modelo balanceado identificou 3.428 dos 5.448 atrasos reais presentes no conjunto de validação.

Entretanto, 2.020 atrasos permaneceram sem identificação.

Esse resultado demonstra que a ponderação modificou significativamente o comportamento de classificação do modelo no limiar de 0,50.

#### Interpretação da Precision

A Precision do modelo balanceado foi de 5,98%.

Isso significa que aproximadamente seis em cada cem alertas correspondiam a atrasos reais.

Os demais eram falsos positivos.

O modelo emitiu 57.339 alertas, representando 26,84% dos registros avaliados.

Esse volume pode representar uma carga operacional elevada para uma Torre de Controle, dependendo de sua capacidade de investigação e tratamento.

#### Interpretação da Average Precision

A Average Precision apresentou os seguintes resultados:

- Baseline original: 0,1188.
- Modelo balanceado: 0,1125.

A ponderação não melhorou a capacidade global de ordenação dos riscos segundo essa métrica.

A diferença observada foi de 0,0063 ponto em escala de 0 a 1, equivalente a 0,63 ponto percentual.

Embora o modelo balanceado tenha apresentado Recall superior no limiar de 0,50, isso não significa que ele seja superior em todos os limiares.

A comparação evidencia a importância de analisar tanto as métricas dependentes do limiar quanto as métricas de ordenação.

#### Distribuição das probabilidades

As probabilidades médias estimadas foram:

| Classe real | Baseline original | Modelo balanceado |
|---|---:|---:|
| Classe 0 | 0,0244 | 0,4061 |
| Classe 1 | 0,0708 | 0,5986 |

A ponderação deslocou as probabilidades estimadas para valores mais elevados.

Esse deslocamento contribuiu para o aumento das previsões positivas no limiar de 0,50.

Entretanto, probabilidades maiores não significam necessariamente melhor capacidade preditiva.

Além disso, as probabilidades do modelo ponderado não devem ser automaticamente interpretadas como probabilidades calibradas de atraso.

#### Implicações para a Torre de Controle

O experimento demonstrou que a ponderação pode aumentar a identificação dos atrasos, mas também elevar significativamente os falsos alertas.

A decisão operacional não deve considerar apenas o Recall.

Também é necessário avaliar:

- A quantidade de atrasos identificados.
- A quantidade de atrasos não identificados.
- A proporção de alertas corretos.
- O volume total de alertas.
- A capacidade operacional de tratamento.

O volume observado corresponde ao conjunto de validação e não representa diretamente uma carga diária de produção.

#### Limitações

A comparação foi realizada com limiar fixo de 0,50.

Não foram avaliados diferentes limiares para o modelo balanceado nesta etapa.

Não foi realizado novo treinamento, ajuste de hiperparâmetros ou calibração de probabilidades.

O conjunto de teste permaneceu reservado.

#### Conclusão

A ponderação de classes aumentou substancialmente o Recall no limiar de 0,50, permitindo identificar 62,92% dos atrasos.

Entretanto, a Precision permaneceu baixa, com 53.911 falsos positivos.

A Average Precision foi ligeiramente inferior à do baseline original.

Portanto, **não há evidência suficiente para selecionar o modelo balanceado como melhor alternativa operacional**.

A próxima investigação deverá comparar os modelos em diferentes limiares e considerar o volume de alertas gerados.

**Status da etapa 2.15: concluída.**

**Próxima etapa: 2.16 — Comparação dos modelos em diferentes limiares de classificação.**

## 2.16 — Comparação dos modelos em diferentes limiares

### Objetivo

Comparar a Regressão Logística original e a Regressão Logística balanceada considerando diferentes limiares de classificação.

A análise busca identificar se a ponderação de classes melhora efetivamente a capacidade de identificação dos atrasos ou se seu principal efeito é modificar as probabilidades previstas.

### Contexto

Na etapa 2.15, com limiar de 0,50, obtivemos:

| Métrica | Baseline original | Modelo balanceado |
|---|---:|---:|
| Precision | 0,00% | 5,98% |
| Recall | 0,00% | 62,92% |
| Average Precision | 11,88% | 11,25% |
| Alertas | 7 | 57.339 |

Esses resultados não são suficientes para afirmar que o modelo balanceado é superior.

A ponderação alterou significativamente a distribuição das probabilidades estimadas. Por isso, utilizar o mesmo limiar numérico não significa necessariamente comparar os modelos sob a mesma carga operacional.

### Conceito: comparação por volume de alertas

Em uma Torre de Controle, a capacidade de investigar alertas pode ser limitada.

Por exemplo, dois modelos podem emitir a mesma quantidade de alertas, mas identificar quantidades diferentes de atrasos reais.

Compará-los com um volume semelhante de alertas permite avaliar qual modelo prioriza melhor os casos de maior risco.

### Estratégia experimental

Utilizaremos percentuais de registros sinalizados como positivos:

- 1%.
- 2%.
- 5%.
- 10%.
- 15%.
- 20%.
- 30%.

Para cada percentual, selecionaremos os registros com as maiores probabilidades estimadas por cada modelo.

Essa abordagem permite comparar os classificadores com a mesma quantidade de alertas.

Serão calculados:

- Total de alertas.
- Verdadeiros positivos (TP).
- Falsos positivos (FP).
- Falsos negativos (FN).
- Precision.
- Recall.
- F1-score.

Também será apresentada a Average Precision de cada modelo como referência geral de ordenação dos riscos.

### Cuidados metodológicos

A comparação será realizada exclusivamente sobre o conjunto de validação.

Não haverá novo treinamento ou alteração dos modelos.

Os percentuais de alertas representam cenários experimentais, não capacidades operacionais previamente estabelecidas.

Os resultados não serão utilizados para selecionar automaticamente um modelo definitivo.

O conjunto de teste permanecerá reservado.

In [15]:

# ============================================================
# ETAPA 2.16 — COMPARAÇÃO POR VOLUME DE ALERTAS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import average_precision_score

# ------------------------------------------------------------
# 1. Definir os percentuais de alertas
# ------------------------------------------------------------

percentuais_alertas = [
    0.01,
    0.02,
    0.05,
    0.10,
    0.15,
    0.20,
    0.30
]

# ------------------------------------------------------------
# 2. Preparar as classes reais
# ------------------------------------------------------------

y_real = y_validacao.to_numpy()

total_registros = len(y_real)
total_positivos = int(y_real.sum())

# ------------------------------------------------------------
# 3. Preparar os modelos
# ------------------------------------------------------------

modelos_216 = {
    "Baseline original": pipeline_baseline,
    "Modelo balanceado": pipeline_balanceado
}

resultados_216 = []
metricas_ordenacao_216 = []

# ------------------------------------------------------------
# 4. Avaliar cada modelo
# ------------------------------------------------------------

for nome_modelo, modelo in modelos_216.items():

    probabilidades = modelo.predict_proba(
        X_validacao
    )[:, 1]

    # Ordenar da maior para a menor probabilidade
    # mergesort mantém uma ordenação estável em empates
    indices_ordenados = np.argsort(
        -probabilidades,
        kind="stable"
    )

    ap = average_precision_score(
        y_real,
        probabilidades
    )

    metricas_ordenacao_216.append({
        "Modelo": nome_modelo,
        "Average Precision": ap
    })

    # --------------------------------------------------------
    # 5. Avaliar cada capacidade de alertas
    # --------------------------------------------------------

    for percentual in percentuais_alertas:

        quantidade_alertas = int(
            np.ceil(total_registros * percentual)
        )

        indices_alertados = indices_ordenados[
            :quantidade_alertas
        ]

        tp = int(
            y_real[indices_alertados].sum()
        )

        fp = quantidade_alertas - tp
        fn = total_positivos - tp

        precision = (
            tp / quantidade_alertas
            if quantidade_alertas > 0
            else 0.0
        )

        recall = (
            tp / total_positivos
            if total_positivos > 0
            else 0.0
        )

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall > 0
            else 0.0
        )

        resultados_216.append({
            "Modelo": nome_modelo,
            "Capacidade (%)": percentual * 100,
            "Alertas": quantidade_alertas,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "Precision": precision,
            "Recall": recall,
            "F1-score": f1
        })

# ------------------------------------------------------------
# 6. Organizar os resultados
# ------------------------------------------------------------

df_comparacao_capacidade = pd.DataFrame(
    resultados_216
)

df_ordenacao_216 = pd.DataFrame(
    metricas_ordenacao_216
)

# ------------------------------------------------------------
# 7. Exibir a comparação
# ------------------------------------------------------------

print("=== COMPARAÇÃO POR CAPACIDADE DE ALERTAS ===")

print(
    df_comparacao_capacidade.to_string(
        index=False,
        formatters={
            "Capacidade (%)": "{:.0f}".format,
            "Precision": "{:.4f}".format,
            "Recall": "{:.4f}".format,
            "F1-score": "{:.4f}".format
        }
    )
)

print("\n=== AVERAGE PRECISION ===")

print(
    df_ordenacao_216.to_string(
        index=False,
        float_format=lambda valor: f"{valor:.4f}"
    )
)

# ------------------------------------------------------------
# 8. Validações
# ------------------------------------------------------------

assert (
    df_comparacao_capacidade["TP"]
    + df_comparacao_capacidade["FP"]
    == df_comparacao_capacidade["Alertas"]
).all()

assert (
    df_comparacao_capacidade["TP"]
    + df_comparacao_capacidade["FN"]
    == total_positivos
).all()

assert (
    df_comparacao_capacidade["TP"]
    .ge(0)
    & df_comparacao_capacidade["FP"].ge(0)
    & df_comparacao_capacidade["FN"].ge(0)
).all()

assert (
    df_comparacao_capacidade.groupby(
        "Capacidade (%)"
    )["Alertas"].nunique() == 1
).all()

assert (
    df_comparacao_capacidade.groupby(
        "Modelo"
    )["Recall"].diff().dropna().ge(-1e-12)
).all()

print("\n=== VALIDAÇÕES ===")
print(f"Registros avaliados: {total_registros:,}")
print(f"Atrasos reais: {total_positivos:,}")
print("Mesma quantidade de alertas por cenário: OK")
print("Consistência das métricas: OK")
print("Nenhum novo treinamento realizado.")
print("Conjunto de teste não utilizado.")


=== COMPARAÇÃO POR CAPACIDADE DE ALERTAS ===
           Modelo Capacidade (%)  Alertas   TP    FP   FN Precision Recall F1-score
Baseline original              1     2137  530  1607 4918    0.2480 0.0973   0.1397
Baseline original              2     4273  938  3335 4510    0.2195 0.1722   0.1930
Baseline original              5    10682 1475  9207 3973    0.1381 0.2707   0.1829
Baseline original             10    21363 2106 19257 3342    0.0986 0.3866   0.1571
Baseline original             15    32045 2626 29419 2822    0.0819 0.4820   0.1401
Baseline original             20    42726 2972 39754 2476    0.0696 0.5455   0.1234
Baseline original             30    64089 3529 60560 1919    0.0551 0.6478   0.1015
Modelo balanceado              1     2137  570  1567 4878    0.2667 0.1046   0.1503
Modelo balanceado              2     4273  918  3355 4530    0.2148 0.1685   0.1889
Modelo balanceado              5    10682 1475  9207 3973    0.1381 0.2707   0.1829
Modelo balanceado             1

### Explicação do código — Etapa 2.16

#### 1. Definição das capacidades de alerta

A lista `percentuais_alertas` representa diferentes proporções do conjunto de validação que poderão receber alertas.

Por exemplo, uma capacidade de 5% significa selecionar aproximadamente 5% dos registros com maior risco estimado.

Esses percentuais são cenários experimentais e não representam uma capacidade operacional real já definida.

#### 2. Conversão da variável-alvo

O método `to_numpy()` transforma a Series do Pandas em um array NumPy.

Isso facilita a seleção de observações por posição durante a comparação.

#### 3. Probabilidades previstas

O método `predict_proba()` retorna as probabilidades estimadas pelo modelo.

Selecionamos a coluna correspondente à classe positiva.

Cada modelo utiliza suas próprias probabilidades, produzidas após o respectivo treinamento.

#### 4. Ordenação das probabilidades

A função `np.argsort()` retorna os índices que ordenam os elementos de um array.

Utilizamos o sinal negativo para ordenar as probabilidades da maior para a menor.

O argumento `kind="stable"` mantém a ordem original entre observações com probabilidades iguais.

Isso permite selecionar os registros com maior risco estimado.

#### 5. Seleção dos alertas

Para cada percentual, calculamos a quantidade de registros que serão sinalizados.

A função `np.ceil()` arredonda esse valor para cima.

Em seguida, selecionamos os primeiros índices da lista ordenada.

Assim, os dois modelos são avaliados com a mesma quantidade de alertas.

Essa abordagem equivale a uma política de priorização dos registros com maior pontuação de risco.

#### 6. Cálculo das métricas

Os verdadeiros positivos correspondem aos registros selecionados que realmente apresentaram atraso.

Os falsos positivos correspondem aos registros selecionados que não apresentaram atraso.

Os falsos negativos correspondem aos atrasos reais que não foram selecionados.

A partir desses valores, calculamos Precision, Recall e F1-score.

#### 7. Average Precision

A função `average_precision_score()` calcula uma métrica de ordenação dos riscos considerando diferentes pontos de decisão.

Ela não depende de um limiar fixo.

Sua comparação ajuda a investigar se a ponderação de classes melhorou a capacidade global de priorização dos casos positivos.

#### 8. DataFrames

Os resultados são armazenados em:

- `df_comparacao_capacidade`: comparação dos modelos por percentual de alertas.
- `df_ordenacao_216`: Average Precision de cada modelo.

Essa estrutura permite reutilizar os resultados em análises posteriores.

#### 9. Validações

O código verifica se:

- A soma de TP e FP corresponde à quantidade de alertas.
- A soma de TP e FN corresponde ao total de atrasos reais.
- As contagens não são negativas.
- Os dois modelos possuem a mesma quantidade de alertas em cada cenário.
- O Recall não diminui quando a capacidade de alertas aumenta.

### Limitação importante

Essa comparação seleciona uma quantidade fixa de registros com maior risco estimado.

Portanto, não utiliza diretamente um limiar numérico de probabilidade.

Em um ambiente operacional, uma política desse tipo poderia ser implementada como uma priorização dos casos de maior risco, desde que existissem regras claras de capacidade, disponibilidade dos dados e momento da decisão.

A análise atual não estabelece essas condições.

### Interpretação futura

Após a execução, verificaremos qual modelo identifica mais atrasos para uma mesma quantidade de alertas.

Isso permitirá avaliar a utilidade da ordenação dos riscos independentemente do deslocamento das probabilidades causado pela ponderação de classes.

A decisão final sobre modelos e políticas de alerta permanecerá pendente.

### 2.16 — Resultados, interpretação e conclusão

#### Resultados obtidos

Foram comparados dois modelos de Regressão Logística utilizando capacidades de alerta de 1%, 2%, 5%, 10%, 15%, 20% e 30% do conjunto de validação.

A avaliação utilizou 213.629 registros, incluindo 5.448 coletas realizadas após o limite da janela operacional.

Para cada capacidade, os dois modelos emitiram exatamente a mesma quantidade de alertas.

| Capacidade | TP original | TP balanceado | Recall original | Recall balanceado |
|---|---:|---:|---:|---:|
| 1% | 530 | 570 | 9,73% | 10,46% |
| 2% | 938 | 918 | 17,22% | 16,85% |
| 5% | 1.475 | 1.475 | 27,07% | 27,07% |
| 10% | 2.106 | 2.065 | 38,66% | 37,90% |
| 15% | 2.626 | 2.641 | 48,20% | 48,48% |
| 20% | 2.972 | 3.032 | 54,55% | 55,65% |
| 30% | 3.529 | 3.569 | 64,78% | 65,51% |

Todas as verificações de consistência foram aprovadas.

#### Interpretação da comparação por capacidade

Os resultados demonstram que os modelos apresentam desempenhos próximos quando avaliados com a mesma quantidade de alertas.

O modelo balanceado apresentou Recall superior nas capacidades de 1%, 15%, 20% e 30%.

O baseline original apresentou Recall superior nas capacidades de 2% e 10%.

Na capacidade de 5%, ambos identificaram exatamente 1.475 atrasos.

As diferenças observadas foram relativamente pequenas e não demonstraram uma vantagem consistente da ponderação de classes.

#### Interpretação da Average Precision

A Average Precision apresentou os seguintes resultados:

- Baseline original: 0,1188.
- Modelo balanceado: 0,1125.

O baseline original apresentou resultado ligeiramente superior, com diferença de 0,0063 na escala de 0 a 1.

Essa diferença não foi submetida a uma análise de incerteza estatística.

Portanto, não é possível afirmar que represente uma superioridade estatisticamente significativa.

#### Relação com o experimento anterior

Na etapa 2.15, o modelo balanceado apresentou Recall muito superior ao baseline original quando ambos foram avaliados com limiar fixo de 0,50.

Entretanto, a ponderação modificou significativamente a distribuição das probabilidades estimadas.

A comparação por capacidade demonstrou que, quando os modelos são avaliados com a mesma quantidade de alertas, seus desempenhos são semelhantes.

Isso sugere que parte importante da diferença observada anteriormente estava relacionada ao limiar e à distribuição das probabilidades, e não necessariamente a uma melhoria na ordenação dos riscos.

#### Implicações para a Torre de Controle

Uma política de priorização baseada nos registros de maior risco pode ser mais adequada para avaliar os modelos quando existe uma capacidade limitada de tratamento de alertas.

Entretanto, a capacidade operacional real ainda não foi definida.

Os percentuais utilizados representam cenários experimentais aplicados ao conjunto de validação, não limites operacionais confirmados.

Os resultados também demonstram que identificar uma parcela maior dos atrasos exige sinalizar uma quantidade considerável de coletas que não apresentarão atraso.

#### Limitações

A comparação foi realizada utilizando somente o conjunto de validação.

Foram avaliados dois modelos baseados no mesmo algoritmo de Regressão Logística.

Não foram investigados algoritmos capazes de representar relações não lineares mais complexas.

Também não foi realizada avaliação estatística da diferença entre os modelos.

O conjunto de teste permaneceu reservado.

#### Conclusão

A ponderação de classes não demonstrou uma melhoria consistente na priorização das coletas em risco quando os modelos foram comparados sob a mesma capacidade de alertas.

Os resultados indicam que o baseline original e o modelo balanceado apresentam capacidade de ordenação semelhante.

A Average Precision do baseline original foi ligeiramente superior.

Nenhum modelo foi selecionado como solução operacional definitiva.

Os resultados justificam investigar outros algoritmos, preservando a mesma metodologia de avaliação.

**Status da etapa 2.16: concluída.**

**Próxima etapa proposta: 2.17 — Estruturação de uma função reutilizável para treinamento e avaliação de múltiplos modelos de classificação.**

## 2.17 — Função reutilizável para experimentos de classificação

### Objetivo

Construir uma função Python reutilizável para treinar e avaliar diferentes algoritmos de classificação binária.

A função permitirá padronizar os experimentos, reduzir a repetição de código e facilitar a comparação de modelos utilizando os mesmos conjuntos de treinamento e validação.

### Contexto

Nas etapas anteriores, treinamos dois modelos de Regressão Logística:

- Baseline original, sem ponderação de classes.
- Modelo balanceado, utilizando `class_weight="balanced"`.

Os experimentos demonstraram que a alteração do limiar pode modificar significativamente as métricas de classificação.

Também verificamos que comparar modelos com a mesma quantidade de alertas permite avaliar sua capacidade de priorizar coletas com maior risco de atraso.

### Estratégia

A função será responsável por:

1. Receber um estimador de classificação compatível com a API do Scikit-learn.
2. Criar um pipeline com pré-processamento e classificador.
3. Treinar o pipeline exclusivamente no conjunto de treinamento.
4. Calcular probabilidades no conjunto de validação.
5. Calcular Average Precision.
6. Avaliar Precision, Recall e F1-score para um limiar informado.
7. Retornar o pipeline treinado e os resultados estruturados.

### Premissas

Serão utilizadas as mesmas 10 variáveis preditoras dos experimentos anteriores.

O pré-processamento será ajustado exclusivamente com os dados de treinamento.

O conjunto de teste permanecerá reservado.

A função não realizará seleção automática de modelo, otimização de hiperparâmetros ou definição do limiar operacional.

A comparação por capacidade de alertas poderá ser incorporada em uma etapa posterior, reutilizando a metodologia da etapa 2.16.

### Resultado esperado

Ao final, teremos uma função que poderá ser utilizada com diferentes algoritmos, como Regressão Logística, Random Forest e XGBoost.

Nesta etapa, somente definiremos e verificaremos a função, sem executar novos treinamentos.

In [16]:

# ============================================================
# ETAPA 2.17 — FUNÇÃO REUTILIZÁVEL DE EXPERIMENTAÇÃO
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)


def executar_experimento_classificacao(
    nome_modelo,
    estimador,
    X_treino,
    y_treino,
    X_validacao,
    y_validacao,
    variaveis_categoricas,
    variaveis_numericas,
    limiar=0.50
):
    """
    Treina e avalia um classificador binário.

    O pré-processamento é ajustado somente com os
    dados de treinamento.

    Retorna:
        pipeline treinado
        dicionário de métricas
        probabilidades de validação
    """

    # --------------------------------------------------------
    # 1. Validar parâmetros e dados
    # --------------------------------------------------------

    if not 0 < limiar < 1:
        raise ValueError(
            "O limiar deve estar entre 0 e 1."
        )

    if not hasattr(estimador, "predict_proba"):
        raise ValueError(
            "O estimador deve implementar predict_proba()."
        )

    colunas_esperadas = (
        list(variaveis_categoricas)
        + list(variaveis_numericas)
    )

    if len(colunas_esperadas) != len(set(colunas_esperadas)):
        raise ValueError(
            "Há variáveis repetidas na configuração."
        )

    if (
        list(X_treino.columns) != list(X_validacao.columns)
        or set(X_treino.columns) != set(colunas_esperadas)
    ):
        raise ValueError(
            "As colunas de treinamento, validação "
            "e pré-processamento não são compatíveis."
        )

    if not X_treino.index.equals(y_treino.index):
        raise ValueError(
            "Índices de X_treino e y_treino não coincidem."
        )

    if not X_validacao.index.equals(y_validacao.index):
        raise ValueError(
            "Índices de X_validacao e y_validacao "
            "não coincidem."
        )

    if set(y_treino.unique()) != {0, 1}:
        raise ValueError(
            "O treinamento deve conter as classes 0 e 1."
        )

    if set(y_validacao.unique()) != {0, 1}:
        raise ValueError(
            "A validação deve conter as classes 0 e 1."
        )

    # --------------------------------------------------------
    # 2. Construir o pré-processamento
    # --------------------------------------------------------

    preprocessamento = ColumnTransformer(
        transformers=[
            (
                "categoricas",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                list(variaveis_categoricas)
            ),
            (
                "numericas",
                StandardScaler(),
                list(variaveis_numericas)
            )
        ]
    )

    # --------------------------------------------------------
    # 3. Construir o pipeline
    # --------------------------------------------------------

    pipeline = Pipeline(
        steps=[
            ("preprocessamento", preprocessamento),
            ("classificador", clone(estimador))
        ]
    )

    # --------------------------------------------------------
    # 4. Treinar o modelo
    # --------------------------------------------------------

    inicio = time.perf_counter()

    pipeline.fit(
        X_treino,
        y_treino
    )

    tempo_treinamento = (
        time.perf_counter() - inicio
    )

    # --------------------------------------------------------
    # 5. Obter probabilidades e classificações
    # --------------------------------------------------------

    probabilidades = pipeline.predict_proba(
        X_validacao
    )[:, 1]

    previsoes = (
        probabilidades >= limiar
    ).astype(int)

    # --------------------------------------------------------
    # 6. Calcular métricas
    # --------------------------------------------------------

    tn, fp, fn, tp = confusion_matrix(
        y_validacao,
        previsoes,
        labels=[0, 1]
    ).ravel()

    metricas = {
        "Modelo": nome_modelo,
        "Limiar": limiar,
        "Accuracy": accuracy_score(
            y_validacao, previsoes
        ),
        "Precision": precision_score(
            y_validacao, previsoes,
            zero_division=0
        ),
        "Recall": recall_score(
            y_validacao, previsoes,
            zero_division=0
        ),
        "F1-score": f1_score(
            y_validacao, previsoes,
            zero_division=0
        ),
        "Average Precision": average_precision_score(
            y_validacao, probabilidades
        ),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "Alertas": int(previsoes.sum()),
        "Alertas (%)": (
            100 * previsoes.mean()
        ),
        "Tempo treinamento (s)": tempo_treinamento
    }

    # --------------------------------------------------------
    # 7. Retornar os resultados
    # --------------------------------------------------------

    return pipeline, metricas, probabilidades


# ============================================================
# VALIDAÇÃO DA DEFINIÇÃO
# ============================================================

assert callable(executar_experimento_classificacao)

print("=== ETAPA 2.17 ===")
print("Função definida com sucesso.")
print("Função reutilizável: OK")
print("Nenhum modelo treinado nesta etapa.")
print("Conjunto de teste não utilizado.")


=== ETAPA 2.17 ===
Função definida com sucesso.
Função reutilizável: OK
Nenhum modelo treinado nesta etapa.
Conjunto de teste não utilizado.


### Explicação do código — Etapa 2.17

#### 1. Funções em Python

A palavra-chave `def` permite criar uma função reutilizável.

Em vez de repetir todo o código para cada algoritmo, podemos passar diferentes estimadores para a mesma função.

O parâmetro `nome_modelo` identifica o experimento, enquanto `estimador` recebe o algoritmo que será treinado.

#### 2. Parâmetros e argumentos

Os parâmetros da função representam as informações necessárias para executar o experimento.

O argumento `limiar=0.50` define um valor padrão, que pode ser substituído quando a função for chamada.

O limiar é utilizado apenas para transformar probabilidades em classificações binárias. Ele não altera o treinamento.

#### 3. Validação das entradas

Antes de executar o treinamento, a função verifica se:

- O limiar está dentro do intervalo permitido.
- O estimador disponibiliza `predict_proba()`.
- As variáveis preditoras estão configuradas corretamente.
- Os conjuntos possuem colunas compatíveis.
- Os índices das variáveis preditoras correspondem aos índices das respectivas variáveis-alvo.
- As classes esperadas estão presentes.

Essas verificações ajudam a identificar problemas antes do treinamento.

#### 4. ColumnTransformer

O `ColumnTransformer` permite aplicar transformações diferentes a grupos de variáveis.

As variáveis categóricas passam por One-Hot Encoding.

As variáveis numéricas passam por padronização com StandardScaler.

O argumento `sparse_output=False` faz com que o OneHotEncoder produza uma matriz densa, favorecendo a compatibilidade com alguns algoritmos que poderão ser utilizados posteriormente.

Essa escolha pode aumentar o consumo de memória, especialmente em datasets com muitas categorias.

#### 5. Pipeline

O `Pipeline` organiza o pré-processamento e o classificador em uma única estrutura.

Durante o método `fit()`, as transformações são ajustadas somente com os dados de treinamento.

Na validação, as mesmas transformações são reaplicadas sem utilizar a variável-alvo para ajustar o pré-processamento.

Isso reduz o risco de data leakage.

#### 6. Clone do estimador

A função `clone()` cria um novo estimador com os mesmos hiperparâmetros, mas sem preservar um eventual estado de treinamento anterior.

Assim, podemos reutilizar configurações de algoritmos sem modificar diretamente os objetos originais.

#### 7. Medição do tempo de treinamento

A função `time.perf_counter()` permite medir o tempo decorrido durante o treinamento.

Esse indicador será útil para comparar o custo computacional dos algoritmos.

#### 8. Métricas de avaliação

A função calcula Accuracy, Precision, Recall, F1-score e Average Precision.

Também registra os componentes da matriz de confusão e a quantidade de alertas emitidos.

Essas métricas permitem avaliar tanto o desempenho estatístico quanto o impacto potencial na operação.

#### 9. Retorno de múltiplos resultados

A instrução `return` devolve três objetos:

1. O pipeline treinado.
2. Um dicionário com as métricas.
3. Um array com as probabilidades estimadas para a validação.

Esses resultados poderão ser armazenados em estruturas reutilizáveis e, futuramente, registrados no MLflow.

### Limitações desta implementação

A função utiliza um pré-processamento comum para os algoritmos.

Essa configuração constitui uma base inicial, mas poderá exigir ajustes específicos para determinados classificadores.

O código ainda não realiza validação cruzada, otimização de hiperparâmetros, comparação por capacidade de alertas ou análise de importância das variáveis.

Essas funcionalidades poderão ser incorporadas gradualmente, após validarmos a estrutura inicial.

### Próxima validação

Nesta etapa, verificaremos somente se a função foi definida corretamente.

O primeiro treinamento utilizando essa função será realizado em uma etapa posterior, permitindo validar seu comportamento antes de automatizar experimentos com vários algoritmos.

## 2.18 — Validação da função reutilizável com Regressão Logística

### Objetivo

Validar a função `executar_experimento_classificacao()`, criada na etapa 2.17, utilizando o algoritmo de Regressão Logística já empregado no baseline original.

O objetivo é verificar se a função consegue reproduzir resultados equivalentes aos obtidos anteriormente, confirmando que nossa estrutura reutilizável está funcionando corretamente antes de aplicá-la a outros algoritmos.

### Contexto

Na etapa 2.17, construímos uma função capaz de:

- Receber um algoritmo de classificação.
- Preparar as variáveis categóricas e numéricas.
- Construir e treinar um pipeline.
- Calcular probabilidades no conjunto de validação.
- Avaliar métricas de classificação.
- Retornar o modelo treinado e os resultados do experimento.

Até o momento, validamos apenas a definição da função. Ainda não executamos um treinamento utilizando essa estrutura.

### Estratégia de validação

Executaremos um novo treinamento de Regressão Logística utilizando:

- As mesmas 10 variáveis preditoras do baseline original.
- Os mesmos conjuntos de treinamento e validação.
- Os mesmos hiperparâmetros do classificador: `class_weight=None`, `max_iter=1000` e `random_state=42`.
- O mesmo limiar de classificação de 0,50.

Compararemos os resultados com as métricas registradas na etapa 2.12.

| Métrica | Baseline original |
|---|---:|
| Accuracy | 0,9745 |
| Precision | 0,0000 |
| Recall | 0,0000 |
| F1-score | 0,0000 |
| Average Precision | 0,1188 |
| Verdadeiros positivos (TP) | 0 |
| Falsos positivos (FP) | 7 |
| Falsos negativos (FN) | 5.448 |
| Verdadeiros negativos (TN) | 208.174 |

### Cuidados metodológicos

A função criada na etapa 2.17 utiliza `OneHotEncoder(sparse_output=False)`, enquanto o pipeline original utilizava `sparse_output=True`.

Essa diferença modifica a representação computacional das variáveis categóricas, mas não altera seu significado.

Portanto, esperamos resultados equivalentes, embora pequenas diferenças numéricas possam ocorrer.

O novo treinamento será realizado exclusivamente com o conjunto de treinamento.

O conjunto de validação será utilizado apenas para avaliar os resultados.

O conjunto de teste permanecerá reservado.

### Critérios de validação

Verificaremos:

1. Se a função executa o treinamento sem erros.
2. Se o pipeline treinado é retornado corretamente.
3. Se as probabilidades estimadas são válidas.
4. Se as métricas são consistentes.
5. Se os resultados são equivalentes aos do baseline original, considerando possíveis diferenças numéricas.
6. Se os objetos dos experimentos anteriores permanecem preservados.

### Resultado esperado

Ao final, teremos uma primeira execução real da função reutilizável.

Se a validação for satisfatória, poderemos utilizá-la posteriormente para experimentar outros algoritmos, como Random Forest e XGBoost, mantendo um procedimento padronizado de avaliação.

**Nesta etapa, não selecionaremos um modelo definitivo nem realizaremos otimização de hiperparâmetros.**

In [17]:

# ============================================================
# ETAPA 2.18 — VALIDAÇÃO DA FUNÇÃO REUTILIZÁVEL
# ============================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

# ------------------------------------------------------------
# 1. Recuperar a configuração das 10 variáveis preditoras
# ------------------------------------------------------------

categoricas_218 = [
    "region_id",
    "aoi_type"
]

numericas_218 = [
    "lng",
    "lat",
    "janela_min",
    "aceite_ate_inicio_janela_min",
    "aceite_ate_fim_janela_min",
    "hora_aceite",
    "hora_inicio_janela",
    "hora_fim_janela"
]

# ------------------------------------------------------------
# 2. Configurar o mesmo algoritmo do baseline original
# ------------------------------------------------------------

estimador_218 = LogisticRegression(
    class_weight=None,
    max_iter=1000,
    random_state=42
)

# ------------------------------------------------------------
# 3. Executar a função criada na etapa 2.17
# ------------------------------------------------------------

pipeline_218, metricas_218, probabilidades_218 = (
    executar_experimento_classificacao(
        nome_modelo="Regressão Logística — função reutilizável",
        estimador=estimador_218,
        X_treino=X_treino,
        y_treino=y_treino,
        X_validacao=X_validacao,
        y_validacao=y_validacao,
        variaveis_categoricas=categoricas_218,
        variaveis_numericas=numericas_218,
        limiar=0.50
    )
)

# ------------------------------------------------------------
# 4. Recuperar os resultados do baseline original
# ------------------------------------------------------------

probabilidades_baseline_218 = (
    pipeline_baseline.predict_proba(X_validacao)[:, 1]
)

previsoes_baseline_218 = (
    probabilidades_baseline_218 >= 0.50
).astype(int)

previsoes_funcao_218 = (
    probabilidades_218 >= 0.50
).astype(int)

# ------------------------------------------------------------
# 5. Comparar as métricas
# ------------------------------------------------------------

metricas_baseline_218 = {
    "Accuracy": 0.9745,
    "Precision": 0.0,
    "Recall": 0.0,
    "F1-score": 0.0,
    "Average Precision": average_precision_score(
        y_validacao,
        probabilidades_baseline_218
    ),
    "TN": 208174,
    "FP": 7,
    "FN": 5448,
    "TP": 0
}

metricas_comparadas_218 = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "Average Precision",
    "TN",
    "FP",
    "FN",
    "TP"
]

resultados_comparacao_218 = []

for metrica in metricas_comparadas_218:

    valor_original = metricas_baseline_218[metrica]
    valor_funcao = metricas_218[metrica]

    resultados_comparacao_218.append({
        "Métrica": metrica,
        "Baseline original": valor_original,
        "Função reutilizável": valor_funcao,
        "Diferença": valor_funcao - valor_original
    })

df_comparacao_218 = pd.DataFrame(
    resultados_comparacao_218
)

# ------------------------------------------------------------
# 6. Comparar probabilidades e classificações
# ------------------------------------------------------------

diferenca_probabilidades_218 = np.abs(
    probabilidades_baseline_218 - probabilidades_218
)

classificacoes_diferentes_218 = int(
    np.count_nonzero(
        previsoes_baseline_218 != previsoes_funcao_218
    )
)

# ------------------------------------------------------------
# 7. Validar os resultados
# ------------------------------------------------------------

assert len(probabilidades_218) == len(y_validacao)

assert np.isfinite(probabilidades_218).all()

assert (
    (probabilidades_218 >= 0)
    & (probabilidades_218 <= 1)
).all()

assert (
    metricas_218["TN"]
    + metricas_218["FP"]
    + metricas_218["FN"]
    + metricas_218["TP"]
    == len(y_validacao)
)

assert (
    metricas_218["TP"] + metricas_218["FN"]
    == int(y_validacao.sum())
)

assert pipeline_218 is not pipeline_baseline

assert (
    pipeline_218.named_steps["classificador"]
    is not pipeline_baseline.named_steps["classificador"]
)

# ------------------------------------------------------------
# 8. Exibir resultados
# ------------------------------------------------------------

print("=== COMPARAÇÃO COM O BASELINE ORIGINAL ===")

print(
    df_comparacao_218.to_string(
        index=False,
        float_format=lambda valor: f"{valor:.6f}"
    )
)

print("\n=== COMPARAÇÃO DAS PROBABILIDADES ===")

print(
    "Diferença média absoluta:",
    f"{diferenca_probabilidades_218.mean():.8f}"
)

print(
    "Diferença máxima absoluta:",
    f"{diferenca_probabilidades_218.max():.8f}"
)

print(
    "Classificações diferentes:",
    classificacoes_diferentes_218
)

print("\n=== TREINAMENTO ===")

print(
    "Tempo de treinamento:",
    f"{metricas_218['Tempo treinamento (s)']:.2f} segundos"
)

print(
    "Iterações do classificador:",
    pipeline_218.named_steps["classificador"].n_iter_[0]
)

print("\n=== VALIDAÇÕES ===")
print("Função executada: OK")
print("Probabilidades válidas: OK")
print("Matriz de confusão consistente: OK")
print("Pipeline original preservado: OK")
print("Conjunto de teste não utilizado.")


=== COMPARAÇÃO COM O BASELINE ORIGINAL ===
          Métrica  Baseline original  Função reutilizável  Diferença
         Accuracy           0.974500             0.974465  -0.000035
        Precision           0.000000             0.000000   0.000000
           Recall           0.000000             0.000000   0.000000
         F1-score           0.000000             0.000000   0.000000
Average Precision           0.118805             0.118805   0.000000
               TN      208174.000000        208174.000000   0.000000
               FP           7.000000             7.000000   0.000000
               FN        5448.000000          5448.000000   0.000000
               TP           0.000000             0.000000   0.000000

=== COMPARAÇÃO DAS PROBABILIDADES ===
Diferença média absoluta: 0.00000000
Diferença máxima absoluta: 0.00000000
Classificações diferentes: 0

=== TREINAMENTO ===
Tempo de treinamento: 10.91 segundos
Iterações do classificador: 46

=== VALIDAÇÕES ===
Função executad

### Explicação do código — Etapa 2.18

#### 1. Configuração das variáveis preditoras

Definimos as mesmas 10 variáveis utilizadas no baseline original, separando as duas categóricas das oito numéricas.

Essa configuração permite manter o mesmo conjunto de informações durante a comparação.

#### 2. Instanciação do estimador

Criamos um novo objeto `LogisticRegression` com os mesmos hiperparâmetros do baseline.

Instanciar um estimador significa criar um objeto que contém a configuração do algoritmo, mas que ainda não foi treinado.

#### 3. Chamada da função reutilizável

Utilizamos `executar_experimento_classificacao()` para realizar o treinamento e a avaliação.

A função retorna três objetos:

- `pipeline_218`: pipeline treinado.
- `metricas_218`: dicionário com os resultados da avaliação.
- `probabilidades_218`: probabilidades estimadas no conjunto de validação.

Esse mecanismo permite substituir o algoritmo em experimentos futuros sem reescrever toda a rotina.

#### 4. Recuperação do baseline

Utilizamos o pipeline original, já treinado, para calcular novamente suas probabilidades no conjunto de validação.

As métricas de classificação registradas na etapa 2.12 são utilizadas como referência.

A Average Precision é recalculada com as probabilidades originais para permitir uma comparação com maior precisão numérica.

#### 5. Comparação das métricas

Criamos um DataFrame contendo os resultados dos dois experimentos.

A coluna `Diferença` representa o valor obtido pela função reutilizável menos o valor de referência.

As métricas Accuracy, Precision, Recall, F1-score e Average Precision são comparadas juntamente com os componentes da matriz de confusão.

#### 6. Comparação das probabilidades

A função `np.abs()` calcula o valor absoluto das diferenças entre as probabilidades estimadas pelos dois pipelines.

A diferença média absoluta resume a magnitude média dessas diferenças.

A diferença máxima absoluta identifica a maior divergência individual.

Também verificamos quantas classificações binárias são diferentes entre os modelos, considerando o limiar de 0,50.

#### 7. Validações com assert

A instrução `assert` interrompe a execução quando uma condição esperada não é atendida.

Verificamos se as probabilidades são válidas, se a matriz de confusão é consistente e se o novo pipeline é um objeto independente do baseline original.

Essas verificações não garantem que os dois modelos produzam resultados equivalentes. A equivalência será analisada a partir das diferenças observadas.

#### 8. Tempo e iterações

Registramos o tempo de treinamento e o número de iterações executadas pela Regressão Logística.

Essas informações ajudam a avaliar o comportamento computacional da função.

### Critério de interpretação

Esperamos resultados próximos aos do baseline original, pois os dados, o algoritmo e os hiperparâmetros são os mesmos.

Entretanto, a diferença entre representações esparsas e densas pode produzir pequenas variações numéricas.

Se surgirem divergências relevantes, investigaremos suas causas antes de utilizar a função para comparar outros algoritmos.

O conjunto de teste continuará reservado.

### 2.18 — Resultados, interpretação e conclusão

#### Resultados obtidos

A função `executar_experimento_classificacao()` foi executada utilizando o algoritmo de Regressão Logística, com os mesmos dados e hiperparâmetros do baseline original.

O treinamento foi concluído em 11,86 segundos, com 46 iterações.

A comparação apresentou os seguintes resultados:

| Indicador | Resultado |
|---|---|
| Accuracy | 0,974465 |
| Precision | 0,000000 |
| Recall | 0,000000 |
| F1-score | 0,000000 |
| Average Precision | 0,118805 |
| Verdadeiros negativos | 208.174 |
| Falsos positivos | 7 |
| Falsos negativos | 5.448 |
| Verdadeiros positivos | 0 |
| Classificações divergentes | 0 |

A diferença média absoluta e a diferença máxima absoluta entre as probabilidades foram exibidas como 0,00000000.

Todas as verificações de consistência foram aprovadas.

#### Interpretação

A função reutilizável reproduziu as classificações e as métricas do baseline original.

A pequena diferença apresentada na Accuracy decorreu da comparação entre um valor de referência arredondado e um valor calculado com maior precisão.

Embora os pipelines utilizem representações computacionais diferentes para as variáveis categóricas, não foram observadas divergências nas classificações.

As diferenças entre probabilidades foram nulas na precisão decimal exibida.

O tempo de treinamento foi superior ao registrado no baseline original. Entretanto, uma única medição não permite atribuir essa diferença exclusivamente à representação densa dos dados.

#### Validação da função

A execução demonstrou que a função é capaz de:

- Construir e treinar um pipeline de classificação.
- Aplicar o pré-processamento configurado.
- Calcular probabilidades e métricas de validação.
- Retornar resultados estruturados.
- Preservar o pipeline original.

A validação foi realizada com Regressão Logística. A compatibilidade e o comportamento com outros algoritmos ainda deverão ser verificados individualmente.

#### Conclusão

A função reutilizável foi validada para o experimento de Regressão Logística, reproduzindo os resultados do baseline original.

Essa validação fornece uma referência confiável para ampliar os experimentos com diferentes algoritmos de classificação.

Entretanto, a equivalência dos resultados não representa melhoria do desempenho preditivo. O modelo continua apresentando Recall igual a zero no limiar de 0,50.

A avaliação por diferentes limiares ou capacidades de alertas continuará sendo necessária para analisar sua utilidade operacional.

O conjunto de teste permaneceu reservado.

**Status da etapa 2.18: concluída.**

**Próxima etapa proposta: 2.19 — Primeiro experimento com um novo algoritmo utilizando a função reutilizável.**

## 2.19 — Primeiro experimento com Random Forest

### Objetivo

Treinar e avaliar um modelo Random Forest utilizando a função reutilizável `executar_experimento_classificacao()`, validada na etapa 2.18.

O objetivo é investigar se um algoritmo baseado em árvores de decisão consegue identificar padrões de atraso que não foram adequadamente representados pela Regressão Logística.

### Contexto

Nos experimentos anteriores, utilizamos Regressão Logística para prever se uma coleta seria realizada após o limite da janela operacional.

Os resultados demonstraram alguma capacidade de ordenação dos riscos, mas também limitações na identificação de atrasos e na quantidade de falsos alertas.

A comparação entre os modelos original e balanceado indicou que a ponderação de classes não trouxe uma melhoria consistente quando controlamos a quantidade de alertas.

Agora investigaremos uma família diferente de algoritmos.

### Conceito: Random Forest

Random Forest é um algoritmo de aprendizado supervisionado baseado em um conjunto de árvores de decisão.

Cada árvore aprende regras de classificação a partir dos dados de treinamento.

As previsões das diferentes árvores são combinadas para produzir a classificação final e estimar probabilidades.

Diferentemente da Regressão Logística, o Random Forest consegue representar relações não lineares e interações entre variáveis sem exigir que todas essas relações sejam definidas manualmente.

### Hipótese do experimento

As características temporais e geográficas das coletas podem apresentar interações relevantes para a previsão de atrasos.

Por exemplo, o efeito do horário de aceite pode variar conforme a região, o tipo de área e o tempo disponível até o fim da janela.

O Random Forest poderá investigar esses padrões utilizando as variáveis já disponíveis.

Entretanto, não podemos assumir antecipadamente que ele apresentará desempenho superior.

### Estratégia experimental

Utilizaremos:

- As mesmas 10 variáveis preditoras dos experimentos anteriores.
- O mesmo conjunto de treinamento.
- O mesmo conjunto de validação.
- A função reutilizável criada na etapa 2.17.
- Um Random Forest com configuração inicial controlada e `random_state=42`.

O experimento será realizado inicialmente sem ponderação de classes e sem otimização automática de hiperparâmetros.

A configuração considerará o volume do dataset e os recursos computacionais disponíveis, evitando começar com um modelo excessivamente complexo.

### Métricas de avaliação

Serão analisadas:

- Accuracy.
- Precision.
- Recall.
- F1-score.
- Average Precision.
- Matriz de confusão.
- Quantidade de alertas.
- Tempo de treinamento.

O limiar inicial será 0,50, apenas como referência.

Posteriormente, poderemos comparar os modelos com a mesma capacidade de alertas, conforme a metodologia da etapa 2.16.

### Cuidados metodológicos

O treinamento utilizará exclusivamente o conjunto de treinamento.

O conjunto de validação será utilizado para avaliar os resultados.

O conjunto de teste permanecerá reservado.

Não adicionaremos novas variáveis nesta etapa, para preservar a comparabilidade com a Regressão Logística.

Não realizaremos seleção definitiva de modelo com base apenas no limiar de 0,50.

### Resultado esperado

Obter um primeiro resultado de Random Forest e compará-lo com o baseline de Regressão Logística.

O experimento permitirá investigar se o aumento da flexibilidade do algoritmo produz ganhos preditivos relevantes.

Independentemente do resultado, documentaremos as evidências e as limitações identificadas.

In [18]:

# ============================================================
# ETAPA 2.19 — PRIMEIRO EXPERIMENTO COM RANDOM FOREST
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier

# ------------------------------------------------------------
# 1. Configurar o algoritmo
# ------------------------------------------------------------

estimador_rf_219 = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=20,
    max_features="sqrt",
    class_weight=None,
    n_jobs=4,
    random_state=42
)

# ------------------------------------------------------------
# 2. Executar a função reutilizável
# ------------------------------------------------------------

pipeline_rf_219, metricas_rf_219, probabilidades_rf_219 = (
    executar_experimento_classificacao(
        nome_modelo="Random Forest",
        estimador=estimador_rf_219,
        X_treino=X_treino,
        y_treino=y_treino,
        X_validacao=X_validacao,
        y_validacao=y_validacao,
        variaveis_categoricas=categoricas_218,
        variaveis_numericas=numericas_218,
        limiar=0.50
    )
)

# ------------------------------------------------------------
# 3. Organizar as métricas
# ------------------------------------------------------------

df_metricas_rf_219 = pd.DataFrame(
    [metricas_rf_219]
)

# ------------------------------------------------------------
# 4. Inspecionar o modelo treinado
# ------------------------------------------------------------

modelo_rf_treinado_219 = (
    pipeline_rf_219.named_steps["classificador"]
)

total_arvores_219 = len(
    modelo_rf_treinado_219.estimators_
)

profundidades_219 = np.array([
    arvore.tree_.max_depth
    for arvore in modelo_rf_treinado_219.estimators_
])

# ------------------------------------------------------------
# 5. Validar os resultados
# ------------------------------------------------------------

assert len(probabilidades_rf_219) == len(y_validacao)

assert np.isfinite(probabilidades_rf_219).all()

assert (
    (probabilidades_rf_219 >= 0)
    & (probabilidades_rf_219 <= 1)
).all()

assert (
    metricas_rf_219["TN"]
    + metricas_rf_219["FP"]
    + metricas_rf_219["FN"]
    + metricas_rf_219["TP"]
    == len(y_validacao)
)

assert (
    metricas_rf_219["TP"]
    + metricas_rf_219["FN"]
    == int(y_validacao.sum())
)

assert total_arvores_219 == 100

assert pipeline_rf_219 is not pipeline_baseline

# ------------------------------------------------------------
# 6. Exibir resultados
# ------------------------------------------------------------

print("=== RANDOM FOREST — CONFIGURAÇÃO ===")
print("Árvores:", total_arvores_219)
print("Profundidade máxima configurada: 12")
print("Mínimo de amostras por folha: 20")
print("Ponderação de classes: nenhuma")
print("Processamento paralelo: 4 jobs")

print("\n=== ESTRUTURA DAS ÁRVORES ===")
print(
    "Profundidade mínima:",
    profundidades_219.min()
)
print(
    "Profundidade média:",
    f"{profundidades_219.mean():.2f}"
)
print(
    "Profundidade máxima:",
    profundidades_219.max()
)

print("\n=== MÉTRICAS DE VALIDAÇÃO ===")

print(
    df_metricas_rf_219.to_string(
        index=False,
        float_format=lambda valor: f"{valor:.6f}"
    )
)

print("\n=== VALIDAÇÕES ===")
print("Probabilidades válidas: OK")
print("Matriz de confusão consistente: OK")
print("Quantidade de árvores: OK")
print("Pipeline original preservado: OK")
print("Conjunto de teste não utilizado.")


=== RANDOM FOREST — CONFIGURAÇÃO ===
Árvores: 100
Profundidade máxima configurada: 12
Mínimo de amostras por folha: 20
Ponderação de classes: nenhuma
Processamento paralelo: 4 jobs

=== ESTRUTURA DAS ÁRVORES ===
Profundidade mínima: 12
Profundidade média: 12.00
Profundidade máxima: 12

=== MÉTRICAS DE VALIDAÇÃO ===
       Modelo   Limiar  Accuracy  Precision   Recall  F1-score  Average Precision     TN  FP   FN  TP  Alertas  Alertas (%)  Tempo treinamento (s)
Random Forest 0.500000  0.975130   0.868852 0.029185  0.056473           0.179512 208157  24 5289 159      183     0.085663             152.573258

=== VALIDAÇÕES ===
Probabilidades válidas: OK
Matriz de confusão consistente: OK
Quantidade de árvores: OK
Pipeline original preservado: OK
Conjunto de teste não utilizado.


### Explicação do código — Etapa 2.19

#### 1. RandomForestClassifier

O `RandomForestClassifier` é um algoritmo de classificação baseado em múltiplas árvores de decisão.

Cada árvore aprende regras a partir de amostras dos dados de treinamento. O conjunto combina as previsões das árvores para produzir probabilidades de classificação.

O Random Forest consegue representar relações não lineares e interações entre variáveis.

Isso é relevante para nossa hipótese de que o risco de atraso pode depender da combinação entre horário, localização e tempo disponível para a coleta.

#### 2. Hiperparâmetros utilizados

**`n_estimators=100`**

Define a quantidade de árvores que compõem a floresta.

Mais árvores podem melhorar a estabilidade das previsões, mas também aumentam o custo computacional.

**`max_depth=12`**

Limita a profundidade máxima das árvores.

Essa restrição ajuda a controlar a complexidade do modelo e reduzir o risco de sobreajuste (*overfitting*).

**`min_samples_leaf=20`**

Exige pelo menos 20 registros em cada folha das árvores.

Isso evita criar folhas baseadas em quantidades muito pequenas de observações.

**`max_features="sqrt"`**

Em cada divisão de uma árvore, o algoritmo considera um subconjunto aleatório de variáveis candidatas.

Essa estratégia contribui para a diversidade das árvores.

**`class_weight=None`**

Não aplicamos ponderação de classes neste primeiro experimento.

Assim, teremos uma referência inicial antes de investigar possíveis estratégias para lidar com o desbalanceamento.

**`n_jobs=4`**

Permite utilizar até quatro processos de trabalho paralelos para operações compatíveis do algoritmo.

O objetivo é aproveitar parte da capacidade computacional disponível sem utilizar todos os recursos da máquina.

**`random_state=42`**

Define uma semente para os procedimentos aleatórios do algoritmo, favorecendo a reprodutibilidade do experimento.

#### 3. Reutilização da função

A função `executar_experimento_classificacao()` executa o pré-processamento, o treinamento e a avaliação.

A principal diferença em relação à etapa 2.18 é o estimador fornecido.

Isso demonstra como podemos reutilizar a mesma estrutura com diferentes algoritmos.

#### 4. Inspeção das árvores

O atributo `estimators_` contém as árvores treinadas.

Para cada árvore, consultamos `tree_.max_depth` para verificar a profundidade efetivamente alcançada.

Essa informação ajuda a compreender a complexidade do modelo.

#### 5. Avaliação

Utilizamos as mesmas métricas dos experimentos anteriores:

- Accuracy.
- Precision.
- Recall.
- F1-score.
- Average Precision.
- Matriz de confusão.
- Quantidade de alertas.
- Tempo de treinamento.

O limiar de 0,50 é utilizado apenas como referência inicial.

A comparação definitiva deverá considerar também a capacidade de priorização dos atrasos sob volumes equivalentes de alertas.

#### 6. Validações

As instruções `assert` verificam a consistência das probabilidades, da matriz de confusão e da quantidade de árvores.

Também confirmamos que o novo pipeline é um objeto independente do baseline original.

#### 7. Considerações sobre memória

O pipeline reutilizável utiliza One-Hot Encoding com saída densa.

Como o dataset possui aproximadamente 997 mil registros de treinamento, essa representação pode consumir uma quantidade considerável de memória.

Além disso, o Random Forest precisa armazenar suas árvores durante o treinamento.

Os hiperparâmetros foram escolhidos para iniciar com uma complexidade moderada, mas o consumo real dependerá da estrutura dos dados e do ambiente.

Caso ocorram problemas de memória ou desempenho, investigaremos o comportamento antes de alterar a configuração.

### Resultado esperado

Obter o primeiro modelo Random Forest treinado com as mesmas variáveis preditoras dos experimentos anteriores.

Após a execução, analisaremos suas métricas e verificaremos se existem indícios de melhoria na capacidade de identificação dos atrasos.

Ainda não selecionaremos um modelo definitivo.

### 2.19 — Resultados, interpretação e conclusão

#### Resultados obtidos

O modelo Random Forest foi treinado utilizando a função reutilizável `executar_experimento_classificacao()`.

O experimento utilizou as mesmas 10 variáveis preditoras e os mesmos conjuntos de treinamento e validação dos experimentos anteriores.

A configuração inicial incluiu 100 árvores, profundidade máxima de 12, mínimo de 20 amostras por folha e ausência de ponderação de classes.

O treinamento foi concluído em aproximadamente 130,44 segundos.

Todas as 100 árvores atingiram a profundidade máxima configurada.

#### Métricas de validação

| Métrica | Resultado |
|---|---:|
| Accuracy | 0,975130 |
| Precision | 0,868852 |
| Recall | 0,029185 |
| F1-score | 0,056473 |
| Average Precision | 0,179512 |
| Verdadeiros negativos | 208.157 |
| Falsos positivos | 24 |
| Falsos negativos | 5.289 |
| Verdadeiros positivos | 159 |
| Alertas emitidos | 183 |
| Percentual de alertas | 0,085663% |

#### Interpretação

O Random Forest apresentou Precision elevada no limiar de 0,50, indicando que a maior parte dos alertas emitidos correspondeu a atrasos reais.

Entretanto, o Recall foi de apenas 2,92%, demonstrando que a maioria dos atrasos permaneceu sem identificação nesse limiar.

A elevada Accuracy deve ser interpretada com cautela, considerando que aproximadamente 97,45% das observações pertencem à classe sem atraso.

A Average Precision foi de 0,179512, superior à obtida pela Regressão Logística original, de aproximadamente 0,118805.

Essa diferença corresponde a um aumento absoluto de aproximadamente 0,0607 e relativo de aproximadamente 51%.

O resultado fornece evidência inicial de que o Random Forest apresenta melhor capacidade global de ordenação dos riscos no conjunto de validação.

Contudo, ainda não foi realizada uma comparação por capacidade equivalente de alertas.

#### Relação com a hipótese de negócio

O Random Forest consegue representar relações não lineares e interações entre variáveis temporais, geográficas e operacionais.

O desempenho observado é compatível com a hipótese de que essas relações podem contribuir para a identificação de atrasos.

Entretanto, o experimento não demonstra quais variáveis ou interações explicam a melhoria.

Também não permite atribuir os resultados especificamente a fatores como trânsito, demanda operacional ou modalidade de transporte.

Essas hipóteses exigirão investigações adicionais.

#### Limitações

O experimento utilizou uma única configuração inicial de hiperparâmetros.

Não foram realizados ajustes de limiar, validação cruzada ou otimização de hiperparâmetros.

Todas as árvores atingiram a profundidade máxima configurada, indicando que essa restrição influenciou a complexidade do modelo.

O desempenho no treinamento não foi comparado ao desempenho na validação para investigar possíveis sinais de sobreajuste.

A diferença de Average Precision em relação à Regressão Logística ainda não foi submetida a uma análise de incerteza estatística.

O conjunto de teste permaneceu reservado.

#### Conclusão

O primeiro experimento com Random Forest foi executado e validado com sucesso.

O algoritmo apresentou Average Precision superior à da Regressão Logística original, fornecendo evidência inicial de melhoria na capacidade de priorização das coletas com risco de atraso.

Apesar da Precision elevada, o Recall de 2,92% no limiar de 0,50 demonstra que o modelo ainda identifica uma parcela muito pequena dos atrasos reais.

Não é possível concluir que o modelo seja operacionalmente adequado com base nesses resultados.

A próxima investigação deverá comparar o Random Forest e a Regressão Logística sob capacidades equivalentes de alertas, utilizando a metodologia estabelecida na etapa 2.16.

**Status da etapa 2.19: concluída.**

**Próxima etapa proposta: 2.20 — Comparação do Random Forest e da Regressão Logística por capacidade de alertas.**

## 2.20 — Comparação dos modelos por capacidade de alertas

### Objetivo

Comparar a Regressão Logística original, a Regressão Logística balanceada e o Random Forest sob as mesmas condições de capacidade operacional.

A pergunta que queremos responder é:

**Se a Torre de Controle puder investigar apenas uma determinada quantidade de coletas, qual modelo conseguirá priorizar mais atrasos reais?**

### Fundamentação

Nos experimentos anteriores, observamos que o limiar de probabilidade de 0,50 produz quantidades muito diferentes de alertas entre os modelos.

Isso dificulta a comparação direta de Precision e Recall nesse limiar.

Para realizar uma avaliação operacional mais justa, selecionaremos, em cada modelo, a mesma quantidade de coletas com maiores probabilidades estimadas de atraso.

Assim, poderemos comparar a capacidade de priorização dos algoritmos independentemente das diferenças de escala das probabilidades.

### Metodologia

Utilizaremos exclusivamente o conjunto de validação, contendo 213.629 coletas e 5.448 atrasos reais.

Serão avaliados três modelos:

1. Regressão Logística original.
2. Regressão Logística balanceada.
3. Random Forest.

Para cada modelo, ordenaremos as coletas pela probabilidade estimada de atraso, da maior para a menor.

Repetiremos a avaliação considerando capacidades de 1%, 2%, 5%, 10%, 15%, 20% e 30% das coletas.

Em cada capacidade, selecionaremos exatamente a mesma quantidade de registros para os três modelos.

### Indicadores

Calcularemos:

- **Verdadeiros positivos (TP):** atrasos corretamente priorizados.
- **Falsos positivos (FP):** coletas priorizadas que não atrasaram.
- **Falsos negativos (FN):** atrasos que ficaram fora da seleção.
- **Precision:** proporção de atrasos reais entre os alertas.
- **Recall:** proporção dos atrasos reais identificados.
- **F1-score:** equilíbrio entre Precision e Recall.

Também utilizaremos a Average Precision como indicador complementar de desempenho global.

### Cuidados metodológicos

A seleção por percentual representa uma avaliação retrospectiva da capacidade de priorização dos modelos.

Ela não significa que uma operação real precise esperar acumular todas as coletas para produzir previsões.

Na aplicação operacional, cada coleta poderá receber uma estimativa de risco no momento do aceite.

A definição de políticas de alerta em tempo real será tratada posteriormente.

Não realizaremos novos treinamentos nesta etapa.

Não modificaremos os modelos existentes nem utilizaremos o conjunto de teste.

### Resultado esperado

Produzir uma tabela comparativa mostrando quantos atrasos reais cada modelo consegue identificar sob diferentes capacidades de alertas.

A análise permitirá verificar se a melhoria de Average Precision apresentada pelo Random Forest também se traduz em vantagens operacionais nas capacidades avaliadas.

Os resultados orientarão os próximos experimentos, sem determinar antecipadamente um modelo vencedor.

In [19]:

# ============================================================
# ETAPA 2.20 — COMPARAÇÃO POR CAPACIDADE DE ALERTAS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import average_precision_score

# ------------------------------------------------------------
# 1. Definir os modelos e recuperar suas probabilidades
# ------------------------------------------------------------

modelos_220 = {
    "Regressão Logística original": pipeline_baseline,
    "Regressão Logística balanceada": pipeline_balanceado,
    "Random Forest": pipeline_rf_219
}

capacidades_220 = [1, 2, 5, 10, 15, 20, 30]

total_registros_220 = len(y_validacao)
total_atrasos_220 = int(y_validacao.sum())

# Converter o alvo para array, preservando a ordem original
y_real_220 = y_validacao.to_numpy()

# ------------------------------------------------------------
# 2. Calcular as probabilidades dos três modelos
# ------------------------------------------------------------

probabilidades_modelos_220 = {}

for nome_modelo, pipeline in modelos_220.items():

    probabilidades = pipeline.predict_proba(
        X_validacao
    )[:, 1]

    assert len(probabilidades) == total_registros_220
    assert np.isfinite(probabilidades).all()
    assert np.all((probabilidades >= 0) & (probabilidades <= 1))

    probabilidades_modelos_220[nome_modelo] = probabilidades

# ------------------------------------------------------------
# 3. Comparar os modelos nas mesmas capacidades
# ------------------------------------------------------------

resultados_220 = []

for nome_modelo, probabilidades in probabilidades_modelos_220.items():

    # Ordenar índices por probabilidade decrescente.
    # Em empates, manter uma ordem determinística por posição.
    ordem_risco = np.lexsort((
        np.arange(total_registros_220),
        -probabilidades
    ))

    average_precision = average_precision_score(
        y_real_220,
        probabilidades
    )

    for capacidade in capacidades_220:

        # Arredondar a quantidade de alertas para cima
        quantidade_alertas = int(
            np.ceil(
                total_registros_220 * capacidade / 100
            )
        )

        # Selecionar as coletas de maior risco
        indices_priorizados = ordem_risco[
            :quantidade_alertas
        ]

        # Quantidade de atrasos corretamente identificados
        tp = int(
            y_real_220[indices_priorizados].sum()
        )

        # Demais componentes da avaliação
        fp = quantidade_alertas - tp
        fn = total_atrasos_220 - tp

        precision = (
            tp / quantidade_alertas
            if quantidade_alertas > 0
            else 0.0
        )

        recall = (
            tp / total_atrasos_220
            if total_atrasos_220 > 0
            else 0.0
        )

        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall) > 0
            else 0.0
        )

        resultados_220.append({
            "Modelo": nome_modelo,
            "Capacidade (%)": capacidade,
            "Alertas": quantidade_alertas,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "Precision": precision,
            "Recall": recall,
            "F1-score": f1,
            "Average Precision": average_precision
        })

# ------------------------------------------------------------
# 4. Organizar os resultados
# ------------------------------------------------------------

df_capacidade_220 = pd.DataFrame(
    resultados_220
)

df_capacidade_220 = df_capacidade_220.sort_values(
    by=["Capacidade (%)", "Modelo"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 5. Validar a comparação
# ------------------------------------------------------------

assert len(df_capacidade_220) == (
    len(modelos_220) * len(capacidades_220)
)

for capacidade, grupo in df_capacidade_220.groupby(
    "Capacidade (%)"
):

    # Todos os modelos devem emitir o mesmo número de alertas
    assert grupo["Alertas"].nunique() == 1

    # A quantidade de atrasos reais é fixa
    assert (
        grupo["TP"] + grupo["FN"] == total_atrasos_220
    ).all()

    # Todo alerta é verdadeiro ou falso positivo
    assert (
        grupo["TP"] + grupo["FP"] == grupo["Alertas"]
    ).all()

# ------------------------------------------------------------
# 6. Exibir a tabela comparativa
# ------------------------------------------------------------

print("=== COMPARAÇÃO POR CAPACIDADE DE ALERTAS ===")

print(f"Coletas de validação: {total_registros_220:,}")
print(f"Atrasos reais: {total_atrasos_220:,}")

print(
    df_capacidade_220.to_string(
        index=False,
        float_format=lambda valor: f"{valor:.4f}"
    )
)

# ------------------------------------------------------------
# 7. Comparação direta de atrasos identificados
# ------------------------------------------------------------

tabela_tp_220 = df_capacidade_220.pivot(
    index="Capacidade (%)",
    columns="Modelo",
    values="TP"
)

print("\n=== ATRASOS IDENTIFICADOS POR CAPACIDADE ===")

print(
    tabela_tp_220.to_string()
)

# ------------------------------------------------------------
# 8. Resumo da execução
# ------------------------------------------------------------

print("\n=== VALIDAÇÕES ===")
print("Três modelos avaliados: OK")
print("Sete capacidades avaliadas: OK")
print("Mesma quantidade de alertas por capacidade: OK")
print("Métricas consistentes: OK")
print("Nenhum modelo foi treinado novamente.")
print("Conjunto de teste não utilizado.")


=== COMPARAÇÃO POR CAPACIDADE DE ALERTAS ===
Coletas de validação: 213,629
Atrasos reais: 5,448
                        Modelo  Capacidade (%)  Alertas   TP    FP   FN  Precision  Recall  F1-score  Average Precision
                 Random Forest               1     2137  750  1387 4698     0.3510  0.1377    0.1978             0.1795
Regressão Logística balanceada               1     2137  570  1567 4878     0.2667  0.1046    0.1503             0.1125
  Regressão Logística original               1     2137  530  1607 4918     0.2480  0.0973    0.1397             0.1188
                 Random Forest               2     4273 1109  3164 4339     0.2595  0.2036    0.2282             0.1795
Regressão Logística balanceada               2     4273  918  3355 4530     0.2148  0.1685    0.1889             0.1125
  Regressão Logística original               2     4273  938  3335 4510     0.2195  0.1722    0.1930             0.1188
                 Random Forest               5    10682 1821  88

## 2.20 — Explicação detalhada do código e dos conceitos utilizados

### 1. Organização dos modelos com dicionários Python

```python
modelos_220 = {
    "Regressão Logística original": pipeline_baseline,
    "Regressão Logística balanceada": pipeline_balanceado,
    "Random Forest": pipeline_rf_219
}
```

Um **dicionário (`dict`)** é uma estrutura de dados que armazena pares de chave e valor.

Neste exemplo:

- A chave é uma string que identifica o modelo.
- O valor é o objeto `Pipeline` correspondente, já treinado.

Diferentemente de uma lista, na qual normalmente acessamos elementos por posição, um dicionário permite identificar elementos por suas chaves.

**Por que utilizar um dicionário?**

Porque queremos executar o mesmo procedimento de avaliação para vários modelos, preservando seus nomes.

Isso evita escrever três blocos de código praticamente idênticos.

### 2. Estrutura de repetição `for` e método `.items()`

```python
for nome_modelo, pipeline in modelos_220.items():
```

A estrutura `for` permite percorrer os elementos de uma coleção.

O método `.items()` retorna os pares de chave e valor de um dicionário.

Em cada repetição:

- `nome_modelo` recebe a chave.
- `pipeline` recebe o objeto associado à chave.

Esse mecanismo é chamado de **desempacotamento (*unpacking*)**.

O Python distribui os elementos de cada par entre as duas variáveis.

Assim, conseguimos avaliar todos os modelos com um único bloco de código.

### 3. Método `predict_proba()` e seleção de colunas

```python
probabilidades = pipeline.predict_proba(
    X_validacao
)[:, 1]
```

O método `predict_proba()` calcula probabilidades estimadas para as classes de um problema de classificação.

Em nosso problema binário, as classes são:

- `0`: coleta realizada dentro da janela.
- `1`: coleta realizada após o limite da janela.

O resultado normalmente é uma matriz NumPy com duas colunas, uma para cada classe.

A expressão `[:, 1]` utiliza uma técnica chamada **fatiamento (*slicing*)**.

O símbolo `:` indica que queremos todas as linhas.

O índice `1` seleciona a segunda coluna, correspondente à classe positiva em nossos modelos.

**Por que utilizar probabilidades em vez de `predict()`?**

Porque precisamos ordenar as coletas por risco estimado.

O método `predict()` retornaria apenas as classificações finais, enquanto `predict_proba()` preserva informações necessárias para estabelecer prioridades.

### 4. Conversão de Pandas para NumPy

```python
y_real_220 = y_validacao.to_numpy()
```

O método `.to_numpy()` converte uma `Series` do Pandas em um array NumPy.

Uma `Series` possui índices próprios, que podem representar os índices originais do DataFrame.

Um array NumPy utiliza posições inteiras para acessar seus elementos.

Essa conversão é importante porque nossa ordenação posterior produz posições dos registros, não os índices originais do Pandas.

Assim, garantimos que a seleção das classificações reais corresponda às posições das probabilidades estimadas.

**Atenção:** a conversão não altera a ordem dos registros. A correspondência depende de mantermos a mesma ordem entre `X_validacao` e `y_validacao`.

### 5. Criação de índices com `np.arange()`

```python
np.arange(total_registros_220)
```

A função `np.arange()` cria uma sequência de números inteiros.

Por exemplo:

```python
np.arange(5)
```

Produz:

```text
[0, 1, 2, 3, 4]
```

No experimento, criamos uma sequência contendo as posições de todos os registros de validação.

Essas posições serão utilizadas como critério secundário de ordenação quando existirem probabilidades empatadas.

### 6. Ordenação por múltiplas chaves com `np.lexsort()`

```python
ordem_risco = np.lexsort((
    np.arange(total_registros_220),
    -probabilidades
))
```

A função `np.lexsort()` realiza uma ordenação indireta utilizando múltiplas chaves.

**Ordenação indireta** significa que a função retorna os índices dos elementos na ordem desejada, em vez de reorganizar diretamente os valores.

Um detalhe importante é que `np.lexsort()` utiliza a última chave como critério principal.

Em nosso código:

1. `-probabilidades` é a chave principal.
2. `np.arange(total_registros_220)` é a chave secundária.

**Por que utilizar o sinal negativo?**

Porque a ordenação padrão é crescente.

Ao transformar as probabilidades em valores negativos, fazemos com que as maiores probabilidades apareçam primeiro.

Exemplo:

| Probabilidade | Valor negativo |
|---|---|
| 0,80 | -0,80 |
| 0,35 | -0,35 |
| 0,60 | -0,60 |

Na ordenação crescente dos valores negativos, a ordem será 0,80, 0,60 e 0,35.

Portanto, obtemos as coletas da maior para a menor probabilidade estimada de atraso.

O índice original funciona como desempate determinístico.

**Por que isso importa?**

Porque precisamos selecionar exatamente a mesma quantidade de alertas para todos os modelos, inclusive quando várias coletas apresentam probabilidades iguais.

### 7. Cálculo da capacidade e arredondamento com `np.ceil()`

```python
quantidade_alertas = int(
    np.ceil(
        total_registros_220 * capacidade / 100
    )
)
```

A função `np.ceil()` realiza o arredondamento para o menor inteiro maior ou igual ao valor recebido.

Exemplos:

```python
np.ceil(4.2)   # 5.0
np.ceil(7.0)   # 7.0
```

O resultado é convertido para inteiro com `int()`.

Em nosso experimento, calculamos quantos registros correspondem ao percentual de capacidade.

Para 5%:

```text
213.629 × 5 / 100 = 10.681,45
```

Após o arredondamento:

```text
10.682 alertas
```

Isso estabelece uma quantidade fixa de registros para a comparação.

O arredondamento para cima foi uma convenção metodológica adotada no experimento; uma operação real poderia utilizar um limite inteiro definido diretamente pela capacidade da equipe.

### 8. Seleção de registros com slicing

```python
indices_priorizados = ordem_risco[
    :quantidade_alertas
]
```

Essa expressão seleciona os primeiros elementos do array `ordem_risco`.

Como o array já está ordenado por risco decrescente, esses elementos representam as coletas com maiores probabilidades estimadas de atraso.

Por exemplo:

```python
ordem = np.array([4, 1, 3, 0, 2])

ordem[:3]
```

Resultado:

```text
[4, 1, 3]
```

Selecionamos as três primeiras posições da ordenação.

Em nosso caso, selecionamos a quantidade correspondente à capacidade avaliada.

### 9. Cálculo dos verdadeiros positivos com indexação NumPy

```python
tp = int(
    y_real_220[indices_priorizados].sum()
)
```

A expressão `y_real_220[indices_priorizados]` utiliza **indexação avançada (*fancy indexing*)**.

Em vez de acessar uma única posição, fornecemos um array de posições.

O NumPy retorna os valores correspondentes a essas posições.

Como nosso alvo é binário, a soma dos valores selecionados corresponde à quantidade de registros positivos.

Portanto:

- Cada valor `1` representa uma coleta que realmente atrasou.
- Cada valor `0` representa uma coleta que não atrasou.

O método `.sum()` conta os atrasos reais presentes entre as coletas priorizadas.

Esse resultado corresponde aos verdadeiros positivos (`TP`).

### 10. Cálculo de Precision, Recall e F1-score

```python
precision = tp / quantidade_alertas
recall = tp / total_atrasos_220
```

**Precision** responde:

> Entre as coletas sinalizadas, quantas realmente atrasaram?

**Recall** responde:

> Entre todas as coletas que atrasaram, quantas foram sinalizadas?

As duas métricas são importantes, mas respondem a perguntas diferentes.

A Precision está relacionada à qualidade dos alertas emitidos.

O Recall está relacionado à cobertura dos atrasos reais.

O F1-score combina as duas métricas por meio de uma média harmônica:

\[
F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}
\]

O código também verifica possíveis divisões por zero.

Essas verificações tornam a implementação mais robusta.

### 11. Average Precision

```python
average_precision = average_precision_score(
    y_real_220,
    probabilidades
)
```

A função `average_precision_score()` pertence ao módulo `sklearn.metrics`.

Ela resume o desempenho do modelo considerando diferentes níveis de Precision e Recall ao longo da ordenação por risco.

Diferentemente das métricas calculadas para uma capacidade específica, a Average Precision utiliza todas as probabilidades estimadas.

Por isso, seu valor permanece igual nas diferentes capacidades avaliadas para um mesmo modelo.

Ela permite avaliar o desempenho global de priorização, mas não substitui a análise operacional em capacidades específicas.

### 12. Organização dos resultados com Pandas

```python
df_capacidade_220 = pd.DataFrame(
    resultados_220
)
```

O construtor `pd.DataFrame()` transforma os resultados acumulados em uma estrutura tabular.

No código, `resultados_220` é uma lista de dicionários.

Cada dicionário representa uma combinação entre modelo e capacidade.

O Pandas utiliza as chaves desses dicionários como nomes das colunas.

Essa abordagem facilita a organização de resultados experimentais e sua posterior exportação ou visualização.

### 13. Agrupamento com `groupby()`

```python
for capacidade, grupo in df_capacidade_220.groupby(
    "Capacidade (%)"
):
```

O método `groupby()` separa os registros de um DataFrame em grupos definidos pelos valores de uma ou mais colunas.

Em nosso caso, cada grupo contém os resultados dos três modelos para uma mesma capacidade.

Utilizamos esse agrupamento para verificar se os modelos emitiram exatamente a mesma quantidade de alertas.

A expressão:

```python
grupo["Alertas"].nunique()
```

conta a quantidade de valores distintos na coluna `Alertas`.

Se o resultado for `1`, significa que todos os modelos daquele grupo utilizaram a mesma quantidade de alertas.

### 14. Reorganização dos resultados com `pivot()`

```python
tabela_tp_220 = df_capacidade_220.pivot(
    index="Capacidade (%)",
    columns="Modelo",
    values="TP"
)
```

O método `pivot()` reorganiza os dados de um DataFrame.

Os parâmetros indicam:

- `index`: qual coluna será utilizada nas linhas.
- `columns`: qual coluna fornecerá os nomes das novas colunas.
- `values`: quais valores serão apresentados nas células.

Nesse caso, cada linha representa uma capacidade, cada coluna representa um modelo e cada célula apresenta a quantidade de atrasos corretamente identificados.

**Diferença entre `groupby()` e `pivot()`:**

`groupby()` organiza registros em grupos para permitir cálculos e verificações.

`pivot()` reorganiza os dados para facilitar sua apresentação e comparação.

### 15. Validação com `assert`

As instruções `assert` verificam condições que devem ser verdadeiras durante a execução.

Por exemplo:

```python
assert grupo["Alertas"].nunique() == 1
```

Essa instrução verifica se todos os modelos receberam a mesma quantidade de alertas dentro de cada capacidade.

Caso a condição seja falsa, o Python interrompe a execução com um `AssertionError`.

Essas verificações são importantes porque resultados numericamente plausíveis não garantem que uma comparação tenha sido construída corretamente.

### 16. Fundamentação da metodologia

A avaliação por capacidade de alertas é uma forma de comparar a capacidade de ordenação dos modelos sob uma restrição operacional comum.

Ela evita comparar modelos que emitem quantidades muito diferentes de alertas utilizando o mesmo limiar numérico.

Entretanto, o experimento é retrospectivo: utilizamos todas as previsões do conjunto de validação para selecionar os registros de maior risco.

Em produção, as previsões poderão ser calculadas individualmente no momento do aceite, enquanto a política de priorização deverá considerar as condições operacionais disponíveis naquele momento.

Portanto, a avaliação por capacidade não substitui uma futura validação do comportamento do sistema em tempo real.

### Síntese didática

Nesta etapa, utilizamos conceitos importantes de Python, NumPy, Pandas e scikit-learn:

- Dicionários e desempacotamento.
- Estruturas de repetição.
- Arrays NumPy e indexação avançada.
- Ordenação indireta por múltiplas chaves.
- Arredondamento e seleção de registros.
- Cálculo de métricas de classificação.
- Construção, agrupamento e reorganização de DataFrames.
- Validações de consistência.

A combinação desses recursos permitiu construir uma comparação reproduzível entre diferentes algoritmos, preservando as mesmas condições de avaliação.

## 2.20 — Resultados, interpretação e conclusão

### 1. Objetivo da avaliação

O objetivo desta etapa foi comparar a capacidade de priorização de três modelos de classificação:

- Regressão Logística original.
- Regressão Logística balanceada.
- Random Forest.

A avaliação utilizou o mesmo conjunto de validação, com **213.629 coletas**, das quais **5.448 apresentaram atraso**.

Foram consideradas sete capacidades de alertas: 1%, 2%, 5%, 10%, 15%, 20% e 30%.

Em cada capacidade, os modelos selecionaram exatamente a mesma quantidade de coletas, ordenadas pela probabilidade estimada de atraso.

### 2. Resultados obtidos

A tabela apresenta a quantidade de atrasos reais identificados por cada modelo.

| Capacidade | Alertas | Logística original | Logística balanceada | Random Forest |
|---|---:|---:|---:|---:|
| 1% | 2.137 | 530 | 570 | **750** |
| 2% | 4.273 | 938 | 918 | **1.109** |
| 5% | 10.682 | 1.475 | 1.475 | **1.821** |
| 10% | 21.363 | 2.106 | 2.065 | **2.502** |
| 15% | 32.045 | 2.626 | 2.641 | **2.954** |
| 20% | 42.726 | 2.972 | 3.032 | **3.298** |
| 30% | 64.089 | 3.529 | 3.569 | **3.871** |

O Random Forest identificou mais atrasos reais em todas as capacidades avaliadas.

### 3. Análise da capacidade de 5%

Considerando uma capacidade de 5%, cada modelo priorizou 10.682 coletas.

A Regressão Logística original identificou 1.475 atrasos reais, enquanto o Random Forest identificou 1.821.

Isso representa:

- **346 atrasos adicionais identificados** pelo Random Forest.
- Aumento relativo de aproximadamente **23,5%** na quantidade de atrasos identificados.
- Precision de 17,05% para o Random Forest, contra 13,81% para a Regressão Logística original.
- Recall de 33,43% para o Random Forest, contra 27,07% para a Regressão Logística original.

A comparação demonstra que o Random Forest conseguiu priorizar mais atrasos reais utilizando a mesma quantidade de alertas.

Entretanto, mesmo nessa capacidade, aproximadamente 66,57% dos atrasos reais permaneceram fora da seleção do Random Forest.

Portanto, o resultado representa uma melhoria de priorização, não uma solução completa para a identificação de atrasos.

### 4. Comportamento das métricas

À medida que aumentamos a capacidade de alertas, observamos dois comportamentos principais.

**Aumento do Recall**

Ao selecionar mais coletas, os modelos conseguem identificar uma parcela maior dos atrasos reais.

No Random Forest, o Recall aumentou de 13,77%, na capacidade de 1%, para 71,05%, na capacidade de 30%.

**Redução da Precision**

Com a ampliação da seleção, também são incluídas mais coletas que não apresentam atraso.

Consequentemente, a Precision do Random Forest diminuiu de 35,10%, na capacidade de 1%, para 6,04%, na capacidade de 30%.

Esse comportamento evidencia o compromisso entre cobertura dos atrasos e quantidade de falsos alertas.

A escolha de uma capacidade operacional deverá considerar esse equilíbrio.

### 5. Comparação da Average Precision

Os modelos apresentaram os seguintes resultados:

| Modelo | Average Precision |
|---|---:|
| Regressão Logística original | 0,1188 |
| Regressão Logística balanceada | 0,1125 |
| **Random Forest** | **0,1795** |

O Random Forest apresentou a maior Average Precision.

Em relação à Regressão Logística original, a melhoria relativa foi de aproximadamente 51%.

Esse resultado é consistente com a vantagem observada nas sete capacidades avaliadas.

Entretanto, a Average Precision não representa diretamente a proporção de atrasos identificados nem o percentual de alertas corretos em uma capacidade específica.

Ela resume o desempenho da ordenação das previsões ao longo de diferentes níveis de Precision e Recall.

### 6. Avaliação da Regressão Logística balanceada

A Regressão Logística balanceada não apresentou vantagem consistente sobre a versão original.

Em algumas capacidades, identificou mais atrasos; em outras, identificou menos.

Sua Average Precision também foi inferior à da Regressão Logística original.

Portanto, os resultados disponíveis não justificam considerar a ponderação de classes como uma melhoria automática para esse problema.

Isso não significa que técnicas de balanceamento sejam inadequadas em geral, mas que sua aplicação precisa ser avaliada empiricamente.

### 7. Implicações para a Torre de Controle

Os resultados indicam que o Random Forest é, entre os três modelos avaliados, o candidato mais promissor para priorizar coletas com risco de atraso.

Uma aplicação operacional poderia utilizar suas probabilidades estimadas para apoiar a definição de prioridades de acompanhamento.

Entretanto, a análise realizada foi retrospectiva.

A seleção dos registros de maior risco utilizou o conjunto completo de validação, não uma sequência de eventos recebidos em tempo real.

Por isso, ainda não podemos afirmar que os ganhos observados serão reproduzidos integralmente em uma operação com chegada contínua de solicitações.

Também não podemos afirmar que os atrasos identificados seriam evitados por uma intervenção operacional.

Essa última conclusão dependeria de informações sobre ações realizadas e seus resultados, que não foram avaliadas neste experimento.

### 8. Limitações da avaliação

Os resultados devem ser interpretados considerando as seguintes limitações:

- Os modelos foram avaliados em um único conjunto de validação.
- O Random Forest utilizou uma configuração inicial de hiperparâmetros.
- Não foram realizadas análises de calibração das probabilidades.
- Não foi investigada a estabilidade dos resultados entre diferentes subconjuntos operacionais.
- Não foi avaliado o desempenho em uma simulação de eventos em tempo real.
- Não foram realizadas estimativas de incerteza estatística para as diferenças observadas.
- Não foram investigadas as variáveis que mais contribuíram para as previsões.

Além disso, o conjunto de teste permanece reservado para a avaliação final, após a definição da estratégia de modelagem.

### 9. Decisão técnica

Com base nas evidências obtidas, o Random Forest passa a ser o **principal candidato para os próximos experimentos**, mas ainda não será considerado o modelo definitivo.

A decisão é sustentada por dois resultados:

1. Maior Average Precision no conjunto de validação.
2. Maior quantidade de atrasos identificados em todas as capacidades de alertas avaliadas.

A próxima fase deverá aprofundar a avaliação do Random Forest, investigando sua generalização, suas variáveis preditoras e possíveis melhorias de configuração.

Não há justificativa, neste momento, para selecionar um limiar operacional definitivo ou utilizar o conjunto de teste.

### 10. Conclusão

A comparação por capacidade de alertas demonstrou que o Random Forest apresentou desempenho superior ao das duas versões de Regressão Logística nas sete capacidades avaliadas.

Na capacidade de 5%, identificou 346 atrasos adicionais, mantendo a mesma quantidade de alertas da Regressão Logística original.

Os resultados reforçam a hipótese de que relações não lineares e interações entre variáveis podem contribuir para a previsão de atrasos em coletas logísticas.

Entretanto, a contribuição específica dessas relações ainda precisa ser investigada.

A etapa forneceu evidências para priorizar o Random Forest nos próximos experimentos, preservando o rigor metodológico e evitando uma seleção prematura do modelo final.

**Status da etapa 2.20: concluída.**

**Próxima etapa: 2.21 — Diagnóstico de generalização e sobreajuste do Random Forest.**

## 2.21 — Diagnóstico de generalização e sobreajuste do Random Forest

### 1. Objetivo

Investigar se o Random Forest treinado na etapa 2.19 apresenta capacidade de generalização ou sinais de sobreajuste (*overfitting*).

A análise será realizada comparando o desempenho do modelo nos conjuntos de treinamento e validação.

**Pergunta central:**

O Random Forest aprendeu padrões que também funcionam em registros não utilizados no treinamento ou seu desempenho depende excessivamente dos dados que já conhece?

### 2. Contexto

Na etapa 2.20, o Random Forest apresentou desempenho superior ao das duas versões de Regressão Logística nas sete capacidades de alertas avaliadas.

Sua Average Precision foi de 0,1795, contra 0,1188 da Regressão Logística original.

Entretanto, um desempenho superior na validação não elimina a possibilidade de sobreajuste.

Antes de ampliar os experimentos, precisamos investigar a diferença entre o comportamento do modelo nos dados utilizados para seu treinamento e nos dados reservados para validação.

### 3. Conceitos fundamentais

#### Generalização

Generalização é a capacidade de um modelo de Machine Learning produzir previsões úteis para observações que não participaram de seu treinamento.

Um modelo com boa generalização aprende relações que se mantêm relevantes fora dos exemplos utilizados para ajustar seus parâmetros.

#### Overfitting — Sobreajuste

Ocorre quando um modelo aprende características excessivamente específicas dos dados de treinamento, incluindo padrões que não se reproduzem adequadamente em novos registros.

Um possível sinal de sobreajuste é apresentar desempenho muito superior no treinamento em comparação com a validação.

Entretanto, essa diferença precisa ser interpretada considerando o algoritmo, a métrica e a dificuldade do problema.

#### Underfitting — Subajuste

Ocorre quando o modelo possui capacidade insuficiente para representar adequadamente as relações presentes nos dados.

Nesse caso, o desempenho pode ser limitado tanto no treinamento quanto na validação.

No Random Forest, restrições como profundidade máxima das árvores e quantidade mínima de registros por folha influenciam a complexidade do modelo.

### 4. Metodologia

Utilizaremos o Random Forest já treinado na etapa 2.19.

Não realizaremos novo treinamento.

Serão calculadas probabilidades para:

- Conjunto de treinamento: 996.936 registros.
- Conjunto de validação: 213.629 registros.

Compararemos principalmente a Average Precision, pois nosso problema apresenta forte desbalanceamento entre as classes.

Também analisaremos o desempenho sob capacidades equivalentes de alertas, especialmente 1%, 2%, 5% e 10%.

As capacidades serão calculadas separadamente para cada conjunto, respeitando seus respectivos tamanhos.

### 5. Cuidados metodológicos

As métricas de treinamento representam desempenho sobre registros que o algoritmo utilizou para aprender.

Portanto, não constituem uma estimativa independente de desempenho futuro.

A validação fornece uma avaliação mais adequada da capacidade de generalização, desde que o particionamento represente o cenário operacional pretendido.

Uma diferença entre treinamento e validação não demonstra, isoladamente, a existência de sobreajuste.

Além disso, a análise não substitui uma avaliação posterior sobre possíveis mudanças na distribuição dos dados, dependências entre registros ou diferenças entre períodos operacionais.

O conjunto de teste permanecerá reservado.

### 6. Resultado esperado

Produzir uma comparação entre treinamento e validação para identificar:

- A magnitude da diferença de Average Precision.
- A diferença de Precision e Recall nas capacidades selecionadas.
- Possíveis indícios de sobreajuste ou subajuste.
- A necessidade, ou não, de investigar ajustes na complexidade do Random Forest.

Não alteraremos hiperparâmetros nesta etapa.

### 7. Continuidade do projeto

Após esse diagnóstico, manteremos a sequência planejada:

- **2.22:** primeiro experimento com XGBoost.
- **2.23:** implementação do loop reutilizável de experimentos.
- **2.24:** comparação consolidada dos algoritmos.

Posteriormente, aprofundaremos a otimização, a interpretação das variáveis e a integração com MLflow.

A etapa atual é exclusivamente diagnóstica.

In [20]:

# ============================================================
# ETAPA 2.21 — DIAGNÓSTICO DE GENERALIZAÇÃO
# RANDOM FOREST: TREINAMENTO X VALIDAÇÃO
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import average_precision_score

# ------------------------------------------------------------
# 1. Recuperar o Random Forest já treinado
# ------------------------------------------------------------

modelo_221 = pipeline_rf_219

# Não executar fit() nesta etapa.

# ------------------------------------------------------------
# 2. Organizar os conjuntos de avaliação
# ------------------------------------------------------------

conjuntos_221 = {
    "Treinamento": (X_treino, y_treino),
    "Validação": (X_validacao, y_validacao)
}

capacidades_221 = [1, 2, 5, 10]

resultados_221 = []
resumo_conjuntos_221 = []

# ------------------------------------------------------------
# 3. Avaliar cada conjunto
# ------------------------------------------------------------

for nome_conjunto, (X, y) in conjuntos_221.items():

    # Converter o alvo para um array NumPy
    y_real = y.to_numpy()

    total_registros = len(y_real)
    total_atrasos = int(y_real.sum())

    # Calcular probabilidades com o modelo já treinado
    probabilidades = modelo_221.predict_proba(X)[:, 1]

    # Validar as probabilidades
    assert len(probabilidades) == total_registros
    assert np.isfinite(probabilidades).all()
    assert np.all(
        (probabilidades >= 0) & (probabilidades <= 1)
    )

    # Calcular a Average Precision
    ap = average_precision_score(
        y_real,
        probabilidades
    )

    # Registrar informações gerais
    resumo_conjuntos_221.append({
        "Conjunto": nome_conjunto,
        "Registros": total_registros,
        "Atrasos": total_atrasos,
        "Prevalência (%)": (
            100 * total_atrasos / total_registros
        ),
        "Average Precision": ap
    })

    # Ordenar as coletas por risco decrescente
    ordem_risco = np.lexsort((
        np.arange(total_registros),
        -probabilidades
    ))

    # --------------------------------------------------------
    # 4. Avaliar as capacidades de alertas
    # --------------------------------------------------------

    for capacidade in capacidades_221:

        quantidade_alertas = int(
            np.ceil(total_registros * capacidade / 100)
        )

        indices_priorizados = ordem_risco[
            :quantidade_alertas
        ]

        tp = int(
            y_real[indices_priorizados].sum()
        )

        fp = quantidade_alertas - tp
        fn = total_atrasos - tp

        precision = (
            tp / quantidade_alertas
            if quantidade_alertas > 0
            else 0.0
        )

        recall = (
            tp / total_atrasos
            if total_atrasos > 0
            else 0.0
        )

        resultados_221.append({
            "Conjunto": nome_conjunto,
            "Capacidade (%)": capacidade,
            "Alertas": quantidade_alertas,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "Precision": precision,
            "Recall": recall
        })

# ------------------------------------------------------------
# 5. Construir os DataFrames
# ------------------------------------------------------------

df_resumo_221 = pd.DataFrame(
    resumo_conjuntos_221
)

df_capacidades_221 = pd.DataFrame(
    resultados_221
)

# ------------------------------------------------------------
# 6. Calcular a diferença de Average Precision
# ------------------------------------------------------------

ap_treinamento_221 = df_resumo_221.loc[
    df_resumo_221["Conjunto"] == "Treinamento",
    "Average Precision"
].iloc[0]

ap_validacao_221 = df_resumo_221.loc[
    df_resumo_221["Conjunto"] == "Validação",
    "Average Precision"
].iloc[0]

diferenca_ap_221 = (
    ap_treinamento_221 - ap_validacao_221
)

# ------------------------------------------------------------
# 7. Comparar Precision e Recall
# ------------------------------------------------------------

df_comparacao_221 = df_capacidades_221.pivot(
    index="Capacidade (%)",
    columns="Conjunto",
    values=["Precision", "Recall"]
)

# Calcular diferenças: treinamento menos validação
df_diferencas_221 = pd.DataFrame({
    "Capacidade (%)": capacidades_221
})

for metrica in ["Precision", "Recall"]:

    df_diferencas_221[
        f"Diferença {metrica}"
    ] = (
        df_comparacao_221[
            (metrica, "Treinamento")
        ].to_numpy()
        -
        df_comparacao_221[
            (metrica, "Validação")
        ].to_numpy()
    )

# ------------------------------------------------------------
# 8. Validações de consistência
# ------------------------------------------------------------

assert len(df_resumo_221) == 2

assert len(df_capacidades_221) == (
    2 * len(capacidades_221)
)

assert (
    df_capacidades_221["TP"]
    + df_capacidades_221["FP"]
    == df_capacidades_221["Alertas"]
).all()

for nome_conjunto, grupo in df_capacidades_221.groupby(
    "Conjunto"
):

    atrasos_conjunto = int(
        df_resumo_221.loc[
            df_resumo_221["Conjunto"] == nome_conjunto,
            "Atrasos"
        ].iloc[0]
    )

    assert (
        grupo["TP"] + grupo["FN"] == atrasos_conjunto
    ).all()

# ------------------------------------------------------------
# 9. Exibir os resultados
# ------------------------------------------------------------

print("=== RESUMO DOS CONJUNTOS ===")

print(
    df_resumo_221.to_string(
        index=False,
        float_format=lambda valor: f"{valor:.6f}"
    )
)

print("\n=== DIFERENÇA DE AVERAGE PRECISION ===")

print(
    "Treinamento - Validação:",
    f"{diferenca_ap_221:.6f}"
)

print("\n=== DESEMPENHO POR CAPACIDADE ===")

print(
    df_capacidades_221.to_string(
        index=False,
        float_format=lambda valor: f"{valor:.6f}"
    )
)

print("\n=== DIFERENÇAS: TREINAMENTO - VALIDAÇÃO ===")

print(
    df_diferencas_221.to_string(
        index=False,
        float_format=lambda valor: f"{valor:.6f}"
    )
)

print("\n=== VALIDAÇÕES ===")
print("Dois conjuntos avaliados: OK")
print("Quatro capacidades avaliadas: OK")
print("Métricas consistentes: OK")
print("Modelo não treinado novamente.")
print("Conjunto de teste não utilizado.")


=== RESUMO DOS CONJUNTOS ===
   Conjunto  Registros  Atrasos  Prevalência (%)  Average Precision
Treinamento     996936    25423         2.550114           0.190431
  Validação     213629     5448         2.550216           0.179512

=== DIFERENÇA DE AVERAGE PRECISION ===
Treinamento - Validação: 0.010919

=== DESEMPENHO POR CAPACIDADE ===
   Conjunto  Capacidade (%)  Alertas    TP    FP    FN  Precision   Recall
Treinamento               1     9970  3594  6376 21829   0.360481 0.141368
Treinamento               2    19939  5296 14643 20127   0.265610 0.208315
Treinamento               5    49847  8752 41095 16671   0.175577 0.344255
Treinamento              10    99694 12032 87662 13391   0.120689 0.473272
  Validação               1     2137   750  1387  4698   0.350959 0.137665
  Validação               2     4273  1109  3164  4339   0.259537 0.203561
  Validação               5    10682  1821  8861  3627   0.170474 0.334251
  Validação              10    21363  2502 18861  2946   0

## 2.21 — Explicação detalhada do código Python

### 1. Reutilização de um modelo treinado

`modelo_221 = pipeline_rf_219`

Essa instrução realiza uma atribuição de variável em Python.

A variável `modelo_221` passa a referenciar o mesmo objeto `Pipeline` armazenado em `pipeline_rf_219`.

Isso significa que não estamos criando outro Random Forest nem copiando seus parâmetros. Estamos reutilizando o pipeline já ajustado na etapa 2.19.

**Por que não executar `fit()` novamente?**

O método `fit()` realiza o treinamento ou ajuste de um estimador. Como nosso objetivo é diagnosticar o comportamento do modelo existente, um novo treinamento poderia alterar as condições do experimento.

Nesta etapa, utilizamos exclusivamente `predict_proba()` para gerar previsões.

### 2. Organização dos conjuntos com tuplas e dicionários

```python
conjuntos_221 = {
    "Treinamento": (X_treino, y_treino),
    "Validação": (X_validacao, y_validacao)
}
```

Aqui combinamos duas estruturas fundamentais de Python.

**Dicionário (`dict`):** associa uma chave a um valor. As chaves são os nomes dos conjuntos.

**Tupla (`tuple`):** agrupa elementos em uma sequência ordenada, normalmente representada por parênteses.

Cada tupla contém dois objetos:

- `X`: matriz de variáveis preditoras.
- `y`: vetor da variável-alvo.

A instrução:

`for nome_conjunto, (X, y) in conjuntos_221.items():`

utiliza **desempacotamento aninhado**.

Primeiro, o método `.items()` fornece a chave e o valor do dicionário. Em seguida, o Python desempacota a tupla em `X` e `y`.

Isso permite aplicar exatamente o mesmo procedimento de avaliação aos dois conjuntos, reduzindo duplicações de código.

### 3. Conversão do alvo para NumPy

`y_real = y.to_numpy()`

O método `.to_numpy()` transforma uma `Series` do Pandas em um array NumPy.

Essa conversão é útil porque posteriormente utilizaremos índices posicionais para selecionar registros classificados como prioritários.

Por exemplo, se `indices_priorizados` contém `[2, 5, 8]`, a expressão `y_real[indices_priorizados]` seleciona os valores nas posições 2, 5 e 8.

**Importante:** essa indexação é posicional. Por isso, a ordem de `X` e `y` precisa permanecer correspondente.

### 4. Contagem de registros e atrasos

`total_registros = len(y_real)`

A função nativa `len()` retorna a quantidade de elementos de uma coleção.

`total_atrasos = int(y_real.sum())`

O método `.sum()` soma os valores do array.

Como nossa variável-alvo é binária, com valores `0` e `1`, a soma corresponde à quantidade de coletas que realmente ultrapassaram o limite da janela.

A função `int()` converte o resultado para um inteiro Python.

### 5. Probabilidades estimadas e validações

`probabilidades = modelo_221.predict_proba(X)[:, 1]`

O método `predict_proba()` produz as probabilidades estimadas das classes.

O fatiamento `[:, 1]` seleciona todas as linhas da coluna correspondente à classe positiva (`1`).

Em seguida, aplicamos três verificações:

- `len(probabilidades) == total_registros`: confirma que cada registro recebeu uma probabilidade.
- `np.isfinite(probabilidades).all()`: verifica se todos os valores são finitos, sem `NaN` ou infinito.
- `np.all((probabilidades >= 0) & (probabilidades <= 1))`: verifica se todos os valores estão entre zero e um.

O operador `&` representa uma operação lógica elemento a elemento sobre arrays booleanos.

O método `.all()` retorna verdadeiro somente quando todas as condições verificadas são verdadeiras.

Essas verificações protegem a avaliação contra inconsistências numéricas ou estruturais.

### 6. Average Precision

```python
ap = average_precision_score(
    y_real,
    probabilidades
)
```

A função `average_precision_score()` pertence ao `sklearn.metrics`.

Ela resume a relação entre Precision e Recall considerando os diferentes níveis de decisão derivados das probabilidades estimadas.

É especialmente útil quando a classe positiva é pouco frequente, como ocorre em nosso conjunto de coletas.

Neste experimento, calculamos a Average Precision separadamente para treinamento e validação.

A diferença é calculada por:

`diferenca_ap_221 = ap_treinamento_221 - ap_validacao_221`

Uma diferença positiva significa que a métrica foi maior no treinamento.

Entretanto, não existe um valor universal que, isoladamente, determine a presença de overfitting. Precisamos interpretar a diferença em conjunto com outras evidências.

### 7. Lista de dicionários e método `append()`

```python
resumo_conjuntos_221.append({
    "Conjunto": nome_conjunto,
    "Registros": total_registros,
    "Atrasos": total_atrasos,
    "Prevalência (%)": 100 * total_atrasos / total_registros,
    "Average Precision": ap
})
```

O método `.append()` adiciona um elemento ao final de uma lista.

Neste caso, adicionamos um dicionário contendo os indicadores de cada conjunto.

A prevalência é calculada dividindo a quantidade de atrasos pelo total de registros e multiplicando por 100.

A prevalência é importante porque métricas como Precision e Average Precision dependem, em parte, da frequência da classe positiva.

Comparações entre conjuntos com prevalências muito diferentes exigiriam cuidados adicionais.

### 8. Ordenação por risco com `np.lexsort()`

```python
ordem_risco = np.lexsort((
    np.arange(total_registros),
    -probabilidades
))
```

Reutilizamos a mesma estratégia da etapa 2.20.

A função `np.lexsort()` devolve índices ordenados segundo múltiplas chaves.

A última chave, `-probabilidades`, é a principal. O sinal negativo permite ordenar as probabilidades da maior para a menor.

A primeira chave, `np.arange(total_registros)`, define um desempate baseado na posição original.

Essa estratégia garante uma seleção reproduzível quando existem probabilidades iguais.

**Por que repetir a mesma metodologia?**

Porque queremos comparar treinamento e validação utilizando critérios equivalentes, evitando introduzir diferenças causadas pelo procedimento de seleção.

### 9. Capacidade percentual e seleção dos registros

```python
quantidade_alertas = int(
    np.ceil(total_registros * capacidade / 100)
)
```

A função `np.ceil()` arredonda para cima.

A quantidade de alertas é calculada separadamente para cada conjunto, pois treinamento e validação possuem quantidades diferentes de registros.

Depois:

`indices_priorizados = ordem_risco[:quantidade_alertas]`

Selecionamos as posições correspondentes às coletas de maior risco estimado.

Por exemplo, na capacidade de 5%, selecionamos aproximadamente 5% dos registros de cada conjunto.

**Não estamos comparando quantidades absolutas iguais de alertas**, mas sim proporções equivalentes.

### 10. Matriz de confusão parcial: TP, FP e FN

```python
tp = int(y_real[indices_priorizados].sum())

fp = quantidade_alertas - tp
fn = total_atrasos - tp
```

Essas operações calculam três componentes da matriz de confusão:

**TP — Verdadeiros positivos:** coletas priorizadas que realmente atrasaram.

**FP — Falsos positivos:** coletas priorizadas que não atrasaram.

**FN — Falsos negativos:** coletas que atrasaram, mas ficaram fora da seleção.

O código não calcula explicitamente os verdadeiros negativos (`TN`), pois as métricas analisadas nesta etapa podem ser obtidas sem eles.

### 11. Precision e Recall

A Precision é calculada por:

\[
Precision = \frac{TP}{TP + FP}
\]

Como `TP + FP` corresponde à quantidade de alertas, utilizamos:

`precision = tp / quantidade_alertas`

O Recall é calculado por:

\[
Recall = \frac{TP}{TP + FN}
\]

Como `TP + FN` corresponde ao total de atrasos reais, utilizamos:

`recall = tp / total_atrasos`

O código emprega expressões condicionais para evitar divisão por zero.

Exemplo:

`tp / total_atrasos if total_atrasos > 0 else 0.0`

Essa construção é uma **expressão condicional**, também chamada de operador ternário em Python.

Sua estrutura é:

`valor_se_verdadeiro if condição else valor_se_falso`

### 12. Construção dos DataFrames

```python
df_resumo_221 = pd.DataFrame(
    resumo_conjuntos_221
)

df_capacidades_221 = pd.DataFrame(
    resultados_221
)
```

O construtor `pd.DataFrame()` transforma listas de dicionários em tabelas.

Criamos duas estruturas:

- `df_resumo_221`: indicadores gerais dos conjuntos.
- `df_capacidades_221`: métricas para cada combinação de conjunto e capacidade.

Essa separação facilita tanto a análise geral quanto a comparação detalhada.

### 13. Filtragem com `.loc[]` e seleção com `.iloc[]`

```python
ap_treinamento_221 = df_resumo_221.loc[
    df_resumo_221["Conjunto"] == "Treinamento",
    "Average Precision"
].iloc[0]
```

O acessador `.loc[]` seleciona dados utilizando rótulos e condições.

A expressão `df_resumo_221["Conjunto"] == "Treinamento"` produz uma sequência de valores booleanos, indicando quais linhas atendem à condição.

A seleção retorna a coluna `Average Precision` das linhas correspondentes.

Em seguida, `.iloc[0]` acessa o primeiro elemento por posição.

**Diferença importante:**

- `.loc[]`: seleção baseada em rótulos e condições.
- `.iloc[]`: seleção baseada em posições inteiras.

Como existe apenas uma linha para cada conjunto, `.iloc[0]` recupera o valor desejado.

### 14. Reorganização dos dados com `pivot()`

```python
df_comparacao_221 = df_capacidades_221.pivot(
    index="Capacidade (%)",
    columns="Conjunto",
    values=["Precision", "Recall"]
)
```

O método `pivot()` reorganiza uma tabela.

Neste caso:

- As linhas representam as capacidades.
- As colunas representam os conjuntos.
- Os valores representam Precision e Recall.

Como selecionamos duas métricas, o Pandas cria colunas com múltiplos níveis, chamadas de **MultiIndex**.

Por isso, acessamos uma coluna utilizando uma tupla:

`df_comparacao_221[("Precision", "Treinamento")]`

A tupla identifica simultaneamente a métrica e o conjunto.

### 15. Cálculo das diferenças entre conjuntos

```python
for metrica in ["Precision", "Recall"]:
```

Esse loop percorre os nomes das métricas.

Em cada repetição, calculamos:

`métrica no treinamento - métrica na validação`

A expressão:

`f"Diferença {metrica}"`

é uma **f-string**, recurso de Python utilizado para construir strings com valores de variáveis.

Por exemplo, quando `metrica` vale `"Precision"`, o nome da coluna será `"Diferença Precision"`.

O método `.to_numpy()` converte as colunas em arrays para realizar a subtração elemento a elemento.

Assim, cada capacidade terá sua própria diferença de Precision e Recall.

### 16. Verificações com `assert` e `groupby()`

O código utiliza `assert` para verificar condições obrigatórias.

Também utiliza:

`df_capacidades_221.groupby("Conjunto")`

O método `groupby()` separa os resultados em grupos de treinamento e validação.

Para cada grupo, verificamos se:

`TP + FN == total de atrasos`

E, para todas as linhas, verificamos se:

`TP + FP == quantidade de alertas`

Essas são identidades matemáticas que precisam ser satisfeitas pela contagem dos resultados.

As verificações ajudam a detectar erros de implementação antes da interpretação das métricas.

### 17. Exibição formatada dos resultados

O método `.to_string()` converte um DataFrame em uma representação textual.

O argumento:

`float_format=lambda valor: f"{valor:.6f}"`

define a apresentação dos números com seis casas decimais.

A palavra-chave `lambda` cria uma função anônima, isto é, uma função definida sem utilizar `def`.

Nesse exemplo, a função recebe um número e retorna sua representação formatada.

A formatação não altera os valores armazenados no DataFrame; modifica apenas sua apresentação.

### 18. Fundamentação metodológica

O diagnóstico compara o desempenho do mesmo modelo em dois conjuntos com funções diferentes.

**Treinamento:** dados utilizados para ajustar o modelo.

**Validação:** dados separados do treinamento e utilizados para avaliar seu comportamento em observações não vistas durante o ajuste.

Uma diferença elevada pode sugerir sobreajuste, mas não comprova automaticamente sua existência.

Também devemos observar a prevalência da classe positiva, a dificuldade do problema e a estratégia de particionamento.

A comparação realizada nesta etapa permite investigar a general

## 2.21 — Resultados, interpretação e conclusão

### 1. Objetivo da análise

Avaliar a capacidade de generalização do Random Forest, comparando seu desempenho nos conjuntos de treinamento e validação.

O objetivo foi identificar possíveis sinais de sobreajuste (*overfitting*) antes de avançar para novos algoritmos.

O modelo utilizado foi o mesmo treinado na etapa 2.19, sem alteração de hiperparâmetros ou novo treinamento.

### 2. Características dos conjuntos avaliados

| Indicador | Treinamento | Validação |
|---|---:|---:|
| Registros | 996.936 | 213.629 |
| Atrasos reais | 25.423 | 5.448 |
| Prevalência de atrasos | 2,5501% | 2,5502% |
| Average Precision | 0,190431 | 0,179512 |

A prevalência de atrasos foi praticamente idêntica nos dois conjuntos.

Essa característica é importante porque reduz a influência de diferenças na frequência da classe positiva sobre a comparação das métricas.

Entretanto, a semelhança da prevalência não garante, por si só, que todas as demais características dos conjuntos sejam equivalentes.

### 3. Comparação da Average Precision

<grid columns={2} gap={3}>
<grid-item>
<box border radius="lg" padding={3} gap={1}>
<text color="secondary" size="xs">Treinamento</text>
<title size="xl" color="default" tabularNums>0,190431</title>
</box>
</grid-item>
<grid-item>
<box border radius="lg" padding={3} gap={1}>
<text color="secondary" size="xs">Validação</text>
<title size="xl" color="default" tabularNums>0,179512</title>
</box>
</grid-item>
</grid>

A diferença absoluta foi:

\[
0{,}190431 - 0{,}179512 = 0{,}010919
\]

Isso representa uma redução relativa de aproximadamente **5,73%** da Average Precision do treinamento para a validação.

A diferença é relativamente pequena, considerando os resultados observados.

Não identificamos, por essa métrica, evidências de sobreajuste acentuado.

Entretanto, essa conclusão é restrita ao experimento realizado e não representa uma garantia de generalização em produção.

### 4. Comparação por capacidade de alertas

| Capacidade | Precision treino | Precision validação | Recall treino | Recall validação |
|---|---:|---:|---:|---:|
| 1% | 36,05% | 35,10% | 14,14% | 13,77% |
| 2% | 26,56% | 25,95% | 20,83% | 20,36% |
| 5% | 17,56% | 17,05% | 34,43% | 33,43% |
| 10% | 12,07% | 11,71% | 47,33% | 45,93% |

Observamos que o desempenho no treinamento foi superior ao da validação em todas as capacidades analisadas.

Entretanto, as diferenças foram moderadas.

Na capacidade de 5%, por exemplo:

- Precision: diferença de aproximadamente 0,51 ponto percentual.
- Recall: diferença de aproximadamente 1,00 ponto percentual.

Isso indica que o comportamento do modelo na validação permaneceu relativamente próximo ao observado no treinamento.

### 5. Análise do sobreajuste

O Random Forest foi treinado com restrições de complexidade, incluindo:

- `max_depth=12`: limita a profundidade máxima das árvores.
- `min_samples_leaf=20`: exige uma quantidade mínima de registros nas folhas.
- `n_estimators=100`: utiliza um conjunto de 100 árvores.

Os resultados sugerem que a configuração atual não apresenta sinais evidentes de memorização excessiva dos dados de treinamento.

Essa interpretação é sustentada pela proximidade entre as métricas de treinamento e validação.

Entretanto, não podemos atribuir causalmente essa proximidade aos hiperparâmetros sem realizar experimentos comparativos com outras configurações.

Também não podemos descartar limitações de generalização relacionadas a dependências entre registros, mudanças operacionais ou diferenças entre períodos.

### 6. Análise de subajuste

A ausência de um grande desvio entre treinamento e validação não significa que o modelo tenha atingido seu potencial máximo.

A Average Precision de treinamento, aproximadamente 0,1904, ainda indica espaço para melhoria na capacidade de priorização.

Isso pode decorrer de diferentes fatores, incluindo:

- Limitações das variáveis preditoras disponíveis.
- Complexidade das relações entre as variáveis.
- Restrições dos hiperparâmetros utilizados.
- Incerteza inerente ao comportamento operacional das coletas.

Não temos evidências suficientes para classificar o modelo como subajustado.

Para investigar essa possibilidade, seriam necessários novos experimentos controlados.

### 7. Validação metodológica

Todas as verificações programadas foram concluídas com sucesso:

- Dois conjuntos avaliados.
- Quatro capacidades analisadas.
- Consistência das métricas confirmada.
- Nenhum novo treinamento realizado.
- Conjunto de teste preservado.

A comparação foi realizada com o mesmo modelo, evitando introduzir diferenças decorrentes de novos ajustes.

### 8. Decisão técnica

**Não foram identificados sinais de sobreajuste acentuado no Random Forest, considerando as métricas e o particionamento utilizados.**

Por isso, não há necessidade de interromper a sequência planejada para realizar uma investigação específica de overfitting neste momento.

Também não há evidências suficientes para afirmar que a configuração atual seja ótima.

O Random Forest permanece como referência para os próximos experimentos.

A próxima etapa será avaliar o XGBoost, mantendo as mesmas variáveis preditoras, os mesmos conjuntos de treinamento e validação e critérios comparáveis de avaliação.

### 9. Conclusão

O Random Forest apresentou comportamento relativamente consistente entre treinamento e validação.

A diferença de Average Precision foi de 0,010919, e as diferenças de Precision e Recall permaneceram moderadas nas quatro capacidades avaliadas.

Os resultados são compatíveis com uma capacidade de generalização razoável dentro do particionamento utilizado, sem evidências de sobreajuste acentuado.

A análise não comprova desempenho futuro em produção nem elimina a necessidade de avaliações adicionais.

**Status: etapa 2.21 concluída.**

**Próxima etapa: 2.22 — Implementação e avaliação inicial do XGBoost.**

## 2.22 — XGBoost: fundamentação teórica e planejamento do experimento

### 1. Objetivo

Implementar e avaliar um modelo de classificação utilizando **XGBoost (Extreme Gradient Boosting)** para prever o risco de uma coleta ocorrer após o limite de sua janela operacional.

O objetivo é investigar se o XGBoost consegue melhorar a identificação de coletas com risco de atraso em comparação com os modelos já avaliados:

- Regressão Logística original.
- Regressão Logística balanceada.
- Random Forest.

**Pergunta central:**

O XGBoost consegue identificar mais atrasos reais, mantendo a mesma capacidade de alertas utilizada na comparação dos modelos anteriores?

### 2. O que é XGBoost?

XGBoost é uma implementação otimizada de algoritmos de *Gradient Boosting*, frequentemente utilizados em problemas de classificação e regressão com dados tabulares.

Assim como o Random Forest, o XGBoost pode utilizar árvores de decisão como modelos básicos.

Entretanto, a maneira como essas árvores são construídas e combinadas é diferente.

O XGBoost incorpora mecanismos de regularização, tratamento de valores ausentes e otimizações computacionais.

Essas características tornam o algoritmo relevante para problemas nos quais existem relações não lineares e interações entre variáveis.

### 3. Conceito de Ensemble Learning

**Ensemble Learning** é uma abordagem de Machine Learning que combina múltiplos modelos para produzir uma previsão.

A ideia é que um conjunto de modelos possa representar relações que um único modelo teria dificuldade de aprender.

Dois métodos importantes de Ensemble Learning são:

**Bagging — Bootstrap Aggregating**

Constrói modelos que podem ser treinados de forma independente, geralmente utilizando diferentes amostras dos dados.

O Random Forest utiliza essa abordagem, combinando árvores de decisão para produzir uma previsão final.

**Boosting**

Constrói modelos sequencialmente, fazendo com que cada novo modelo contribua para melhorar a previsão do conjunto existente.

O XGBoost pertence a essa família.

### 4. Random Forest versus XGBoost

| Característica | Random Forest | XGBoost |
|---|---|---|
| Estratégia | Bagging | Gradient Boosting |
| Construção das árvores | Predominantemente independente | Sequencial |
| Combinação na classificação | Média das probabilidades das árvores | Soma das contribuições das árvores, transformada em probabilidade |
| Controle de complexidade | Profundidade, folhas, amostragem e outros | Profundidade, regularização, taxa de aprendizado e outros |
| Risco de sobreajuste | Existe | Existe |
| Ajuste de hiperparâmetros | Importante | Importante |

Nenhum dos dois algoritmos é universalmente superior.

O desempenho depende das características dos dados, da configuração do modelo e da metodologia de avaliação.

Por isso, realizaremos uma comparação experimental.

### 5. Como funciona o Gradient Boosting?

O Gradient Boosting constrói um modelo aditivo.

Em termos simplificados, cada nova árvore acrescenta uma contribuição à previsão produzida pelas árvores anteriores.

O treinamento utiliza informações relacionadas ao gradiente da função de perda para determinar como melhorar o modelo.

No XGBoost, o processo também pode utilizar informações de segunda ordem, associadas à curvatura da função de perda.

Uma representação simplificada é:

\[
F_m(x) = F_{m-1}(x) + \eta f_m(x)
\]

Onde:

- \(F_m(x)\): previsão acumulada após a etapa \(m\).
- \(F_{m-1}(x)\): previsão acumulada anterior.
- \(f_m(x)\): contribuição da nova árvore.
- \(\eta\): taxa de aprendizado (*learning rate*).

Na classificação binária, essas contribuições normalmente são acumuladas na escala de margem ou log-odds e posteriormente transformadas em probabilidades.

**Interpretação:** o modelo é construído progressivamente, adicionando árvores que procuram reduzir a função de perda.

### 6. Principais hiperparâmetros do XGBoost

**`n_estimators`**

Define a quantidade máxima de etapas de boosting, normalmente associadas à quantidade de árvores no classificador binário convencional.

**`max_depth`**

Limita a profundidade das árvores e influencia a complexidade das relações aprendidas.

**`learning_rate`**

Controla o tamanho da contribuição de cada nova árvore.

Taxas menores podem exigir mais árvores para alcançar um desempenho semelhante.

**`subsample`**

Define a fração de registros utilizada na construção de cada árvore.

Pode introduzir aleatoriedade e contribuir para controlar o sobreajuste.

**`colsample_bytree`**

Define a fração de variáveis considerada na construção de cada árvore.

**`reg_lambda`**

Controla a regularização L2 dos pesos das folhas.

**`scale_pos_weight`**

Permite atribuir maior peso à classe positiva.

É um recurso relevante para investigar problemas desbalanceados, mas não deve ser considerado uma melhoria automática.

### 7. Particularidades do nosso problema

O conjunto de treinamento contém 996.936 registros, dos quais 25.423 representam atrasos.

A prevalência da classe positiva é aproximadamente 2,55%.

Isso significa que estamos trabalhando com um problema de **classificação binária fortemente desbalanceada**.

Nesse cenário, uma Accuracy elevada pode ser obtida mesmo quando o modelo identifica poucos atrasos.

Por isso, continuaremos priorizando:

- Average Precision.
- Precision.
- Recall.
- Quantidade de atrasos identificados sob capacidades fixas de alertas.

O limiar padrão de 0,50 poderá ser apresentado como diagnóstico complementar, mas não será o único critério de avaliação.

### 8. Planejamento do experimento

Utilizaremos os mesmos conjuntos de treinamento e validação dos experimentos anteriores.

Também manteremos as dez variáveis preditoras previamente selecionadas, respeitando a disponibilidade das informações no momento do aceite.

A implementação seguirá estas decisões:

1. Utilizar `XGBClassifier`, da biblioteca `xgboost`.
2. Reutilizar a função de experimentação criada anteriormente, desde que sua compatibilidade seja confirmada.
3. Começar com uma configuração inicial explícita e reproduzível.
4. Não aplicar balanceamento adicional de classes no primeiro experimento.
5. Avaliar o modelo na validação, sem utilizar o conjunto de teste.
6. Comparar os resultados com os modelos anteriores sob as mesmas capacidades de alertas.

Antes do treinamento, verificaremos se a biblioteca XGBoost está instalada e compatível com o ambiente Python.

### 9. Cuidados metodológicos

A implementação inicial do XGBoost não representa uma otimização de hiperparâmetros.

Seu objetivo é estabelecer uma referência experimental.

Não utilizaremos o conjunto de teste para escolher parâmetros, modelos ou capacidades operacionais.

Também não incluiremos variáveis que revelem o resultado da coleta ou informações indisponíveis no momento da previsão.

A comparação entre algoritmos deverá considerar tanto o desempenho preditivo quanto o custo computacional.

### 10. Resultado esperado

Ao final da etapa, teremos:

- Um pipeline XGBoost treinado.
- Probabilidades estimadas para o conjunto de validação.
- Métricas de classificação.
- Average Precision.
- Comparação por capacidade de alertas.
- Registro do tempo de treinamento.
- Uma conclusão técnica sobre a contribuição do XGBoost para nosso problema.

**Próxima atividade:** verificar a instalação e a versão da biblioteca XGBoost no ambiente virtual do projeto.

## 2.22.1 — Configuração e execução inicial do XGBoost

### 1. Objetivo

Treinar e avaliar o primeiro modelo XGBoost utilizando a função reutilizável `executar_experimento_classificacao()`, desenvolvida na etapa 2.17 e validada anteriormente com Regressão Logística e Random Forest.

O objetivo é investigar se o XGBoost consegue melhorar a identificação de coletas com risco de ocorrer após o limite da janela operacional.

### 2. Justificativa

O Random Forest apresentou, até o momento, o melhor desempenho na validação, com Average Precision de aproximadamente 0,179512.

O XGBoost utiliza Gradient Boosting, construindo árvores sequencialmente para melhorar as previsões do conjunto.

Essa abordagem pode representar relações complexas entre características geográficas, horários e disponibilidade de tempo para realizar a coleta.

Entretanto, não existe garantia de que o XGBoost apresentará desempenho superior ao Random Forest.

A comparação será baseada nos resultados experimentais.

### 3. Estratégia de implementação

Utilizaremos:

- O algoritmo `XGBClassifier`, da biblioteca XGBoost.
- A função reutilizável `executar_experimento_classificacao()`.
- As mesmas dez variáveis preditoras dos experimentos anteriores.
- Os mesmos conjuntos de treinamento e validação.
- O mesmo pré-processamento definido na função.
- O limiar inicial de classificação de 0,50.

O conjunto de teste permanecerá reservado.

### 4. Configuração inicial dos hiperparâmetros

| Hiperparâmetro | Valor | Justificativa |
|---|---|---|
| `n_estimators` | 200 | Controlar a quantidade de árvores |
| `max_depth` | 4 | Limitar a complexidade das árvores |
| `learning_rate` | 0.05 | Realizar atualizações graduais |
| `subsample` | 0.8 | Utilizar amostragem de registros |
| `colsample_bytree` | 0.8 | Utilizar amostragem de variáveis |
| `objective` | `binary:logistic` | Classificação binária probabilística |
| `eval_metric` | `aucpr` | Métrica interna relacionada à curva Precision-Recall |
| `tree_method` | `hist` | Treinamento eficiente em CPU |
| `n_jobs` | 4 | Limitar o paralelismo |
| `random_state` | 42 | Favorecer a reprodutibilidade |

Não utilizaremos balanceamento adicional de classes neste primeiro experimento.

Os hiperparâmetros representam uma configuração inicial controlada, não uma configuração otimizada.

### 5. Métricas de avaliação

A função reutilizável calculará:

- Accuracy.
- Precision.
- Recall.
- F1-score.
- Average Precision.
- Matriz de confusão.
- Quantidade e percentual de alertas.
- Tempo de treinamento.

A **Average Precision calculada pelo scikit-learn** continuará sendo nossa principal métrica de comparação entre algoritmos.

O limiar de 0,50 será utilizado como diagnóstico inicial, não como definição da capacidade operacional de alertas.

A avaliação por capacidades fixas será realizada posteriormente, utilizando as probabilidades produzidas pelo modelo.

### 6. Cuidados metodológicos

O pré-processamento será ajustado exclusivamente com os dados de treinamento.

O conjunto de validação será utilizado para avaliar o desempenho.

Não utilizaremos o conjunto de teste.

Não incluiremos novas variáveis nem modificaremos a definição da variável-alvo `coleta_apos_limite`.

O experimento será executado em CPU, mantendo o ambiente computacional atual.

### 7. Resultado esperado

Ao final da execução, teremos um pipeline XGBoost treinado, suas probabilidades de validação e um conjunto padronizado de métricas.

Esses resultados permitirão avaliar o desempenho inicial do algoritmo e preparar sua comparação com os modelos anteriores.

**Esta etapa não inclui otimização de hiperparâmetros nem seleção definitiva do melhor modelo.**

In [21]:

# ============================================================
# ETAPA 2.22.1 — PRIMEIRO EXPERIMENTO COM XGBOOST
# ============================================================

import numpy as np
import pandas as pd

from xgboost import XGBClassifier

# ------------------------------------------------------------
# 1. Configurar o algoritmo XGBoost
# ------------------------------------------------------------

estimador_xgb_222 = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    n_jobs=4,
    random_state=42
)

# ------------------------------------------------------------
# 2. Executar a função reutilizável
# ------------------------------------------------------------

pipeline_xgb_222, metricas_xgb_222, probabilidades_xgb_222 = (
    executar_experimento_classificacao(
        nome_modelo="XGBoost",
        estimador=estimador_xgb_222,
        X_treino=X_treino,
        y_treino=y_treino,
        X_validacao=X_validacao,
        y_validacao=y_validacao,
        variaveis_categoricas=categoricas_218,
        variaveis_numericas=numericas_218,
        limiar=0.50
    )
)

# ------------------------------------------------------------
# 3. Organizar as métricas
# ------------------------------------------------------------

df_metricas_xgb_222 = pd.DataFrame(
    [metricas_xgb_222]
)

# ------------------------------------------------------------
# 4. Recuperar o classificador treinado
# ------------------------------------------------------------

modelo_xgb_treinado_222 = (
    pipeline_xgb_222.named_steps["classificador"]
)

total_arvores_222 = (
    modelo_xgb_treinado_222.get_booster()
    .num_boosted_rounds()
)

# ------------------------------------------------------------
# 5. Validar os resultados
# ------------------------------------------------------------

assert len(probabilidades_xgb_222) == len(y_validacao)

assert np.isfinite(probabilidades_xgb_222).all()

assert np.all(
    (probabilidades_xgb_222 >= 0)
    & (probabilidades_xgb_222 <= 1)
)

assert (
    metricas_xgb_222["TN"]
    + metricas_xgb_222["FP"]
    + metricas_xgb_222["FN"]
    + metricas_xgb_222["TP"]
    == len(y_validacao)
)

assert total_arvores_222 == 200

# ------------------------------------------------------------
# 6. Apresentar os resultados
# ------------------------------------------------------------

print("=== ETAPA 2.22.1 — XGBOOST ===")

print(f"Rodadas de boosting: {total_arvores_222}")

print(
    "Tempo de treinamento (s):",
    round(metricas_xgb_222["Tempo treinamento (s)"], 2)
)

print("\n=== MÉTRICAS DE VALIDAÇÃO ===")

display(
    df_metricas_xgb_222.T.rename(
        columns={0: "Resultado"}
    )
)

print("\nConjunto de teste não utilizado.")


=== ETAPA 2.22.1 — XGBOOST ===
Rodadas de boosting: 200
Tempo de treinamento (s): 16.04

=== MÉTRICAS DE VALIDAÇÃO ===


,Resultado
Modelo,XGBoost
Limiar,0.5
Accuracy,0.974886
Precision,0.646643
Recall,0.03359
F1-score,0.063863
Average Precision,0.171407
TN,208081
FP,100
FN,5265



Conjunto de teste não utilizado.


## 2.22.1 — Explicação detalhada do código Python

### 1. Importação das bibliotecas

```python
import numpy as np
import pandas as pd

from xgboost import XGBClassifier
```

**NumPy (`np`)**

Biblioteca utilizada para operações numéricas com arrays.

Neste experimento, utilizamos `np.isfinite()` e `np.all()` para validar as probabilidades estimadas.

**Pandas (`pd`)**

Biblioteca utilizada para manipulação e análise de dados estruturados.

Utilizamos `pd.DataFrame()` para organizar as métricas retornadas pelo experimento.

**XGBClassifier**

É a implementação do classificador XGBoost compatível com a interface de estimadores do scikit-learn.

Essa compatibilidade permite utilizar o algoritmo dentro do `Pipeline` já implementado na função reutilizável.

### 2. Instanciação do classificador

```python
estimador_xgb_222 = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    n_jobs=4,
    random_state=42
)
```

A expressão `XGBClassifier(...)` cria uma instância do classificador.

**O que é uma instância?**

Na programação orientada a objetos, uma classe define características e comportamentos de um objeto.

Quando executamos `XGBClassifier(...)`, criamos um objeto com determinada configuração.

Esse objeto ainda não foi treinado.

O treinamento acontecerá posteriormente, quando a função reutilizável executar `pipeline.fit()`.

**Principais parâmetros:**

- `n_estimators=200`: estabelece 200 rodadas de boosting.
- `max_depth=4`: limita a profundidade máxima das árvores.
- `learning_rate=0.05`: controla a contribuição de cada nova árvore.
- `subsample=0.8`: utiliza uma amostra de aproximadamente 80% dos registros em cada rodada.
- `colsample_bytree=0.8`: utiliza uma amostra de aproximadamente 80% das variáveis disponíveis para cada árvore.
- `objective="binary:logistic"`: define a função objetivo para classificação binária probabilística.
- `eval_metric="aucpr"`: configura uma métrica interna do XGBoost relacionada à área sob a curva Precision-Recall.
- `tree_method="hist"`: utiliza um algoritmo baseado em histogramas para construir as árvores.
- `n_jobs=4`: limita a quatro threads o paralelismo do XGBoost.
- `random_state=42`: define uma semente para os procedimentos aleatórios.

O parâmetro `scale_pos_weight` não foi informado. Portanto, utilizamos seu comportamento padrão, sem ponderação adicional da classe positiva.

**Observação:** `eval_metric="aucpr"` não substitui a Average Precision calculada pelo scikit-learn. São medidas relacionadas, mas não necessariamente numericamente idênticas.

### 3. Reutilização da função de experimentação

```python
pipeline_xgb_222, metricas_xgb_222, probabilidades_xgb_222 = (
    executar_experimento_classificacao(...)
)
```

Aqui utilizamos um conceito importante do Python: **desempacotamento de tuplas**.

Nossa função retorna três objetos:

1. O pipeline treinado.
2. Um dicionário contendo as métricas.
3. Um array com as probabilidades estimadas na validação.

O Python distribui automaticamente esses resultados entre as três variáveis.

Por exemplo:

```python
a, b, c = (10, 20, 30)
```

Após a execução:

- `a` contém 10.
- `b` contém 20.
- `c` contém 30.

O mesmo princípio é aplicado aos resultados do nosso experimento.

### 4. O que acontece dentro da função reutilizável?

A função `executar_experimento_classificacao()` realiza as seguintes operações:

1. Valida os parâmetros e os conjuntos de dados.
2. Configura o `ColumnTransformer`.
3. Aplica `OneHotEncoder` às variáveis categóricas.
4. Aplica `StandardScaler` às variáveis numéricas.
5. Constrói o `Pipeline` com o estimador recebido.
6. Executa o treinamento com `fit()`.
7. Calcula as probabilidades com `predict_proba()`.
8. Calcula as métricas.
9. Retorna os resultados.

**O ponto fundamental é que não precisamos repetir essas operações para cada algoritmo.**

Basta fornecer um estimador diferente.

O `Pipeline` também garante que o pré-processamento seja ajustado durante o treinamento, evitando utilizar informações da validação para calcular seus parâmetros.

### 5. Construção do DataFrame de métricas

```python
df_metricas_xgb_222 = pd.DataFrame(
    [metricas_xgb_222]
)
```

A variável `metricas_xgb_222` contém um dicionário.

Por exemplo:

```python
metricas = {
    "Modelo": "XGBoost",
    "Average Precision": 0.18
}
```

O exemplo acima é ilustrativo, não representa o resultado do experimento.

Os colchetes em `[metricas_xgb_222]` criam uma lista contendo o dicionário.

O Pandas interpreta cada dicionário da lista como uma linha do DataFrame.

Isso facilita a visualização e a comparação posterior dos experimentos.

### 6. Recuperação do modelo treinado

```python
modelo_xgb_treinado_222 = (
    pipeline_xgb_222.named_steps["classificador"]
)
```

O atributo `named_steps` permite acessar os componentes de um `Pipeline` pelo nome.

Na função reutilizável, o pipeline possui dois componentes:

- `preprocessamento`
- `classificador`

Ao acessar `named_steps["classificador"]`, recuperamos o objeto XGBoost já treinado.

Não estamos realizando um novo treinamento.

### 7. Consulta à quantidade de rodadas

```python
total_arvores_222 = (
    modelo_xgb_treinado_222.get_booster()
    .num_boosted_rounds()
)
```

O método `get_booster()` retorna o objeto interno do XGBoost responsável pelo modelo treinado.

O método `num_boosted_rounds()` informa a quantidade de rodadas de boosting realizadas.

Na configuração inicial, esperamos 200 rodadas, pois definimos `n_estimators=200` e não configuramos interrupção antecipada (*early stopping*).

Em nossa classificação binária convencional, cada rodada corresponde à adição de uma árvore.

### 8. Validação das probabilidades

```python
assert len(probabilidades_xgb_222) == len(y_validacao)
```

A instrução `assert` verifica uma condição lógica.

Se a condição for verdadeira, o programa continua normalmente.

Se for falsa, o Python gera uma exceção `AssertionError`.

Neste caso, verificamos se existe uma probabilidade para cada registro de validação.

```python
assert np.isfinite(probabilidades_xgb_222).all()
```

A função `np.isfinite()` verifica se os valores são finitos, descartando resultados como `NaN` ou infinito.

O método `.all()` confirma que todos os elementos atendem à condição.

Também verificamos se todas as probabilidades estão entre zero e um.

### 9. Validação da matriz de confusão

```python
assert (
    metricas_xgb_222["TN"]
    + metricas_xgb_222["FP"]
    + metricas_xgb_222["FN"]
    + metricas_xgb_222["TP"]
    == len(y_validacao)
)
```

A matriz de confusão contém quatro quantidades:

- **TN:** coletas dentro do limite corretamente identificadas.
- **FP:** coletas dentro do limite classificadas incorretamente como atrasadas.
- **FN:** coletas atrasadas que o modelo não identificou.
- **TP:** coletas atrasadas corretamente identificadas.

A soma desses quatro valores precisa corresponder ao total de registros avaliados.

Essa verificação permite identificar inconsistências na contabilização das classificações.

### 10. Apresentação dos resultados

```python
display(
    df_metricas_xgb_222.T.rename(
        columns={0: "Resultado"}
    )
)
```

O atributo `.T` realiza a transposição do DataFrame.

Isso transforma as métricas, originalmente organizadas em colunas, em linhas.

O método `.rename()` altera o nome da coluna de resultados para facilitar a leitura.

A função `display()` apresenta o DataFrame formatado no Jupyter Notebook.

### 11. Relação com o objetivo de negócio

O modelo XGBoost produzirá uma probabilidade estimada de atraso para cada coleta do conjunto de validação.

Essas probabilidades poderão ser utilizadas para ordenar as coletas por risco.

Por exemplo, uma Torre de Controle com capacidade para analisar apenas 5% das coletas poderia priorizar aquelas com maiores probabilidades estimadas.

Entretanto, ainda precisamos verificar experimentalmente quantos atrasos reais o modelo consegue identificar dentro dessa capacidade.

Uma probabilidade elevada não garante que a coleta atrasará, e a qualidade dessa estimativa depende do desempenho e da calibração do modelo.

### 12. Conclusão metodológica

Este experimento reutiliza a estrutura desenvolvida anteriormente, alterando apenas o algoritmo e sua configuração.

Assim, mantemos consistência entre os experimentos e reduzimos a duplicação de código.

Após a execução, analisaremos a Average Precision, as demais métricas e o tempo de treinamento.

A comparação operacional por capacidades fixas de alertas será realizada na sequência da etapa 2.22.

**O conjunto de teste permanece reservado para a avaliação final.**

## 2.22.1 — Resultados, interpretação e conclusão do primeiro experimento XGBoost

### 1. Execução do experimento

O primeiro experimento com XGBoost foi executado com sucesso, reutilizando a função `executar_experimento_classificacao()`.

A configuração utilizou 200 rodadas de boosting, profundidade máxima de quatro níveis e taxa de aprendizado de 0,05, sem balanceamento adicional de classes.

O tempo de treinamento registrado foi de aproximadamente **16,04 segundos**.

O conjunto de teste não foi utilizado.

### 2. Resultados de validação

| Métrica | Resultado |
|---|---:|
| Accuracy | 0,974886 |
| Precision | 0,646643 |
| Recall | 0,033590 |
| F1-score | 0,063863 |
| Average Precision | 0,171407 |
| Verdadeiros negativos (TN) | 208.081 |
| Falsos positivos (FP) | 100 |
| Falsos negativos (FN) | 5.265 |
| Verdadeiros positivos (TP) | 183 |
| Total de alertas | 283 |
| Percentual de alertas | 0,132473% |
| Tempo de treinamento | 16,04 segundos |

Os resultados foram obtidos utilizando o limiar de classificação de 0,50, exceto a Average Precision, que considera as probabilidades estimadas ao longo dos diferentes limiares.

### 3. Interpretação das métricas

**Accuracy — 97,49%**

O modelo classificou corretamente a maior parte das observações.

Entretanto, como aproximadamente 2,55% das coletas apresentam atraso, a Accuracy isoladamente não representa adequadamente a capacidade de identificar a classe positiva.

Um modelo que classificasse todas as coletas como pontuais já apresentaria Accuracy elevada.

**Precision — 64,66%**

Entre as 283 coletas classificadas como atrasadas, 183 realmente apresentaram atraso.

Isso significa que aproximadamente 64,66% dos alertas emitidos estavam corretos.

**Recall — 3,36%**

Entre os 5.448 atrasos reais existentes na validação, o modelo identificou apenas 183.

Consequentemente, 5.265 atrasos não foram identificados utilizando o limiar de 0,50.

Esse resultado demonstra que o limiar inicial é bastante restritivo para a identificação de atrasos.

**F1-score — 0,063863**

O F1-score combina Precision e Recall por meio de uma média harmônica.

O valor baixo é consequência principalmente do Recall reduzido.

**Average Precision — 0,171407**

A Average Precision resume a relação entre Precision e Recall considerando diferentes limiares de decisão.

Essa métrica é especialmente relevante porque nosso problema apresenta forte desbalanceamento entre coletas pontuais e atrasadas.

O resultado indica que o XGBoost possui capacidade de discriminação superior à da Regressão Logística original, embora tenha apresentado desempenho ligeiramente inferior ao Random Forest neste primeiro experimento.

### 4. Comparação preliminar entre algoritmos

| Algoritmo | Average Precision |
|---|---:|
| Regressão Logística original | 0,118805 |
| Random Forest | 0,179512 |
| XGBoost | 0,171407 |

O Random Forest permanece com a maior Average Precision entre os modelos avaliados até o momento.

A diferença entre Random Forest e XGBoost é de aproximadamente 0,008105 ponto absoluto.

Em termos relativos, a Average Precision do XGBoost foi aproximadamente 4,5% inferior à do Random Forest.

Entretanto, essa diferença ainda não permite estabelecer uma superioridade definitiva, pois os algoritmos não passaram por uma etapa sistemática de otimização de hiperparâmetros.

### 5. Implicações para a Torre de Controle Logística

O objetivo de negócio não é apenas classificar corretamente as coletas, mas identificar antecipadamente aquelas com maior risco de atraso.

Em uma operação logística, a capacidade de análise e intervenção pode ser limitada.

Por isso, a Torre de Controle pode trabalhar com uma quantidade máxima de alertas, priorizando as coletas com maiores probabilidades estimadas de atraso.

O limiar de 0,50 gerou alertas para apenas 0,1325% das observações de validação.

Esse volume é muito inferior às capacidades operacionais de 1%, 2%, 5%, 10%, 15%, 20% e 30% consideradas nos experimentos anteriores.

Consequentemente, a próxima análise deverá utilizar as probabilidades estimadas pelo XGBoost para ordenar as coletas por risco e avaliar seu desempenho nessas mesmas capacidades.

Isso permitirá verificar quantos atrasos reais poderiam ser identificados dentro de diferentes limites de atendimento da Torre de Controle.

### 6. Limitações do experimento

O experimento apresenta algumas limitações importantes:

- Os hiperparâmetros ainda não foram otimizados.
- Não foi aplicado balanceamento adicional de classes.
- A análise por capacidade fixa de alertas ainda não foi realizada.
- A calibração das probabilidades ainda não foi avaliada.
- A comparação atual não estabelece superioridade estatística ou operacional definitiva entre os modelos.

Essas limitações serão consideradas nas etapas posteriores.

### 7. Conclusão

O primeiro experimento demonstrou que o XGBoost pode ser integrado à função reutilizável desenvolvida no projeto, mantendo a padronização metodológica dos experimentos.

O modelo alcançou Average Precision de 0,171407, superando a Regressão Logística original, mas permanecendo ligeiramente abaixo do Random Forest.

Com limiar de 0,50, apresentou Precision elevada e Recall muito baixo, identificando apenas 183 dos 5.448 atrasos reais.

A próxima investigação deverá avaliar o XGBoost por capacidade operacional de alertas, utilizando as mesmas capacidades adotadas anteriormente.

**Conclusão da etapa 2.22.1:** o experimento inicial foi concluído com sucesso. O Random Forest permanece como referência de desempenho, mas ainda não há evidências suficientes para selecionar definitivamente o melhor algoritmo.

O conjunto de teste continua preservado.

## 2.22.2 — Avaliação do XGBoost por capacidade operacional de alertas

### 1. Objetivo

Avaliar a capacidade do XGBoost de priorizar coletas com maior risco de atraso, considerando diferentes limites de atendimento da Torre de Controle Logística.

### 2. Justificativa

Na etapa 2.22.1, o XGBoost alcançou Average Precision de 0,171407.

Entretanto, utilizando o limiar de 0,50, o modelo emitiu apenas 283 alertas e identificou 183 dos 5.448 atrasos reais.

Para avaliar sua utilidade operacional, precisamos investigar seu desempenho quando a Torre de Controle possui uma capacidade fixa de investigação.

### 3. Metodologia

Utilizaremos exclusivamente o conjunto de validação, com 213.629 registros e 5.448 atrasos reais.

Serão consideradas as capacidades de 1%, 2%, 5%, 10%, 15%, 20% e 30%.

Para cada capacidade:

1. Ordenar as coletas pela probabilidade estimada de atraso, em ordem decrescente.
2. Selecionar as coletas com maiores probabilidades até atingir a capacidade definida.
3. Contabilizar quantos atrasos reais foram identificados.
4. Calcular Precision e Recall dentro da capacidade.
5. Organizar os resultados para comparação posterior.

### 4. Cuidados metodológicos

- Reutilizar o XGBoost treinado na etapa 2.22.1.
- Não realizar novo treinamento.
- Não modificar hiperparâmetros.
- Não utilizar o conjunto de teste.
- Preservar os mesmos percentuais e critérios de seleção da etapa 2.20.

### 5. Resultado esperado

Obter uma tabela com a quantidade de alertas, atrasos reais identificados, Precision e Recall para cada capacidade operacional.

Esses resultados permitirão avaliar a utilidade do XGBoost na priorização de coletas com risco de atraso.

In [22]:

# ============================================================
# ETAPA 2.22.2 — XGBOOST POR CAPACIDADE DE ALERTAS
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

# ------------------------------------------------------------
# 1. Recuperar as probabilidades da etapa 2.22.1
# ------------------------------------------------------------

capacidades_222 = [1, 2, 5, 10, 15, 20, 30]

y_real_222 = y_validacao.to_numpy()
total_registros_222 = len(y_real_222)
total_atrasos_222 = int(y_real_222.sum())

probabilidades_222 = np.asarray(
    probabilidades_xgb_222
)

# ------------------------------------------------------------
# 2. Validar os dados
# ------------------------------------------------------------

assert len(probabilidades_222) == total_registros_222
assert np.isfinite(probabilidades_222).all()

assert np.all(
    (probabilidades_222 >= 0)
    & (probabilidades_222 <= 1)
)

assert total_atrasos_222 == 5448

# ------------------------------------------------------------
# 3. Ordenar as coletas por risco decrescente
# ------------------------------------------------------------

ordem_risco_222 = np.lexsort((
    np.arange(total_registros_222),
    -probabilidades_222
))

# ------------------------------------------------------------
# 4. Calcular a Average Precision
# ------------------------------------------------------------

average_precision_222 = average_precision_score(
    y_real_222,
    probabilidades_222
)

# ------------------------------------------------------------
# 5. Avaliar cada capacidade operacional
# ------------------------------------------------------------

resultados_capacidade_222 = []

for capacidade in capacidades_222:

    quantidade_alertas = int(
        np.ceil(total_registros_222 * capacidade / 100)
    )

    indices_priorizados = ordem_risco_222[
        :quantidade_alertas
    ]

    tp = int(y_real_222[indices_priorizados].sum())

    fp = quantidade_alertas - tp
    fn = total_atrasos_222 - tp

    precision = tp / quantidade_alertas
    recall = tp / total_atrasos_222

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    resultados_capacidade_222.append({
        "Modelo": "XGBoost",
        "Capacidade (%)": capacidade,
        "Alertas": quantidade_alertas,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "Average Precision": average_precision_222
    })

# ------------------------------------------------------------
# 6. Organizar os resultados
# ------------------------------------------------------------

df_capacidade_222 = pd.DataFrame(
    resultados_capacidade_222
)

# ------------------------------------------------------------
# 7. Validar a consistência
# ------------------------------------------------------------

assert len(df_capacidade_222) == 7

assert (
    df_capacidade_222["TP"]
    + df_capacidade_222["FN"]
    == total_atrasos_222
).all()

assert (
    df_capacidade_222["TP"]
    + df_capacidade_222["FP"]
    == df_capacidade_222["Alertas"]
).all()

assert np.isclose(
    average_precision_222,
    metricas_xgb_222["Average Precision"]
)

# ------------------------------------------------------------
# 8. Apresentar os resultados
# ------------------------------------------------------------

print("=== XGBOOST — CAPACIDADE DE ALERTAS ===")

print(f"Coletas de validação: {total_registros_222:,}")
print(f"Atrasos reais: {total_atrasos_222:,}")
print(f"Average Precision: {average_precision_222:.6f}")

print("\n=== RESULTADOS POR CAPACIDADE ===")

print(
    df_capacidade_222.to_string(
        index=False,
        float_format=lambda valor: f"{valor:.4f}"
    )
)

print("\n=== VALIDAÇÕES ===")
print("Sete capacidades avaliadas: OK")
print("Métricas consistentes: OK")
print("Nenhum novo treinamento realizado.")
print("Conjunto de teste não utilizado.")


=== XGBOOST — CAPACIDADE DE ALERTAS ===
Coletas de validação: 213,629
Atrasos reais: 5,448
Average Precision: 0.171407

=== RESULTADOS POR CAPACIDADE ===
 Modelo  Capacidade (%)  Alertas   TP    FP   FN  Precision  Recall  F1-score  Average Precision
XGBoost               1     2137  750  1387 4698     0.3510  0.1377    0.1978             0.1714
XGBoost               2     4273 1099  3174 4349     0.2572  0.2017    0.2261             0.1714
XGBoost               5    10682 1746  8936 3702     0.1635  0.3205    0.2165             0.1714
XGBoost              10    21363 2550 18813 2898     0.1194  0.4681    0.1902             0.1714
XGBoost              15    32045 3004 29041 2444     0.0937  0.5514    0.1602             0.1714
XGBoost              20    42726 3339 39387 2109     0.0781  0.6129    0.1386             0.1714
XGBoost              30    64089 3899 60190 1549     0.0608  0.7157    0.1121             0.1714

=== VALIDAÇÕES ===
Sete capacidades avaliadas: OK
Métricas consistent

## 2.22.2 — Explicação do código e conceitos utilizados

### 1. Reutilização das probabilidades

```python
probabilidades_222 = np.asarray(
    probabilidades_xgb_222
)
```

A função `np.asarray()` converte os valores recebidos para um array NumPy, caso ainda não estejam nesse formato.

As probabilidades foram produzidas pelo XGBoost treinado na etapa anterior.

Portanto, não precisamos executar novamente `fit()` ou `predict_proba()`.

### 2. Ordenação por risco

```python
ordem_risco_222 = np.lexsort((
    np.arange(total_registros_222),
    -probabilidades_222
))
```

A função `np.lexsort()` realiza uma ordenação considerando múltiplas chaves.

Neste caso:

- `-probabilidades_222` determina a ordenação decrescente do risco.
- `np.arange()` fornece as posições originais dos registros para resolver eventuais empates.

A ordenação produz um array de índices, e não uma cópia dos registros.

Essa estratégia é a mesma utilizada na etapa 2.20.

### 3. Estrutura de repetição `for`

```python
for capacidade in capacidades_222:
```

O comando `for` permite executar repetidamente um bloco de instruções.

A cada repetição, a variável `capacidade` recebe um dos valores da lista:

`[1, 2, 5, 10, 15, 20, 30]`

Assim, calculamos as métricas para sete cenários operacionais sem duplicar o código.

### 4. Quantidade de alertas

```python
quantidade_alertas = int(
    np.ceil(total_registros_222 * capacidade / 100)
)
```

A expressão calcula quantas coletas serão priorizadas.

A função `np.ceil()` arredonda o resultado para cima.

Por exemplo, com 213.629 coletas e capacidade de 5%:

213.629 × 0,05 = 10.681,45

Após o arredondamento, serão selecionadas **10.682 coletas**.

A função `int()` converte o resultado para um número inteiro.

### 5. Seleção dos registros prioritários

```python
indices_priorizados = ordem_risco_222[
    :quantidade_alertas
]
```

Essa expressão utiliza *slicing*, um recurso do Python para selecionar parte de uma sequência.

O símbolo `:` indica que queremos os elementos desde o início até a posição informada, sem incluir o índice final.

Como os registros já estão ordenados por risco, selecionamos aqueles com maiores probabilidades de atraso.

### 6. Cálculo dos verdadeiros positivos

```python
tp = int(y_real_222[indices_priorizados].sum())
```

A expressão `y_real_222[indices_priorizados]` recupera os valores reais da variável-alvo para as coletas selecionadas.

Como a classe positiva é representada por 1, o método `.sum()` conta quantas dessas coletas realmente apresentaram atraso.

Esse resultado corresponde aos verdadeiros positivos, ou TP.

### 7. Precision e Recall por capacidade

```python
precision = tp / quantidade_alertas
recall = tp / total_atrasos_222
```

**Precision por capacidade**

Representa a proporção de alertas que correspondem a atrasos reais.

Por exemplo, se a Torre de Controle investigar 1.000 coletas e 200 apresentarem atraso, a Precision será de 20%.

**Recall por capacidade**

Representa a proporção de todos os atrasos reais que foram identificados dentro da capacidade disponível.

Se existirem 500 atrasos reais e 200 forem identificados, o Recall será de 40%.

Essas métricas permitem avaliar diferentes estratégias de priorização operacional.

### 8. F1-score

```python
f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0
    else 0.0
)
```

O F1-score é a média harmônica entre Precision e Recall.

O trecho `if ... else` representa uma expressão condicional do Python.

Ela evita divisão por zero quando ambas as métricas são iguais a zero.

### 9. Armazenamento dos resultados

```python
resultados_capacidade_222.append({
    "Modelo": "XGBoost",
    "Capacidade (%)": capacidade,
    "TP": tp
})
```

O método `.append()` adiciona um novo elemento ao final de uma lista.

Neste exemplo simplificado, adicionamos um dicionário contendo informações da capacidade avaliada.

No código completo, também armazenamos alertas, FP, FN, Precision, Recall, F1-score e Average Precision.

Posteriormente, `pd.DataFrame()` transforma essa lista de dicionários em uma tabela estruturada.

### 10. Validações com `assert`

As instruções `assert` verificam se os resultados atendem às condições esperadas.

Por exemplo:

```python
assert (
    df_capacidade_222["TP"]
    + df_capacidade_222["FP"]
    == df_capacidade_222["Alertas"]
).all()
```

A expressão verifica se a soma de verdadeiros e falsos positivos corresponde à quantidade de alertas em todas as capacidades.

Também confirmamos que a Average Precision permanece igual à calculada no experimento anterior, dentro da tolerância numérica utilizada por `np.isclose()`.

### 11. Relação com o negócio

A avaliação por capacidade operacional permite aproximar as métricas de Machine Learning das decisões de uma Torre de Controle.

Em vez de perguntar apenas se o modelo classifica corretamente uma coleta, passamos a investigar:

**Quantos atrasos reais conseguimos identificar quando temos recursos para investigar somente uma parcela das coletas?**

Essa abordagem é especialmente relevante em problemas com classes desbalanceadas e recursos operacionais limitados.

### 12. Conclusão metodológica

A etapa utiliza as probabilidades do XGBoost já treinado e reproduz os critérios utilizados anteriormente na comparação por capacidade de alertas.

Nenhum hiperparâmetro é alterado, nenhum novo treinamento é realizado e o conjunto de teste permanece preservado.

Os resultados permitirão verificar o desempenho do XGBoost nas sete capacidades operacionais e preparar sua comparação com os demais algoritmos.

## 2.22.2 — Resultados, interpretação e conclusão da avaliação por capacidade de alertas

### 1. Execução e validação

A avaliação do XGBoost por capacidade operacional foi executada com sucesso, reutilizando as probabilidades produzidas na etapa 2.22.1.

Não houve novo treinamento, alteração de hiperparâmetros ou utilização do conjunto de teste.

Foram avaliados 213.629 registros de validação, dos quais 5.448 representam coletas realizadas após o limite da janela operacional.

A Average Precision permaneceu em **0,171407**, confirmando a consistência com o experimento anterior.

### 2. Resultados por capacidade operacional

| Capacidade | Alertas | Atrasos identificados (TP) | Precision | Recall |
|---|---:|---:|---:|---:|
| 1% | 2.137 | 750 | 35,10% | 13,77% |
| 2% | 4.273 | 1.099 | 25,72% | 20,17% |
| 5% | 10.682 | 1.746 | 16,35% | 32,05% |
| 10% | 21.363 | 2.550 | 11,94% | 46,81% |
| 15% | 32.045 | 3.004 | 9,37% | 55,14% |
| 20% | 42.726 | 3.339 | 7,81% | 61,29% |
| 30% | 64.089 | 3.899 | 6,08% | 71,57% |

### 3. Interpretação dos resultados

**Capacidade de 1%**

Ao priorizar apenas 2.137 coletas, o modelo identificou 750 atrasos reais.

A Precision de 35,10% indica que aproximadamente 35 de cada 100 alertas correspondem a atrasos efetivos.

Esse resultado demonstra concentração de risco nas coletas com maiores probabilidades estimadas.

**Capacidade de 5%**

Ao selecionar 10.682 coletas, o modelo identificou 1.746 atrasos reais.

Isso representa 32,05% de todos os atrasos existentes na validação.

A Precision foi de 16,35%.

Portanto, uma operação capaz de investigar 5% das coletas conseguiria identificar antecipadamente aproximadamente um terço dos atrasos, caso o desempenho observado se reproduzisse em operação.

**Capacidade de 10%**

Com 21.363 alertas, o modelo identificou 2.550 atrasos reais.

O Recall alcançou 46,81%, enquanto a Precision foi de 11,94%.

Isso representa uma ampliação significativa da cobertura de atrasos em comparação à capacidade de 5%.

**Capacidade de 30%**

Ao selecionar 64.089 coletas, o modelo identificou 3.899 atrasos reais.

O Recall alcançou 71,57%, mas a Precision diminuiu para 6,08%.

Esse cenário oferece maior cobertura, porém exige investigar uma quantidade muito superior de coletas.

### 4. Relação entre Precision e Recall

Os resultados demonstram um compromisso operacional entre Precision e Recall.

Quando a capacidade de alertas aumenta:

- Mais atrasos reais são identificados.
- A quantidade de falsos positivos também aumenta.
- A Precision acumulada tende a diminuir.
- O Recall acumulado aumenta ou permanece constante.

Esse comportamento é esperado em uma estratégia que ordena os registros por probabilidade estimada de risco.

A escolha da capacidade ideal dependerá dos recursos disponíveis para investigação, do custo dos falsos positivos e do impacto dos atrasos não identificados.

### 5. Comparação com a incidência geral de atrasos

A incidência de atrasos no conjunto de validação é de aproximadamente 2,55%.

Na capacidade de 5%, a Precision do XGBoost foi de 16,35%.

Isso significa que a concentração de atrasos no grupo priorizado foi aproximadamente **6,4 vezes superior à incidência geral**.

Esse indicador é conhecido como *Lift*.

O Lift mede quanto a seleção realizada pelo modelo melhora a concentração de eventos positivos em relação à seleção aleatória esperada.

Um Lift superior a 1 indica que o modelo concentra mais eventos positivos do que a média da população.

O resultado de aproximadamente 6,4 na capacidade de 5% reforça a utilidade potencial do XGBoost como mecanismo de priorização operacional.

### 6. Limitações da avaliação

A avaliação foi realizada retrospectivamente, utilizando os resultados reais do conjunto de validação.

Os números representam desempenho preditivo observado, não uma redução comprovada de atrasos.

Para obter benefícios operacionais reais, seria necessário integrar os alertas a um processo de acompanhamento e intervenção.

Também seria necessário avaliar os custos e a capacidade efetiva da Torre de Controle.

### 7. Conclusão

O XGBoost demonstrou capacidade de concentrar coletas atrasadas entre os registros com maior risco estimado.

Na capacidade de 5%, identificou 32,05% dos atrasos reais, investigando apenas 5% das coletas.

Na capacidade de 10%, identificou 46,81% dos atrasos.

Esses resultados mostram que a avaliação por capacidade operacional fornece informações mais diretamente relacionadas ao objetivo de negócio do que a utilização isolada do limiar de classificação de 0,50.

A Average Precision de 0,171407 permanece como referência para a comparação global entre algoritmos.

**Conclusão da etapa 2.22.2:** a avaliação operacional do XGBoost foi concluída e validada. Os resultados estão disponíveis para comparação posterior com Regressão Logística e Random Forest, mantendo o conjunto de teste reservado.

## 2.22.3 — Encerramento dos experimentos iniciais com XGBoost

### 1. Objetivo alcançado

A etapa 2.22 teve como objetivo implementar, treinar e avaliar o algoritmo XGBoost para identificar coletas com risco de ocorrer após o limite da janela operacional.

O experimento reutilizou a função `executar_experimento_classificacao()`, mantendo o pré-processamento, as dez variáveis preditoras e a divisão dos dados utilizados nos experimentos anteriores.

### 2. Resultados consolidados

O XGBoost foi treinado com 200 rodadas de boosting, sem balanceamento adicional de classes.

O tempo de treinamento foi de aproximadamente 16,04 segundos.

A Average Precision obtida no conjunto de validação foi de 0,171407.

Na avaliação por capacidade operacional, destacaram-se os seguintes resultados:

| Capacidade | Atrasos identificados | Recall |
|---|---:|---:|
| 1% | 750 | 13,77% |
| 5% | 1.746 | 32,05% |
| 10% | 2.550 | 46,81% |
| 20% | 3.339 | 61,29% |
| 30% | 3.899 | 71,57% |

### 3. Comparação preliminar

O Random Forest permanece com a maior Average Precision entre os algoritmos inicialmente avaliados:

- Regressão Logística original: 0,118805.
- Random Forest: 0,179512.
- XGBoost: 0,171407.

Entretanto, a avaliação por capacidade demonstrou que o XGBoost identificou mais atrasos que o Random Forest em algumas capacidades operacionais, especialmente entre 10% e 30%.

Portanto, a escolha do modelo não deverá considerar exclusivamente a Average Precision.

Também deverão ser avaliados a capacidade de priorização, o Recall operacional, a Precision, o custo dos alertas e a complexidade de implementação.

### 4. Conclusões metodológicas

A etapa demonstrou a possibilidade de reutilizar uma função padronizada de experimentação com diferentes algoritmos de classificação.

Também confirmou a importância de separar:

- Treinamento do modelo.
- Estimativa das probabilidades.
- Avaliação das métricas.
- Priorização operacional por capacidade de alertas.

Essa separação permite avaliar diferentes estratégias operacionais sem necessariamente treinar novamente o modelo.

### 5. Limitações

Os modelos ainda não passaram por otimização sistemática de hiperparâmetros.

A avaliação foi realizada retrospectivamente sobre o conjunto de validação.

Não foi realizada avaliação final no conjunto de teste.

Os resultados representam a capacidade de identificar coletas que efetivamente atrasaram, não uma redução comprovada de atrasos por intervenção operacional.

### 6. Encerramento

**A etapa 2.22 está concluída.**

Os resultados estão documentados e disponíveis para a automatização e comparação consolidada dos experimentos.

O conjunto de teste permanece reservado para a avaliação final do modelo selecionado.

A próxima etapa será **2.23 — Automatização dos experimentos de classificação com loop**, utilizando a função reutilizável desenvolvida anteriormente.

## 2.23 — Automatização dos experimentos de classificação com loop

### 1. Objetivo

Automatizar a execução de diferentes algoritmos de classificação utilizando uma única estrutura de repetição em Python.

O objetivo é reduzir a duplicação de código, padronizar os experimentos e facilitar a inclusão de novas configurações de modelos.

### 2. Contexto

Nas etapas anteriores, desenvolvemos e avaliamos:

- Regressão Logística original.
- Regressão Logística com balanceamento de classes.
- Random Forest.
- XGBoost.

Também construímos a função reutilizável `executar_experimento_classificacao()`, responsável pelo pré-processamento, treinamento, cálculo das probabilidades e avaliação das métricas.

Agora utilizaremos essa função dentro de um loop para executar diferentes configurações de modelos.

### 3. Conceitos de Python

**Dicionário (`dict`)**

Estrutura que associa chaves a valores.

Podemos utilizar um dicionário para relacionar o nome de cada experimento ao seu estimador.

**Estrutura de repetição (`for`)**

Permite percorrer uma coleção de elementos e executar um bloco de instruções para cada elemento.

**Método `.items()`**

Permite percorrer simultaneamente as chaves e os valores de um dicionário.

**Lista (`list`)**

Será utilizada para armazenar as métricas retornadas pelos experimentos.

**DataFrame**

Organizará os resultados em formato tabular, facilitando análises e comparações posteriores.

### 4. Estratégia metodológica

A automatização deverá preservar:

- As mesmas dez variáveis preditoras.
- Os mesmos conjuntos de treinamento e validação.
- A mesma função de experimentação.
- O mesmo limiar inicial de classificação.
- As configurações de hiperparâmetros já utilizadas.
- A separação entre treinamento, validação e teste.

Cada modelo será treinado independentemente.

Os resultados serão armazenados com identificação do algoritmo e de sua configuração.

### 5. Reprodutibilidade e custo computacional

Os experimentos anteriores já foram executados individualmente.

Executar novamente os quatro modelos permitirá validar a automatização, mas produzirá novos treinamentos e consumirá recursos computacionais.

Os resultados anteriores permanecerão preservados para comparação.

A automatização não constitui uma otimização de hiperparâmetros: nesta etapa, utilizaremos as configurações previamente definidas.

### 6. Resultado esperado

Ao final, teremos uma estrutura reutilizável capaz de executar os quatro experimentos, armazenar seus pipelines, métricas e probabilidades de validação e apresentar uma tabela consolidada.

Essa estrutura será utilizada posteriormente para apoiar a comparação de modelos e a integração com MLflow.

O conjunto de teste permanecerá reservado.

In [23]:

# ============================================================
# ETAPA 2.23 — AUTOMATIZAÇÃO DOS EXPERIMENTOS COM LOOP
# ============================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# ------------------------------------------------------------
# 1. Definir os quatro experimentos
# ------------------------------------------------------------

estimadores_223 = {
    "Regressão Logística — Original": LogisticRegression(
        class_weight=None,
        max_iter=1000,
        random_state=42
    ),

    "Regressão Logística — Balanceada": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        min_samples_leaf=20,
        max_features="sqrt",
        class_weight=None,
        n_jobs=4,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        n_jobs=4,
        random_state=42
    )
}

# ------------------------------------------------------------
# 2. Preparar as estruturas de armazenamento
# ------------------------------------------------------------

pipelines_223 = {}
metricas_223 = {}
probabilidades_223 = {}

# ------------------------------------------------------------
# 3. Executar os experimentos com loop
# ------------------------------------------------------------

for nome_modelo, estimador in estimadores_223.items():

    print(f"\nIniciando: {nome_modelo}")

    pipeline, metricas, probabilidades = (
        executar_experimento_classificacao(
            nome_modelo=nome_modelo,
            estimador=estimador,
            X_treino=X_treino,
            y_treino=y_treino,
            X_validacao=X_validacao,
            y_validacao=y_validacao,
            variaveis_categoricas=categoricas_218,
            variaveis_numericas=numericas_218,
            limiar=0.50
        )
    )

    pipelines_223[nome_modelo] = pipeline
    metricas_223[nome_modelo] = metricas
    probabilidades_223[nome_modelo] = probabilidades

    print(
        f"Average Precision: "
        f"{metricas['Average Precision']:.6f}"
    )

    print(
        f"Tempo de treinamento: "
        f"{metricas['Tempo treinamento (s)']:.2f} s"
    )

# ------------------------------------------------------------
# 4. Consolidar as métricas
# ------------------------------------------------------------

df_metricas_223 = pd.DataFrame(
    list(metricas_223.values())
)

# ------------------------------------------------------------
# 5. Validar a execução
# ------------------------------------------------------------

assert len(pipelines_223) == 4
assert len(metricas_223) == 4
assert len(probabilidades_223) == 4

for nome_modelo in estimadores_223:

    probabilidades = np.asarray(
        probabilidades_223[nome_modelo]
    )

    metricas = metricas_223[nome_modelo]

    assert len(probabilidades) == len(y_validacao)

    assert np.isfinite(probabilidades).all()

    assert (
        (probabilidades >= 0)
        & (probabilidades <= 1)
    ).all()

    assert (
        metricas["TN"]
        + metricas["FP"]
        + metricas["FN"]
        + metricas["TP"]
        == len(y_validacao)
    )

    assert (
        metricas["TP"] + metricas["FN"]
        == int(y_validacao.sum())
    )

# ------------------------------------------------------------
# 6. Apresentar os resultados
# ------------------------------------------------------------

print("\n=== ETAPA 2.23 — EXPERIMENTOS AUTOMATIZADOS ===")

print(f"Experimentos executados: {len(metricas_223)}")

colunas_resumo_223 = [
    "Modelo",
    "Average Precision",
    "Precision",
    "Recall",
    "F1-score",
    "Alertas",
    "Tempo treinamento (s)"
]

display(
    df_metricas_223[colunas_resumo_223]
    .sort_values(
        "Average Precision",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n=== VALIDAÇÕES ===")
print("Quatro experimentos executados: OK")
print("Probabilidades válidas: OK")
print("Matrizes de confusão consistentes: OK")
print("Pipelines armazenados: OK")
print("Conjunto de teste não utilizado.")



Iniciando: Regressão Logística — Original
Average Precision: 0.118805
Tempo de treinamento: 9.50 s

Iniciando: Regressão Logística — Balanceada
Average Precision: 0.112580
Tempo de treinamento: 20.09 s

Iniciando: Random Forest
Average Precision: 0.179512
Tempo de treinamento: 49.02 s

Iniciando: XGBoost
Average Precision: 0.171407
Tempo de treinamento: 7.94 s

=== ETAPA 2.23 — EXPERIMENTOS AUTOMATIZADOS ===
Experimentos executados: 4


,Modelo,Average Precision,Precision,Recall,F1-score,Alertas,Tempo treinamento (s)
0,Random Forest,0.179512,0.868852,0.029185,0.056473,183,49.023437
1,XGBoost,0.171407,0.646643,0.033590,0.063863,283,7.935502
2,Regressão Logística — Original,0.118805,0.000000,0.000000,0.000000,7,9.502818
3,Regressão Logística — Balanceada,0.112580,0.059771,0.629222,0.109172,57352,20.087296



=== VALIDAÇÕES ===
Quatro experimentos executados: OK
Probabilidades válidas: OK
Matrizes de confusão consistentes: OK
Pipelines armazenados: OK
Conjunto de teste não utilizado.


## 2.23 — Explicação do código e conceitos utilizados

### 1. Dicionário de estimadores

```python
estimadores_223 = {
    "Regressão Logística — Original": LogisticRegression(...),
    "Random Forest": RandomForestClassifier(...)
}
```

O dicionário (`dict`) é uma estrutura de dados do Python que armazena pares de chave e valor.

Neste experimento:

- A chave identifica o nome do modelo.
- O valor contém o objeto estimador configurado.

Essa estrutura permite adicionar novos experimentos sem modificar o funcionamento do loop.

Os estimadores são instanciados, mas ainda não estão treinados.

### 2. Estruturas para armazenar resultados

```python
pipelines_223 = {}
metricas_223 = {}
probabilidades_223 = {}
```

Criamos três dicionários vazios.

Cada um possui uma responsabilidade:

**`pipelines_223`**

Armazena os pipelines treinados, incluindo o pré-processamento e o classificador.

**`metricas_223`**

Armazena os dicionários de métricas retornados pela função reutilizável.

**`probabilidades_223`**

Armazena os arrays com as probabilidades estimadas para os registros de validação.

Essa separação facilita a organização e a recuperação dos resultados.

### 3. Estrutura de repetição `for`

```python
for nome_modelo, estimador in estimadores_223.items():
```

O comando `for` percorre os elementos do dicionário.

O método `.items()` retorna pares contendo a chave e o valor.

Em cada repetição:

1. `nome_modelo` recebe o nome do experimento.
2. `estimador` recebe o objeto classificador correspondente.
3. A função reutilizável executa o treinamento e a avaliação.
4. Os resultados são armazenados.

O loop será executado quatro vezes, uma para cada configuração.

### 4. Reutilização da função de classificação

```python
pipeline, metricas, probabilidades = (
    executar_experimento_classificacao(...)
)
```

A função executa o procedimento que já desenvolvemos na etapa 2.17.

Ela realiza o pré-processamento, constrói o pipeline, treina o estimador e calcula as métricas de validação.

A atribuição de três variáveis utiliza o desempacotamento de tuplas do Python.

Assim, os três objetos retornados são atribuídos às variáveis correspondentes.

### 5. Armazenamento dos resultados

```python
pipelines_223[nome_modelo] = pipeline
metricas_223[nome_modelo] = metricas
probabilidades_223[nome_modelo] = probabilidades
```

A sintaxe `dicionario[chave] = valor` permite adicionar ou atualizar um elemento do dicionário.

Por exemplo:

```python
resultados = {}
resultados["Modelo A"] = 0.18
```

Após essa operação, o dicionário contém a chave `"Modelo A"` associada ao valor `0.18`.

No nosso experimento, armazenamos objetos mais complexos, como pipelines treinados e arrays de probabilidades.

### 6. Criação do DataFrame

```python
df_metricas_223 = pd.DataFrame(
    list(metricas_223.values())
)
```

O método `.values()` retorna os valores armazenados no dicionário.

A função `list()` converte esses valores para uma lista.

Como cada valor é um dicionário de métricas, `pd.DataFrame()` organiza os resultados em uma tabela, utilizando uma linha por experimento.

### 7. Validações com `assert`

```python
assert len(pipelines_223) == 4
```

A instrução `assert` verifica se uma condição é verdadeira.

Nesse caso, esperamos quatro pipelines treinados.

Também verificamos:

- Se cada modelo produziu uma probabilidade por registro de validação.
- Se todas as probabilidades são valores finitos.
- Se todas estão entre zero e um.
- Se as quantidades da matriz de confusão correspondem ao total de registros.
- Se a quantidade de positivos reais permanece consistente.

Essas verificações ajudam a identificar falhas na automatização.

### 8. Ordenação dos resultados

```python
.sort_values(
    "Average Precision",
    ascending=False
)
```

O método `sort_values()` ordena as linhas do DataFrame por uma coluna.

O parâmetro `ascending=False` estabelece a ordem decrescente.

Assim, os modelos com maior Average Precision aparecem primeiro na tabela.

Essa ordenação é apenas uma apresentação dos resultados, não uma seleção definitiva do melhor modelo.

### 9. Diferença entre automatização e otimização

**Automatização**

Executa vários experimentos de forma padronizada, reduzindo a repetição de código.

**Otimização de hiperparâmetros**

Investiga diferentes configurações de um algoritmo para identificar aquelas que apresentam melhor desempenho.

Nesta etapa, estamos automatizando a execução de configurações já utilizadas.

Não estamos procurando automaticamente novos hiperparâmetros.

### 10. Relação com o MLflow

A estrutura criada poderá facilitar a integração futura com MLflow.

Para cada experimento, teremos:

- Nome e configuração do estimador.
- Pipeline treinado.
- Métricas de validação.
- Probabilidades estimadas.
- Tempo de treinamento.

Posteriormente, poderemos registrar esses elementos como experimentos rastreáveis.

### 11. Conclusão metodológica

O loop transforma a execução individual dos modelos em um procedimento reutilizável e padronizado.

Isso facilita a manutenção do notebook, a comparação entre algoritmos e a ampliação controlada dos experimentos.

Os quatro modelos utilizam as mesmas variáveis e conjuntos de dados, preservando a consistência metodológica.

A avaliação permanece restrita aos conjuntos de treinamento e validação.

O conjunto de teste continua reservado para a avaliação final.

## 2.23.1 — Resultados, interpretação e conclusão da automatização

### 1. Objetivo da validação

Verificar se a estrutura automatizada com `for` e a função `executar_experimento_classificacao()` consegue reproduzir os experimentos individuais de classificação, preservando as configurações e a metodologia estabelecidas.

### 2. Resultados obtidos

Foram executados quatro experimentos:

| Modelo | Average Precision | Precision | Recall | Alertas | Treinamento (s) |
|---|---:|---:|---:|---:|---:|
| Random Forest | 0,179512 | 0,868852 | 0,029185 | 183 | 49,02 |
| XGBoost | 0,171407 | 0,646643 | 0,033590 | 283 | 7,94 |
| Regressão Logística original | 0,118805 | 0,000000 | 0,000000 | 7 | 9,50 |
| Regressão Logística balanceada | 0,112580 | 0,059771 | 0,629222 | 57.352 | 20,09 |

Todos os experimentos utilizaram o limiar de classificação de 0,50.

### 3. Validação da automatização

As verificações executadas apresentaram os seguintes resultados:

- Quatro experimentos executados: OK.
- Probabilidades válidas: OK.
- Matrizes de confusão consistentes: OK.
- Pipelines armazenados: OK.
- Conjunto de teste preservado.

As métricas de Average Precision foram reproduzidas em relação aos experimentos individuais.

Isso demonstra que a automatização manteve os resultados dos modelos previamente avaliados, nas condições utilizadas.

### 4. Interpretação dos resultados

**Random Forest**

Obteve a maior Average Precision, com valor de 0,179512.

No limiar de 0,50, apresentou Precision elevada, porém Recall baixo.

Isso significa que os poucos alertas emitidos apresentaram alta proporção de acertos, mas a maior parte dos atrasos não foi identificada nesse limiar.

**XGBoost**

Obteve Average Precision de 0,171407, próxima à do Random Forest.

Apresentou tempo de treinamento de aproximadamente 7,94 segundos, inferior ao observado para o Random Forest nesta execução.

Entretanto, uma única medição de tempo não é suficiente para estabelecer uma superioridade definitiva de desempenho computacional.

**Regressão Logística original**

Obteve Average Precision de 0,118805.

No limiar de 0,50, emitiu apenas sete alertas e não identificou nenhum atraso real.

Isso não significa que o modelo seja incapaz de ordenar riscos, pois sua Average Precision permanece superior à prevalência aproximada de atrasos na validação.

**Regressão Logística balanceada**

Obteve Recall de aproximadamente 62,92%, porém emitiu 57.352 alertas.

A Precision de aproximadamente 5,98% demonstra que a maior sensibilidade foi acompanhada por um volume elevado de falsos positivos.

Essa característica deve ser considerada na avaliação de viabilidade operacional.

### 5. Importância da capacidade operacional

O limiar de 0,50 não representa necessariamente a melhor estratégia para a Torre de Controle.

Em uma aplicação operacional, a quantidade de alertas precisa ser compatível com a capacidade de investigação.

Por isso, a comparação dos modelos deverá considerar também a seleção das coletas com maiores probabilidades estimadas.

Essa abordagem permite comparar modelos sob uma mesma restrição de capacidade.

### 6. Ganhos técnicos da automatização

A utilização do loop trouxe os seguintes benefícios:

- Redução da duplicação de código.
- Padronização dos experimentos.
- Centralização das configurações dos estimadores.
- Armazenamento estruturado dos pipelines.
- Armazenamento das métricas e probabilidades.
- Facilidade de inclusão de novas configurações.
- Preparação para integração futura com MLflow.

A função reutilizável continua responsável pelo treinamento e avaliação, enquanto o loop coordena a execução dos diferentes experimentos.

Essa separação melhora a organização e a manutenção do código.

### 7. Limitações

Os experimentos ainda não passaram por otimização sistemática de hiperparâmetros.

As métricas foram calculadas no conjunto de validação.

O tempo de treinamento pode variar entre execuções.

A classificação com limiar fixo não representa, isoladamente, a capacidade operacional de priorização da Torre de Controle.

A seleção definitiva do modelo ainda não foi realizada.

### 8. Conclusão

**A etapa 2.23 foi concluída com sucesso.**

A automatização reproduziu os resultados individuais de Average Precision e demonstrou que diferentes algoritmos podem ser executados por uma estrutura comum, com armazenamento organizado dos resultados.

O Random Forest apresentou a maior Average Precision, enquanto o XGBoost demonstrou desempenho competitivo e menor tempo de treinamento nesta execução.

Entretanto, ainda não existem elementos suficientes para selecionar definitivamente o modelo mais adequado à Torre de Controle.

A próxima etapa será **2.24 — Comparação consolidada dos modelos de classificação**, considerando métricas estatísticas, capacidade operacional de alertas e custo computacional.

O conjunto de teste permanece reservado para a avaliação final.

## 2.24 — Comparação consolidada dos modelos de classificação

### 1. Objetivo

Comparar os quatro experimentos de classificação desenvolvidos no laboratório, considerando tanto o desempenho estatístico quanto sua aplicação operacional na Torre de Controle Logística.

O objetivo não é simplesmente identificar o algoritmo com maior pontuação, mas compreender os benefícios e limitações de cada modelo para a priorização de coletas com risco de atraso.

### 2. Modelos avaliados

Serão considerados os quatro experimentos da etapa 2.23:

1. Regressão Logística original.
2. Regressão Logística balanceada.
3. Random Forest.
4. XGBoost.

Todos foram treinados utilizando os mesmos conjuntos de treinamento e validação e as mesmas variáveis preditoras.

### 3. Critérios de comparação

**Average Precision (AP)**

Avalia a qualidade da ordenação das previsões em diferentes níveis de Precision e Recall, resumindo o comportamento da curva Precision-Recall.

É especialmente relevante para nosso problema, pois os atrasos representam aproximadamente 2,55% das coletas.

**Precision**

Representa a proporção de alertas emitidos que correspondem a atrasos reais.

**Recall**

Representa a proporção dos atrasos reais que foram identificados pelo modelo.

**F1-score**

Combina Precision e Recall em uma média harmônica.

**Capacidade operacional**

Representa a porcentagem de coletas que a Torre de Controle consegue priorizar para investigação.

Serão avaliadas as capacidades de 1%, 2%, 5%, 10%, 15%, 20% e 30%.

Para cada capacidade, selecionaremos os registros com maiores probabilidades estimadas e calcularemos:

- Quantidade de alertas.
- Verdadeiros positivos (TP).
- Falsos positivos (FP).
- Falsos negativos (FN).
- Precision operacional.
- Recall operacional.

**Tempo de treinamento**

Será utilizado como indicador complementar de custo computacional, reconhecendo que uma única execução não constitui um benchmark definitivo.

### 4. Estratégia de comparação

A análise será dividida em duas perspectivas.

**Perspectiva estatística**

Compararemos as métricas calculadas na etapa 2.23, identificando diferenças entre os algoritmos.

**Perspectiva operacional**

Compararemos os modelos sob a mesma capacidade de investigação.

Essa abordagem é importante porque um modelo pode apresentar melhor Average Precision global e, ainda assim, identificar menos atrasos do que outro em determinada capacidade.

### 5. Reutilização dos resultados

Serão reutilizadas as estruturas:

- `pipelines_223`: pipelines treinados.
- `metricas_223`: métricas dos experimentos.
- `probabilidades_223`: probabilidades estimadas na validação.
- `df_metricas_223`: tabela consolidada de métricas.

Não haverá novo treinamento nesta etapa.

A comparação utilizará exclusivamente o conjunto de validação.

### 6. Critérios para seleção futura

A análise deverá considerar:

- Qualidade da ordenação dos riscos.
- Quantidade de atrasos identificados.
- Quantidade de alertas necessários.
- Restrições de capacidade operacional.
- Custo computacional.
- Possibilidade de utilização e manutenção do modelo.

A escolha definitiva não será feita exclusivamente com base em uma única métrica.

### 7. Resultado esperado

Ao final da etapa, teremos tabelas comparativas e uma análise consolidada dos quatro experimentos.

Esses resultados apoiarão a decisão sobre quais modelos deverão avançar para a otimização de hiperparâmetros.

O conjunto de teste permanecerá reservado para a avaliação final, após a seleção do modelo e de suas configurações.

In [36]:

# ============================================================
# ETAPA 2.24 — COMPARAÇÃO CONSOLIDADA DOS CLASSIFICADORES
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Recuperar os resultados da etapa 2.23
# ------------------------------------------------------------

modelos_224 = list(probabilidades_223.keys())

capacidades_224 = [1, 2, 5, 10, 15, 20, 30]

y_real_224 = y_validacao.to_numpy()

total_registros_224 = len(y_real_224)
total_atrasos_224 = int(y_real_224.sum())

# ------------------------------------------------------------
# 2. Validar os dados disponíveis
# ------------------------------------------------------------

assert len(modelos_224) == 4

assert set(modelos_224) == set(metricas_223.keys())

assert set(modelos_224) == set(pipelines_223.keys())

assert total_atrasos_224 > 0

# ------------------------------------------------------------
# 3. Comparar os modelos por capacidade operacional
# ------------------------------------------------------------

resultados_224 = []

for nome_modelo in modelos_224:

    probabilidades = np.asarray(
        probabilidades_223[nome_modelo]
    )

    assert len(probabilidades) == total_registros_224

    assert np.isfinite(probabilidades).all()

    assert np.all(
        (probabilidades >= 0)
        & (probabilidades <= 1)
    )

    # Ordenação decrescente do risco.
    # Empates são resolvidos pela posição original.
    ordem_risco = np.lexsort((
        np.arange(total_registros_224),
        -probabilidades
    ))

    for capacidade in capacidades_224:

        quantidade_alertas = int(
            np.ceil(
                total_registros_224 * capacidade / 100
            )
        )

        indices_priorizados = ordem_risco[
            :quantidade_alertas
        ]

        tp = int(
            y_real_224[indices_priorizados].sum()
        )

        fp = quantidade_alertas - tp
        fn = total_atrasos_224 - tp

        precision = (
            tp / quantidade_alertas
            if quantidade_alertas > 0
            else 0.0
        )

        recall = tp / total_atrasos_224

        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall) > 0
            else 0.0
        )

        resultados_224.append({
            "Modelo": nome_modelo,
            "Capacidade (%)": capacidade,
            "Alertas": quantidade_alertas,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "Precision": precision,
            "Recall": recall,
            "F1-score": f1,
            "Average Precision": metricas_223[
                nome_modelo
            ]["Average Precision"],
            "Tempo treinamento (s)": metricas_223[
                nome_modelo
            ]["Tempo treinamento (s)"]
        })

# ------------------------------------------------------------
# 4. Organizar os resultados
# ------------------------------------------------------------

df_capacidade_224 = pd.DataFrame(resultados_224)

df_capacidade_224 = (
    df_capacidade_224
    .sort_values(["Capacidade (%)", "Modelo"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 5. Criar a comparação de atrasos identificados
# ------------------------------------------------------------

tabela_tp_224 = df_capacidade_224.pivot(
    index="Capacidade (%)",
    columns="Modelo",
    values="TP"
)

# ------------------------------------------------------------
# 6. Criar a comparação de Recall operacional
# ------------------------------------------------------------

tabela_recall_224 = df_capacidade_224.pivot(
    index="Capacidade (%)",
    columns="Modelo",
    values="Recall"
)

# ------------------------------------------------------------
# 7. Validar a consistência dos resultados atuais
# ------------------------------------------------------------

assert len(df_capacidade_224) == (
    len(modelos_224) * len(capacidades_224)
)

for capacidade, grupo in df_capacidade_224.groupby(
    "Capacidade (%)"
):

    assert len(grupo) == len(modelos_224)

    assert grupo["Modelo"].nunique() == len(modelos_224)

    assert grupo["Alertas"].nunique() == 1

    assert (
        grupo["TP"] + grupo["FP"] == grupo["Alertas"]
    ).all()

    assert (
        grupo["TP"] + grupo["FN"] == total_atrasos_224
    ).all()

    assert (
        (grupo["Precision"] >= 0)
        & (grupo["Precision"] <= 1)
    ).all()

    assert (
        (grupo["Recall"] >= 0)
        & (grupo["Recall"] <= 1)
    ).all()

# ------------------------------------------------------------
# 8. Comparação histórica com a etapa 2.20
# ------------------------------------------------------------

# Os experimentos da etapa 2.20 foram treinados
# individualmente.
#
# Na etapa 2.23, os modelos foram treinados novamente
# utilizando a função reutilizável padronizada.
#
# A investigação identificou uma diferença no formato
# de saída do OneHotEncoder:
#
# Etapa 2.20: sparse_output=True
# Etapa 2.23: sparse_output=False
#
# Essa diferença pode influenciar o processo numérico
# de treinamento, embora sua contribuição exata para
# as divergências ainda não tenha sido isolada.
#
# Portanto, a comparação histórica será informativa.
# As validações internas da etapa 2.24 permanecem
# obrigatórias.

nomes_anteriores_224 = {
    "Regressão Logística — Original":
        "Regressão Logística original",

    "Regressão Logística — Balanceada":
        "Regressão Logística balanceada",

    "Random Forest":
        "Random Forest"
}

comparacoes_historicas_224 = []

for nome_atual, nome_anterior in nomes_anteriores_224.items():

    atual = (
        df_capacidade_224[
            df_capacidade_224["Modelo"] == nome_atual
        ]
        .set_index("Capacidade (%)")["TP"]
        .sort_index()
    )

    anterior = (
        df_capacidade_220[
            df_capacidade_220["Modelo"] == nome_anterior
        ]
        .set_index("Capacidade (%)")["TP"]
        .sort_index()
    )

    # A estrutura histórica deve permanecer comparável.
    assert atual.index.equals(anterior.index)

    assert not atual.isna().any()
    assert not anterior.isna().any()

    diferencas = atual - anterior

    comparacoes_historicas_224.append({
        "Modelo": nome_atual,
        "Valores exatamente iguais": atual.equals(anterior),
        "Maior diferença absoluta de TP": int(
            diferencas.abs().max()
        ),
        "Capacidades com divergência": int(
            (diferencas != 0).sum()
        )
    })

df_historico_224 = pd.DataFrame(
    comparacoes_historicas_224
)

# ------------------------------------------------------------
# 9. Apresentar os resultados consolidados
# ------------------------------------------------------------

print("=== ETAPA 2.24 — COMPARAÇÃO CONSOLIDADA ===")

print(f"Modelos avaliados: {len(modelos_224)}")

print(
    f"Coletas de validação: "
    f"{total_registros_224:,}"
)

print(
    f"Atrasos reais: "
    f"{total_atrasos_224:,}"
)

print("\n=== MÉTRICAS ESTATÍSTICAS ===")

display(
    df_metricas_223[
        [
            "Modelo",
            "Average Precision",
            "Precision",
            "Recall",
            "F1-score",
            "Alertas",
            "Tempo treinamento (s)"
        ]
    ]
    .sort_values(
        "Average Precision",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n=== ATRASOS IDENTIFICADOS POR CAPACIDADE ===")

display(tabela_tp_224)

print("\n=== RECALL OPERACIONAL POR CAPACIDADE ===")

display(tabela_recall_224.round(4))

print("\n=== COMPARAÇÃO COMPLETA ===")

display(
    df_capacidade_224[
        [
            "Modelo",
            "Capacidade (%)",
            "Alertas",
            "TP",
            "FP",
            "FN",
            "Precision",
            "Recall",
            "F1-score"
        ]
    ]
)

print("\n=== COMPARAÇÃO HISTÓRICA — ETAPA 2.20 ===")

display(df_historico_224)

# ------------------------------------------------------------
# 10. Resultado das validações
# ------------------------------------------------------------

print("\n=== VALIDAÇÕES ===")

print("Quatro modelos avaliados: OK")
print("Sete capacidades avaliadas: OK")
print("Quantidade de alertas consistente: OK")
print("Matrizes operacionais consistentes: OK")
print("Probabilidades válidas: OK")
print("Comparação histórica realizada: OK")
print("Divergências históricas preservadas para análise.")
print("Nenhum novo treinamento realizado.")
print("Conjunto de teste não utilizado.")


=== ETAPA 2.24 — COMPARAÇÃO CONSOLIDADA ===
Modelos avaliados: 4
Coletas de validação: 213,629
Atrasos reais: 5,448

=== MÉTRICAS ESTATÍSTICAS ===


,Modelo,Average Precision,Precision,Recall,F1-score,Alertas,Tempo treinamento (s)
0,Random Forest,0.179512,0.868852,0.029185,0.056473,183,49.023437
1,XGBoost,0.171407,0.646643,0.033590,0.063863,283,7.935502
2,Regressão Logística — Original,0.118805,0.000000,0.000000,0.000000,7,9.502818
3,Regressão Logística — Balanceada,0.112580,0.059771,0.629222,0.109172,57352,20.087296



=== ATRASOS IDENTIFICADOS POR CAPACIDADE ===


Modelo,Random Forest,Regressão Logística — Balanceada,Regressão Logística — Original,XGBoost
Capacidade (%),,,,
1,750,573,530,750
2,1109,919,938,1099
5,1821,1476,1475,1746
10,2502,2064,2106,2550
15,2954,2641,2626,3004
20,3298,3032,2972,3339
30,3871,3569,3529,3899



=== RECALL OPERACIONAL POR CAPACIDADE ===


Modelo,Random Forest,Regressão Logística — Balanceada,Regressão Logística — Original,XGBoost
Capacidade (%),,,,
1,0.1377,0.1052,0.0973,0.1377
2,0.2036,0.1687,0.1722,0.2017
5,0.3343,0.2709,0.2707,0.3205
10,0.4593,0.3789,0.3866,0.4681
15,0.5422,0.4848,0.4820,0.5514
20,0.6054,0.5565,0.5455,0.6129
30,0.7105,0.6551,0.6478,0.7157



=== COMPARAÇÃO COMPLETA ===


,Modelo,Capacidade (%),Alertas,TP,FP,FN,Precision,Recall,F1-score
0,Random Forest,1,2137,750,1387,4698,0.350959,0.137665,0.197759
1,Regressão Logística — Balanceada,1,2137,573,1564,4875,0.268133,0.105176,0.151088
2,Regressão Logística — Original,1,2137,530,1607,4918,0.248011,0.097283,0.139750
3,XGBoost,1,2137,750,1387,4698,0.350959,0.137665,0.197759
4,Random Forest,2,4273,1109,3164,4339,0.259537,0.203561,0.228166
5,Regressão Logística — Balanceada,2,4273,919,3354,4529,0.215071,0.168686,0.189075
6,Regressão Logística — Original,2,4273,938,3335,4510,0.219518,0.172173,0.192984
7,XGBoost,2,4273,1099,3174,4349,0.257196,0.201725,0.226108
8,Random Forest,5,10682,1821,8861,3627,0.170474,0.334251,0.225790
9,Regressão Logística — Balanceada,5,10682,1476,9206,3972,0.138176,0.270925,0.183013



=== COMPARAÇÃO HISTÓRICA — ETAPA 2.20 ===


,Modelo,Valores exatamente iguais,Maior diferença absoluta de TP,Capacidades com divergência
0,Regressão Logística — Original,True,0,0
1,Regressão Logística — Balanceada,False,3,4
2,Random Forest,True,0,0



=== VALIDAÇÕES ===
Quatro modelos avaliados: OK
Sete capacidades avaliadas: OK
Quantidade de alertas consistente: OK
Matrizes operacionais consistentes: OK
Probabilidades válidas: OK
Comparação histórica realizada: OK
Divergências históricas preservadas para análise.
Nenhum novo treinamento realizado.
Conjunto de teste não utilizado.


## 2.24 — Explicação do código e conceitos utilizados

### 1. Reutilização dos experimentos

A etapa utiliza os objetos `probabilidades_223`, `metricas_223` e `df_metricas_223`, produzidos anteriormente.

Isso evita treinar novamente os quatro classificadores.

As probabilidades representam as estimativas dos modelos para os registros do conjunto de validação.

### 2. Recuperação dos nomes dos modelos

```python
modelos_224 = list(probabilidades_223.keys())
```

O método `.keys()` retorna as chaves de um dicionário.

A função `list()` transforma essas chaves em uma lista.

Assim, obtemos os nomes dos quatro experimentos que serão percorridos pelo loop.

### 3. Conversão do alvo para NumPy

```python
y_real_224 = y_validacao.to_numpy()
```

O método `.to_numpy()` converte uma Series do Pandas em um array NumPy.

Essa conversão permite selecionar os valores reais utilizando posições inteiras, mantendo a correspondência com as probabilidades estimadas.

### 4. Ordenação por risco

```python
ordem_risco = np.lexsort((
    np.arange(total_registros_224),
    -probabilidades
))
```

A função `np.lexsort()` realiza uma ordenação utilizando múltiplas chaves.

Neste caso:

- `-probabilidades` determina a ordem decrescente de risco.
- `np.arange()` fornece as posições originais.
- As posições originais são utilizadas para resolver empates de forma determinística.

Assim, as coletas com maiores probabilidades de atraso são priorizadas.

### 5. Definição da capacidade

```python
quantidade_alertas = int(
    np.ceil(total_registros_224 * capacidade / 100)
)
```

A capacidade representa a porcentagem das coletas que poderão receber atenção prioritária.

A função `np.ceil()` arredonda o resultado para cima.

Por exemplo, uma capacidade de 5% significa selecionar aproximadamente 5% das coletas com maior risco estimado.

### 6. Seleção dos registros priorizados

```python
indices_priorizados = ordem_risco[:quantidade_alertas]
```

A sintaxe `[:quantidade_alertas]` é um exemplo de fatiamento (*slicing*) do Python.

Ela seleciona os primeiros elementos do array, até a quantidade estabelecida.

Como o array já está ordenado por risco, selecionamos as coletas com maiores probabilidades de atraso.

### 7. Cálculo das métricas operacionais

**Verdadeiros positivos (TP)**

Coletas priorizadas que realmente apresentaram atraso.

**Falsos positivos (FP)**

Coletas priorizadas que não apresentaram atraso.

**Falsos negativos (FN)**

Coletas que apresentaram atraso, mas ficaram fora da capacidade selecionada.

A Precision operacional é calculada dividindo os verdadeiros positivos pela quantidade de alertas.

O Recall operacional é calculado dividindo os verdadeiros positivos pela quantidade total de atrasos reais.

Essas métricas são calculadas para cada capacidade e para cada modelo.

### 8. Transformação dos resultados com `pivot()`

```python
tabela_tp_224 = df_capacidade_224.pivot(
    index="Capacidade (%)",
    columns="Modelo",
    values="TP"
)
```

O método `pivot()` reorganiza um DataFrame.

Neste exemplo:

- Cada linha representa uma capacidade.
- Cada coluna representa um modelo.
- Cada célula contém a quantidade de atrasos identificados.

Essa estrutura facilita a comparação direta dos quatro classificadores.

O mesmo procedimento é utilizado para criar a tabela de Recall operacional.

### 9. Validações com `assert`

As instruções `assert` verificam condições esperadas.

Entre as verificações realizadas estão:

- Existência de quatro modelos.
- Existência de sete capacidades.
- Igualdade da quantidade de alertas entre modelos na mesma capacidade.
- Consistência entre verdadeiros positivos, falsos positivos e alertas.
- Consistência entre verdadeiros positivos, falsos negativos e atrasos reais.

Também comparamos os resultados dos três modelos anteriores com os resultados produzidos na etapa 2.20.

Essa verificação permite identificar possíveis divergências entre os experimentos individuais e a comparação consolidada.

### 10. Distinção entre desempenho estatístico e operacional

A Average Precision resume a qualidade da ordenação das previsões ao longo da curva Precision-Recall.

Já a avaliação por capacidade responde a uma pergunta operacional específica:

**Quantos atrasos conseguimos identificar quando podemos investigar apenas determinada porcentagem das coletas?**

Um modelo com maior Average Precision não necessariamente identificará mais atrasos em todas as capacidades.

Por isso, as duas perspectivas serão consideradas na interpretação final.

### 11. Conclusão metodológica

A etapa 2.24 consolida os resultados dos quatro experimentos em uma estrutura comparável.

A análise utiliza exclusivamente as probabilidades e métricas do conjunto de validação.

Não são realizados novos treinamentos nem ajustes de hiperparâmetros.

A escolha do modelo mais adequado será discutida após a execução e interpretação dos resultados.

O conjunto de teste permanece reservado.

## 2.24.1 — Resultados, interpretação e conclusão da comparação consolidada

### 1. Objetivo alcançado

A etapa 2.24 consolidou a avaliação de quatro experimentos de classificação, considerando métricas estatísticas, capacidade operacional de alertas e tempo de treinamento.

Foram reutilizados os resultados da etapa 2.23, sem realizar novos treinamentos.

A avaliação utilizou exclusivamente o conjunto de validação, composto por 213.629 coletas, das quais 5.448 apresentaram atraso.

### 2. Comparação estatística

| Modelo | Average Precision | Tempo de treinamento |
|---|---:|---:|
| Random Forest | 0,179512 | 49,02 s |
| XGBoost | 0,171407 | 7,94 s |
| Regressão Logística original | 0,118805 | 9,50 s |
| Regressão Logística balanceada | 0,112580 | 20,09 s |

O Random Forest apresentou a maior Average Precision.

O XGBoost obteve resultado próximo ao Random Forest e apresentou o menor tempo de treinamento nesta execução.

As regressões logísticas apresentaram valores inferiores de Average Precision.

Entretanto, a comparação estatística isolada não é suficiente para determinar o modelo mais adequado à aplicação operacional.

### 3. Comparação por capacidade operacional

Os modelos foram avaliados nas capacidades de 1%, 2%, 5%, 10%, 15%, 20% e 30%.

Para cada capacidade, foram selecionadas as coletas com maiores probabilidades estimadas.

O Random Forest apresentou maior quantidade de atrasos identificados nas capacidades de 2% e 5%, empatando com o XGBoost em 1%.

O XGBoost apresentou maior quantidade de atrasos identificados nas capacidades de 10%, 15%, 20% e 30%.

Na capacidade de 5%:

- Random Forest: 1.821 atrasos identificados.
- XGBoost: 1.746 atrasos identificados.
- Diferença: 75 atrasos a favor do Random Forest.

Na capacidade de 10%:

- Random Forest: 2.502 atrasos identificados.
- XGBoost: 2.550 atrasos identificados.
- Diferença: 48 atrasos a favor do XGBoost.

Esses resultados demonstram que o desempenho operacional depende da quantidade de coletas priorizadas.

### 4. Interpretação para a Torre de Controle

A capacidade operacional representa a proporção de coletas que poderão receber atenção prioritária.

Quanto maior a capacidade, maior tende a ser a quantidade de atrasos identificados.

Entretanto, também aumenta o número de alertas que não correspondem a atrasos reais.

Essa relação exige equilíbrio entre Recall, Precision e recursos operacionais disponíveis.

A escolha do modelo deverá considerar a capacidade de investigação e os custos associados aos falsos positivos e falsos negativos.

### 5. Reprodutibilidade e investigação técnica

A comparação histórica com a etapa 2.20 identificou reprodução exata dos verdadeiros positivos para a Regressão Logística original e o Random Forest.

A Regressão Logística balanceada apresentou pequenas divergências em quatro das sete capacidades avaliadas, com diferença máxima de três verdadeiros positivos.

A investigação identificou:

- Mesmos hiperparâmetros do classificador.
- Mesmo solver `lbfgs`.
- Mesmo número de iterações: 109.
- Mesma quantidade de coeficientes: 53.
- Coeficientes e interceptos ligeiramente diferentes.
- Pré-processamento com valores idênticos na amostra comparada.
- Representação esparsa no pipeline histórico e densa no pipeline automatizado.

A diferença de representação pode influenciar o processo numérico de treinamento.

Entretanto, não foi realizado um experimento controlado que demonstre que essa seja a única causa das divergências.

Para as próximas etapas, os experimentos automatizados da etapa 2.23 serão utilizados como referência padronizada.

Os resultados históricos permanecerão preservados.

### 6. Validações realizadas

Todas as validações internas da etapa 2.24 foram concluídas com sucesso.

Foram confirmados:

- Quatro modelos avaliados.
- Sete capacidades operacionais.
- Quantidade consistente de alertas por capacidade.
- Consistência das matrizes operacionais.
- Probabilidades válidas.
- Comparação histórica registrada.
- Ausência de novos treinamentos.
- Preservação do conjunto de teste.

### 7. Limitações

Os resultados foram obtidos exclusivamente no conjunto de validação.

Os modelos ainda não passaram por otimização sistemática de hiperparâmetros.

A capacidade operacional real da Torre de Controle não foi estabelecida.

Os tempos de treinamento representam medições individuais e não constituem uma comparação definitiva de desempenho computacional.

As diferenças observadas entre modelos não foram submetidas a uma análise de incerteza estatística.

A identificação retrospectiva de atrasos não comprova que uma intervenção operacional conseguiria evitá-los.

### 8. Conclusão

**A etapa 2.24 foi concluída com sucesso.**

A comparação consolidada demonstrou que Random Forest e XGBoost são os candidatos mais promissores entre as configurações inicialmente avaliadas.

O Random Forest apresentou a maior Average Precision e melhor desempenho em algumas capacidades menores.

O XGBoost apresentou desempenho competitivo, menor tempo de treinamento nesta execução e maior quantidade de atrasos identificados em capacidades de 10% a 30%.

Não será realizada a seleção definitiva do modelo neste momento.

Os resultados servirão como referência para a próxima etapa de otimização de hiperparâmetros.

Após a otimização, os modelos candidatos serão novamente comparados utilizando o conjunto de validação.

O conjunto de teste permanecerá reservado para a avaliação final.